# Yamada Formula Discovery Applications

> **Role:** advanced exact-computation and publication-reproduction notebook.
> Learn the single-graph Yamada API first; do not use **Run All** as a first test
> of the package.

Use this notebook when you want to use KnottedGraph as an exact-computation
engine for discovering and testing Yamada-polynomial formulae across
parameterized spatial-graph families.

The notebook asks one practical research question:

> How much information about a local motif word survives in the exact Yamada
> polynomial of the resulting spatial graph?

It progresses through three regimes:

\[
\boxed{\text{homogeneous repetition}
\longrightarrow \text{Abelian count-only mixing}
\longrightarrow \text{non-Abelian order-sensitive mixing}.}
\]

Each part defines a graph family, constructs embedded graphs, evaluates exact
Laurent polynomials, exports only the data needed for fitting or audit, freezes a
candidate identity, and then runs separate held-out checks. Outputs live under
`User_guide/applications/results/05_yamada_formula_discovery`; execution outputs
are deliberately not committed in the notebook.

The notebook is browsable on an ordinary review branch. Set
`KNOTTEDGRAPH_STRICT_PUBLICATION_REGENERATION=1` only for an audited clean
regeneration with the required source ancestry and factorized native backend.
Held-out exact agreement is strong computational evidence, but it is not a
substitute for a mathematical proof.


# Part I — Homogeneous theta-derived families

## Goal

Construct three infinite subcubic families from a common positive two-strand
theta skeleton, compute their normalized Yamada polynomials independently, infer
closed forms, freeze those forms, and test them on distant held-out values.

The named families in this application are

$$
\Lambda_m,\qquad
\Lambda_m^{\downarrow},\qquad
\Lambda_m^{\updownarrow}.
$$

They share a canonical two-strand theta skeleton with

$$
q=2m+1
$$

crossings.  The local modifications are, respectively:

1. **Cross-linked**: every separated strand pair receives a rung.
2. **Lower-paired**: consecutive lower subdivision vertices are paired
   non-overlappingly.
3. **Two-sided paired**: the same non-overlapping pairing is applied on both
   lower and upper sides.

The non-overlap condition keeps the graphs subcubic.

## What this part establishes

The exact production evaluator is used to construct a discovery sequence
without feeding any candidate formula into the computation.  Candidate formulas
are frozen only after the discovery data exist, then tested at

$$
m=101,\ 125,\ 150,\ 200.
$$

This separation is important: held-out agreement is a falsification test of a
conjectured formula, not a circular reconstruction.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import csv
import hashlib
import json
import os
import pickle
import subprocess
import time
import warnings
from typing import Callable

import networkx as nx
import numpy as np
import sympy as sp

AUDITED_SOURCE_COMMIT = "49e34e47ca5c182f55ef5f0ea0906220df59befb"
STRICT_PUBLICATION_REGENERATION = (
    os.environ.get("KNOTTEDGRAPH_STRICT_PUBLICATION_REGENERATION", "0") == "1"
)
CONSTRUCTOR_VERSION = "2026-08-25.three-subcubic-theta-families.safe-return.v3"
RESULT_SCHEMA_VERSION = 1

A = sp.Symbol("A")
SIGMA = A + 1 + A**-1


def _is_repo_root(path: Path) -> bool:
    return (
        (path / "pyproject.toml").is_file()
        and (path / "src" / "knotted_graph").is_dir()
        and (path / "User_guide" / "applications").is_dir()
    )


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if _is_repo_root(candidate):
            return candidate
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook from inside the repository checkout."
    )


ROOT = find_repo_root()
RESULTS_DIR = ROOT / "User_guide" / "applications" / "results" / "05_yamada_formula_discovery"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FULL_CSV = RESULTS_DIR / "05a_homogeneous_theta_formula_discovery_full.csv"
SHARE_CSV = RESULTS_DIR / "05a_homogeneous_theta_formula_discovery_share.csv"
HELDOUT_CSV = RESULTS_DIR / "05a_homogeneous_theta_formula_discovery_heldout.csv"
PROJECTION_CACHE_DIR = RESULTS_DIR / "05a_homogeneous_theta_formula_discovery_pd_cache"
PROJECTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def git_text(*args: str) -> str:
    proc = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.returncode:
        raise RuntimeError(
            f"git {' '.join(args)} failed.\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc.stdout.strip()


CURRENT_BRANCH = git_text("rev-parse", "--abbrev-ref", "HEAD")
GIT_COMMIT = git_text("rev-parse", "HEAD")

AUDITED_SOURCE_PRESENT = subprocess.run(
    ["git", "merge-base", "--is-ancestor", AUDITED_SOURCE_COMMIT, "HEAD"],
    cwd=ROOT,
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

if STRICT_PUBLICATION_REGENERATION and not AUDITED_SOURCE_PRESENT:
    raise RuntimeError(
        "Strict publication regeneration requires audited source commit "
        f"{AUDITED_SOURCE_COMMIT}. Current HEAD is {GIT_COMMIT}."
    )
if not AUDITED_SOURCE_PRESENT:
    warnings.warn(
        "The audited source commit is not an ancestor of this checkout. "
        "Exploration may continue, but do not treat regenerated tables as "
        "publication-audited outputs.",
        RuntimeWarning,
    )

library_status = git_text("status", "--porcelain", "--", "src/knotted_graph")
if library_status:
    raise RuntimeError(
        "The KnottedGraph source tree has uncommitted changes. "
        "Commit or stash them before generating a publication dataset:\n"
        + library_status
    )

import knotted_graph

KG_FILE = Path(knotted_graph.__file__).resolve()
if ROOT not in KG_FILE.parents:
    raise RuntimeError(
        "Python imported knotted_graph from outside this checkout.\n"
        f"Repository root: {ROOT}\nImported package: {KG_FILE}"
    )

from knotted_graph.core.embedding import ensure_embedding
from knotted_graph.projection import PDCode, sample_projections, select_projection
from knotted_graph.invariants.yamada.polynomial import Yamada
from knotted_graph.invariants.yamada.factorized_frontier import (
    build_factorized_frontier,
    native_factorized_available,
    factorized_import_error,
)

if not native_factorized_available():
    raise RuntimeError(
        "The optimized native factorized Yamada backend is unavailable. "
        "Rebuild this checkout before running the dataset.\n"
        f"Import error: {factorized_import_error()!r}"
    )

print("Repository :", ROOT)
print("Branch     :", CURRENT_BRANCH)
print("Commit     :", GIT_COMMIT)
print("Package    :", KG_FILE)
print("Native optimized Yamada backend: AVAILABLE")


### Run configuration

In [ ]:
DV_DISCOVERY_N = list(range(3, 11))
DV_HELDOUT_N = list(range(11, 21)) + [50]

# ------------------------------------------------------------
# Main discovery range
# ------------------------------------------------------------
MAX_DISCOVERY_M = 100
MAIN_DISCOVERY_M = list(range(1, MAX_DISCOVERY_M + 1))

# Not computed until formulas are frozen.
MAIN_HELDOUT_M = [101, 125, 150, 200]
COMPUTE_HELDOUT_NOW = False

# ------------------------------------------------------------
# Fast canonical-projection policy
# ------------------------------------------------------------
ROTATION_ORDER = "ZYX"

# These families are DEFINED by the canonical diagram.  Zero rotation is therefore
# the intended projection; no generic projection sampling is necessary.
USE_CANONICAL_ZERO_PROJECTION = True

# Frontier planning repeats part of the Yamada preparation.  It is useful for
# diagnostics but unnecessary for the dataset, so keep it off for maximum speed.
COMPUTE_FRONTIER_DIAGNOSTICS = False

REUSE_PROJECTION_CACHE = True
WRITE_PROJECTION_CACHE = True

# ------------------------------------------------------------
# Correctness / geometry audits
# ------------------------------------------------------------
RUN_UPSTREAM_SANITY = True

# The contact audit is quadratic in the number of PL segments.  Do it only on
# representative members because the geometry is a repeated local motif.
RUN_REPRESENTATIVE_3D_CONTACT_AUDIT = True
REPRESENTATIVE_AUDIT_M = [1, 2, 5, 10]
CONTACT_TOL = 1e-8

# ------------------------------------------------------------
# Runtime / persistence
# ------------------------------------------------------------
RESUME = True
FAIL_FAST = False
NORMALIZE_YAMADA = True

# Rewriting a CSV containing many large exact polynomials after every row becomes
# expensive.  Completed rows remain in memory and are atomically checkpointed every
# SAVE_EVERY_CASES cases, at family boundaries, and on KeyboardInterrupt.
SAVE_EVERY_CASES = 10

# Five PL segments per braid crossing are enough to represent the crossing robustly
# while reducing projection cost substantially versus the previous value 9.
BRAID_SAMPLES_PER_CROSSING = 5
RETURN_SAMPLES = 31
SIDE_CONNECTION_SAMPLES = 9

# Safe canonical return corridors.
# The old long Bezier closures re-entered the side-pair band at large m.
SAFE_RETURN_Y = 1.85
SAFE_RETURN_RIGHT_MARGIN = 0.70
SAFE_RETURN_LEFT_X = -0.55

# Cheap geometry regression before the long Yamada sweep.
RUN_CANONICAL_PROJECTION_PREFLIGHT = True
CANONICAL_PREFLIGHT_M = [1, 5, 10, 25, 50, 100, 101, 200]

COMPUTE_CONFIG = {
    "schema": RESULT_SCHEMA_VERSION,
    "constructor_version": CONSTRUCTOR_VERSION,
    "rotation_order": ROTATION_ORDER,
    "use_canonical_zero_projection": USE_CANONICAL_ZERO_PROJECTION,
    "compute_frontier_diagnostics": COMPUTE_FRONTIER_DIAGNOSTICS,
    "normalize": NORMALIZE_YAMADA,
    "braid_samples_per_crossing": BRAID_SAMPLES_PER_CROSSING,
    "return_samples": RETURN_SAMPLES,
    "side_connection_samples": SIDE_CONNECTION_SAMPLES,
    "safe_return_y": SAFE_RETURN_Y,
    "safe_return_right_margin": SAFE_RETURN_RIGHT_MARGIN,
    "safe_return_left_x": SAFE_RETURN_LEFT_X,
}
COMPUTE_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(COMPUTE_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

RUN_CONFIG = {
    **COMPUTE_CONFIG,
    "dv_discovery_n": DV_DISCOVERY_N,
    "dv_heldout_n": DV_HELDOUT_N,
    "main_discovery_m": MAIN_DISCOVERY_M,
    "main_heldout_m": MAIN_HELDOUT_M,
    "compute_heldout_now": COMPUTE_HELDOUT_NOW,
    "save_every_cases": SAVE_EVERY_CASES,
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

print("Compute config :", COMPUTE_CONFIG_SHA256)
print("Request config :", RUN_CONFIG_SHA256)
print(
    "Main-family discovery m-range:",
    MAIN_DISCOVERY_M[0], "..", MAIN_DISCOVERY_M[-1],
    f"({len(MAIN_DISCOVERY_M)} values per family)",
)
print("Future held-out m-values:", MAIN_HELDOUT_M)
print("Canonical zero projection:", USE_CANONICAL_ZERO_PROJECTION)
print("Frontier diagnostics:", COMPUTE_FRONTIER_DIAGNOSTICS)
print("Checkpoint every:", SAVE_EVERY_CASES, "new cases")
print("FULL DATASET :", FULL_CSV)
print("SHARE DATASET:", SHARE_CSV)
print("HELD-OUT DATASET:", HELDOUT_CSV)


### Upstream Yamada sanity gate

In [ ]:
if RUN_UPSTREAM_SANITY:
    sanity_script = ROOT / "dev" / "run_yamada_sanity_checks.py"
    if not sanity_script.is_file():
        raise RuntimeError(f"Missing sanity script: {sanity_script}")

    proc = subprocess.run(
        [os.sys.executable, str(sanity_script)],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )

    print(proc.stdout)

    if proc.returncode:
        raise RuntimeError(
            "Upstream Yamada sanity checks failed.\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )

    marker = "PASS: all published/independent Yamada sanity checks succeeded."
    if marker not in proc.stdout:
        raise RuntimeError("Expected upstream Yamada sanity success marker was not emitted.")

print("PASS: upstream Yamada sanity gate.")


### Geometry helpers

In [ ]:
def join_paths(*paths: np.ndarray) -> np.ndarray:
    pieces = []
    for path in paths:
        path = np.asarray(path, dtype=float)
        if len(path) == 0:
            continue
        if pieces and np.allclose(pieces[-1][-1], path[0]):
            path = path[1:]
        if len(path):
            pieces.append(path)

    if not pieces:
        return np.empty((0, 3), dtype=float)
    return np.vstack(pieces)


def line(a, b, samples: int = 2) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (1.0 - t) * a + t * b


def bezier_cubic(p0, p1, p2, p3, samples: int = 41) -> np.ndarray:
    p0, p1, p2, p3 = [np.asarray(p, dtype=float) for p in (p0, p1, p2, p3)]
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def segment_distance_3d(p1, q1, p2, q2) -> float:
    p1 = np.asarray(p1, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    u = q1 - p1
    v = q2 - p2
    w = p1 - p2

    a = float(u @ u)
    b = float(u @ v)
    c = float(v @ v)
    d = float(u @ w)
    e = float(v @ w)

    eps = 1e-14

    if a <= eps and c <= eps:
        return float(np.linalg.norm(p1 - p2))
    if a <= eps:
        t = float(np.clip(e / c, 0.0, 1.0))
        return float(np.linalg.norm(p1 - (p2 + t * v)))
    if c <= eps:
        s = float(np.clip(-d / a, 0.0, 1.0))
        return float(np.linalg.norm((p1 + s * u) - p2))

    D = a * c - b * b
    sN = sD = D
    tN = tD = D

    if D < eps:
        sN = 0.0
        sD = 1.0
        tN = e
        tD = c
    else:
        sN = b * e - c * d
        tN = a * e - b * d

        if sN < 0.0:
            sN = 0.0
            tN = e
            tD = c
        elif sN > sD:
            sN = sD
            tN = e + b
            tD = c

    if tN < 0.0:
        tN = 0.0
        if -d < 0.0:
            sN = 0.0
        elif -d > a:
            sN = sD
        else:
            sN = -d
            sD = a
    elif tN > tD:
        tN = tD
        if (-d + b) < 0.0:
            sN = 0.0
        elif (-d + b) > a:
            sN = sD
        else:
            sN = -d + b
            sD = a

    sc = 0.0 if abs(sN) < eps else sN / sD
    tc = 0.0 if abs(tN) < eps else tN / tD

    return float(np.linalg.norm(w + sc * u - tc * v))


def audit_piecewise_linear_embedding(graph: nx.MultiGraph, *, tolerance: float = CONTACT_TOL) -> None:
    graph = ensure_embedding(graph, copy=True, normalize=True)

    segments = []
    for u, v, key, data in graph.edges(keys=True, data=True):
        pts = np.asarray(data["pts"], dtype=float)
        lengths = np.linalg.norm(np.diff(pts, axis=0), axis=1)
        if np.any(lengths <= tolerance):
            raise AssertionError(f"Degenerate segment in edge {(u, v, key)!r}.")
        for segment_index in range(len(pts) - 1):
            segments.append((u, v, key, segment_index, pts[segment_index], pts[segment_index + 1]))

    for i, first in enumerate(segments):
        u1, v1, k1, s1, p1, q1 = first
        for second in segments[i + 1:]:
            u2, v2, k2, s2, p2, q2 = second

            if (u1, v1, k1) == (u2, v2, k2) and abs(s1 - s2) <= 1:
                continue

            if (
                np.any(np.maximum(p1, q1) + tolerance < np.minimum(p2, q2))
                or np.any(np.maximum(p2, q2) + tolerance < np.minimum(p1, q1))
            ):
                continue

            allowed = False
            for node in {u1, v1}.intersection({u2, v2}):
                pos = np.asarray(graph.nodes[node]["pos"], dtype=float)
                first_touches = min(np.linalg.norm(p1 - pos), np.linalg.norm(q1 - pos)) <= tolerance
                second_touches = min(np.linalg.norm(p2 - pos), np.linalg.norm(q2 - pos)) <= tolerance
                if first_touches and second_touches:
                    allowed = True
                    break

            if allowed:
                continue

            if segment_distance_3d(p1, q1, p2, q2) <= tolerance:
                raise AssertionError(
                    "Unintended 3-D contact between "
                    f"{(u1, v1, k1, s1)!r} and {(u2, v2, k2, s2)!r}."
                )


### Shared positive-braid theta geometry and side-connection geometry
The braid geometry is identical for all three target families.

At the \(2m\) subdivision positions between consecutive crossings, the two braid strands are separated into a unique upper and lower vertex.  For the paired families, vertices are connected only in **non-overlapping consecutive pairs**.  The connecting arc is routed outside the braid strip (downward for lower pairs, upward for upper pairs), so the canonical projection introduces no additional crossings.

In [ ]:
def positive_two_braid_geometry(
    q: int,
    *,
    samples_per_crossing: int = BRAID_SAMPLES_PER_CROSSING,
):
    q = int(q)
    if q <= 0 or q % 2 == 0:
        raise ValueError("q must be a positive odd integer.")
    samples_per_crossing = int(samples_per_crossing)
    if samples_per_crossing < 5 or samples_per_crossing % 2 == 0:
        raise ValueError("BRAID_SAMPLES_PER_CROSSING must be an odd integer >=5.")

    y0 = 0.62
    depth = 0.24

    U = np.array([0.0, y0, 0.0])
    V = np.array([0.0, -y0, 0.0])

    x_left = 0.90
    x_right = x_left + float(q)

    count = q * samples_per_crossing + 1
    t = np.linspace(0.0, 1.0, count)
    knots = np.linspace(0.0, 1.0, q + 1)
    lane_values = y0 * ((-1.0) ** np.arange(q + 1))

    y_a = np.interp(t, knots, lane_values)
    z_a = depth * np.sin(np.pi * q * t)

    x = x_left + (x_right - x_left) * t
    braid_a = np.column_stack([x, y_a, z_a])
    braid_b = np.column_stack([x, -y_a, -z_a])

    top_right = np.array([x_right, y0, 0.0])
    bottom_right = np.array([x_right, -y0, 0.0])

    if not np.allclose(braid_a[-1], bottom_right):
        raise AssertionError("Odd positive braid A did not end on bottom lane.")
    if not np.allclose(braid_b[-1], top_right):
        raise AssertionError("Odd positive braid B did not end on top lane.")

    # Safe outside return corridors.
    #
    # The previous long Bezier returns were visually compact but, as q grew,
    # their xy projection entered the same band as the local side-pair arcs.
    # This produced accidental crossings.  These PL returns immediately leave
    # the repeated motif to x>x_right, travel in a far upper/lower corridor,
    # return at x<0, and then connect to U/V.  Hence they are disjoint in the
    # canonical xy projection from every local side-pair edge for all m.
    x_far_right = x_right + SAFE_RETURN_RIGHT_MARGIN
    x_far_left = SAFE_RETURN_LEFT_X

    top_return = np.array(
        [
            top_right,
            [x_far_right, y0, 0.0],
            [x_far_right, SAFE_RETURN_Y, 0.0],
            [x_far_left, SAFE_RETURN_Y, 0.0],
            [x_far_left, y0, 0.0],
            U,
        ],
        dtype=float,
    )

    bottom_return = np.array(
        [
            bottom_right,
            [x_far_right, -y0, 0.0],
            [x_far_right, -SAFE_RETURN_Y, 0.0],
            [x_far_left, -SAFE_RETURN_Y, 0.0],
            [x_far_left, -y0, 0.0],
            V,
        ],
        dtype=float,
    )

    left_u = line(U, braid_a[0], 9)
    left_v = line(V, braid_b[0], 9)

    phi = np.linspace(np.pi / 2, 3 * np.pi / 2, RETURN_SAMPLES)
    exterior = np.column_stack(
        [
            -0.78 * np.cos(phi - np.pi),
            y0 * np.sin(phi),
            np.full_like(phi, 0.38),
        ]
    )
    exterior[0] = U
    exterior[-1] = V

    return {
        "U": U,
        "V": V,
        "braid_a": braid_a,
        "braid_b": braid_b,
        "left_u": left_u,
        "left_v": left_v,
        "top_return": top_return,
        "bottom_return": bottom_return,
        "exterior": exterior,
        "samples_per_crossing": samples_per_crossing,
    }



def lower_upper_nodes_for_boundary(graph: nx.MultiGraph, j: int):
    """Return the lower and upper subdivision node at braid boundary j."""
    a = ("a", int(j))
    b = ("b", int(j))

    ya = float(graph.nodes[a]["pos"][1])
    yb = float(graph.nodes[b]["pos"][1])

    if ya <= yb:
        return a, b
    return b, a


def side_connection_arc(
    p0: np.ndarray,
    p1: np.ndarray,
    *,
    direction: str,
    pair_index: int,
) -> np.ndarray:
    """
    Local outside-the-braid connection used by the degree<=3 paired families.

    Endpoints have the same y-coordinate in the canonical projection.  The arc
    bows away from the braid strip and receives a tiny alternating z-offset so
    distinct 3-D edges stay separated even before projection.
    """
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)

    if direction not in {"down", "up"}:
        raise ValueError("direction must be 'down' or 'up'.")

    sign = -1.0 if direction == "down" else 1.0
    span = float(p1[0] - p0[0])

    if span <= 0:
        raise ValueError("side_connection_arc expects p1 to lie to the right of p0.")

    bow = 0.48
    zoff = 0.06 * (1.0 if pair_index % 2 else -1.0)

    c1 = p0 + np.array([0.28 * span, sign * bow, zoff])
    c2 = p1 + np.array([-0.28 * span, sign * bow, -zoff])

    return bezier_cubic(
        p0,
        c1,
        c2,
        p1,
        samples=SIDE_CONNECTION_SAMPLES,
    )


def add_nonoverlapping_side_pairs(
    graph: nx.MultiGraph,
    *,
    add_bottom: bool,
    add_top: bool,
) -> None:
    """
    Pair boundaries (1,2), (3,4), ..., (2m-1,2m).

    Because q=2m+1, q-1=2m is even.  Every selected subdivision vertex therefore
    receives exactly ONE added edge, keeping its total degree <=3.
    """
    q = int(graph.graph["q"])

    if (q - 1) % 2:
        raise AssertionError("Expected q-1 to be even.")

    for j in range(1, q - 1, 2):
        jp = j + 1
        pair_index = (j + 1) // 2

        lower_j, upper_j = lower_upper_nodes_for_boundary(graph, j)
        lower_p, upper_p = lower_upper_nodes_for_boundary(graph, jp)

        if add_bottom:
            p0 = np.asarray(graph.nodes[lower_j]["pos"], dtype=float)
            p1 = np.asarray(graph.nodes[lower_p]["pos"], dtype=float)
            graph.add_edge(
                lower_j,
                lower_p,
                pts=side_connection_arc(
                    p0,
                    p1,
                    direction="down",
                    pair_index=pair_index,
                ),
                role="bottom_pair_edge",
                pair_index=pair_index,
            )

        if add_top:
            p0 = np.asarray(graph.nodes[upper_j]["pos"], dtype=float)
            p1 = np.asarray(graph.nodes[upper_p]["pos"], dtype=float)
            graph.add_edge(
                upper_j,
                upper_p,
                pts=side_connection_arc(
                    p0,
                    p1,
                    direction="up",
                    pair_index=pair_index,
                ),
                role="top_pair_edge",
                pair_index=pair_index,
            )


### Common subdivided theta skeleton

In [ ]:
def subdivided_theta_skeleton(m: int) -> nx.MultiGraph:
    """
    Shared skeleton with vertices U, V and boundary vertices a_j, b_j, j=1..q-1.
    Constituent-cycle edges are subdivided at every boundary between consecutive crossings.
    """
    m = int(m)
    if m < 1:
        raise ValueError("m must be >=1.")

    q = 2 * m + 1
    geom = positive_two_braid_geometry(q)
    U = geom["U"]
    V = geom["V"]
    braid_a = geom["braid_a"]
    braid_b = geom["braid_b"]
    s = int(geom["samples_per_crossing"])

    graph = nx.MultiGraph()
    graph.add_node("U", pos=U.copy())
    graph.add_node("V", pos=V.copy())

    for j in range(1, q):
        index = j * s
        graph.add_node(("a", j), pos=braid_a[index].copy())
        graph.add_node(("b", j), pos=braid_b[index].copy())

    # Path A: U -> a1 -> ... -> a_(q-1) -> V
    first_a = join_paths(geom["left_u"], braid_a[: s + 1])
    graph.add_edge("U", ("a", 1), pts=first_a, role="constituent_cycle", path_family="A")

    for j in range(1, q - 1):
        pts = braid_a[j * s : (j + 1) * s + 1].copy()
        graph.add_edge(("a", j), ("a", j + 1), pts=pts, role="constituent_cycle", path_family="A")

    last_a = join_paths(braid_a[(q - 1) * s :], geom["bottom_return"])
    graph.add_edge(("a", q - 1), "V", pts=last_a, role="constituent_cycle", path_family="A")

    # Path B: U -> b_(q-1) -> ... -> b_1 -> V
    first_b = join_paths(geom["top_return"][::-1], braid_b[(q - 1) * s :][::-1])
    graph.add_edge("U", ("b", q - 1), pts=first_b, role="constituent_cycle", path_family="B")

    for j in range(q - 1, 1, -1):
        pts = braid_b[(j - 1) * s : j * s + 1][::-1].copy()
        graph.add_edge(("b", j), ("b", j - 1), pts=pts, role="constituent_cycle", path_family="B")

    last_b = join_paths(braid_b[: s + 1][::-1], geom["left_v"][::-1])
    graph.add_edge(("b", 1), "V", pts=last_b, role="constituent_cycle", path_family="B")

    graph.add_edge("U", "V", pts=geom["exterior"].copy(), role="exterior_theta_edge")

    graph.graph.update(
        parameter_name="m",
        m=m,
        q=q,
        constituent_knot=f"T(2,{q})",
        crossing_lower_bound=q,
        expected_canonical_crossings=q,
        expected_vertices=2 * q,
    )

    return ensure_embedding(graph, copy=False, normalize=True)


### Three target family constructors

In [ ]:
def crosslinked_theta_ladder(m: int) -> nx.MultiGraph:
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    for j in range(1, q):
        a = ("a", j)
        b = ("b", j)

        p = np.asarray(graph.nodes[a]["pos"], dtype=float)
        p_other = np.asarray(graph.nodes[b]["pos"], dtype=float)

        graph.add_edge(
            a,
            b,
            pts=line(p, p_other, 2),
            role="rung",
            rung_index=j,
        )

    graph.graph.update(
        family="crosslinked_theta_ladder",
        family_label="Cross-linked theta ladder Lambda_m",
        expected_edges=3 * q,
        expected_rungs=q - 1,
        expected_bottom_pair_edges=0,
        expected_top_pair_edges=0,
        expected_degree_histogram={3: 2 * q},
    )

    return ensure_embedding(graph, copy=False, normalize=True)


def bottom_paired_theta(m: int) -> nx.MultiGraph:
    """
    P_m^downarrow:
    pair consecutive LOWER boundary vertices non-overlappingly:
    (1,2), (3,4), ..., (2m-1,2m).
    """
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    add_nonoverlapping_side_pairs(
        graph,
        add_bottom=True,
        add_top=False,
    )

    # Base subdivided theta has 2q+1 edges; we add m=(q-1)/2 pair edges.
    pair_count = (q - 1) // 2

    graph.graph.update(
        family="bottom_paired_theta",
        family_label="Bottom-paired theta family P_down_m",
        expected_edges=2 * q + 1 + pair_count,
        expected_rungs=0,
        expected_bottom_pair_edges=pair_count,
        expected_top_pair_edges=0,
        # 2m lower boundary vertices + U,V are degree 3;
        # 2m upper boundary vertices remain degree 2.
        expected_degree_histogram={
            2: q - 1,
            3: q + 1,
        },
    )

    return ensure_embedding(graph, copy=False, normalize=True)


def two_sided_paired_theta(m: int) -> nx.MultiGraph:
    """
    P_m^{updownarrow}:
    add the same non-overlapping consecutive pairing on BOTH lower and upper sides.
    """
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    add_nonoverlapping_side_pairs(
        graph,
        add_bottom=True,
        add_top=True,
    )

    pair_count = (q - 1) // 2

    graph.graph.update(
        family="two_sided_paired_theta",
        family_label="Two-sided paired theta family P_updown_m",
        expected_edges=3 * q,
        expected_rungs=0,
        expected_bottom_pair_edges=pair_count,
        expected_top_pair_edges=pair_count,
        expected_degree_histogram={3: 2 * q},
    )

    return ensure_embedding(graph, copy=False, normalize=True)


### Known positive control: canonical Dobrynin–Vesnin theta family

In [ ]:
def reference_dv_theta_graph(n: int, samples_per_crossing: int = 9) -> nx.MultiGraph:
    n = int(n)
    if n < 0:
        raise ValueError("n must be >=0.")

    y0 = 0.56
    U = np.array([0.0, y0, 0.0])
    V = np.array([0.0, -y0, 0.0])

    x_left = 0.62
    x_right = 2.05 + 0.72 * max(n, 1)
    depth = 0.24

    if n == 0:
        t = np.linspace(0.0, 1.0, 24)
        y_a = np.full_like(t, y0)
        z_a = np.zeros_like(t)
    else:
        count = n * int(samples_per_crossing) + 1
        t = np.linspace(0.0, 1.0, count)
        knots = np.linspace(0.0, 1.0, n + 1)
        values = y0 * ((-1.0) ** np.arange(n + 1))
        y_a = np.interp(t, knots, values)
        z_a = depth * np.sin(np.pi * n * t)

    x = x_left + (x_right - x_left) * t
    braid_a = np.column_stack([x, y_a, z_a])
    braid_b = np.column_stack([x, -y_a, -z_a])

    top_right = np.array([x_right, y0, 0.0])
    bottom_right = np.array([x_right, -y0, 0.0])
    outer_y = 1.16

    top_return = bezier_cubic(
        top_right,
        [x_right + 0.46, outer_y, 0.0],
        [0.15, outer_y, 0.0],
        U,
        samples=max(70, 8 * max(n, 1)),
    )

    bottom_return = bezier_cubic(
        bottom_right,
        [x_right + 0.46, -outer_y, 0.0],
        [0.15, -outer_y, 0.0],
        V,
        samples=max(70, 8 * max(n, 1)),
    )

    phi = np.linspace(np.pi / 2, 3 * np.pi / 2, 70)
    exterior = np.column_stack(
        [
            -0.67 * np.cos(phi - np.pi),
            y0 * np.sin(phi),
            np.zeros_like(phi),
        ]
    )
    exterior[0] = U
    exterior[-1] = V

    graph = nx.MultiGraph()
    graph.add_node("u", pos=U.copy())
    graph.add_node("v", pos=V.copy())

    left_u = line(U, braid_a[0], 14)
    left_v = line(V, braid_b[0], 14)

    if n % 2:
        graph.add_edge("u", "v", pts=join_paths(left_u, braid_a, bottom_return), role="torus_a")
        graph.add_edge("u", "v", pts=join_paths(top_return[::-1], braid_b[::-1], left_v[::-1]), role="torus_b")
        graph.add_edge("u", "v", pts=exterior, role="exterior_arc")
        graph.graph["abstract_type"] = "theta"
    else:
        graph.add_edge("u", "u", pts=join_paths(left_u, braid_a, top_return), role="torus_component_u")
        graph.add_edge("v", "v", pts=join_paths(left_v, braid_b, bottom_return), role="torus_component_v")
        graph.add_edge("u", "v", pts=exterior, role="exterior_arc")
        graph.graph["abstract_type"] = "handcuff"

    graph.graph["expected_canonical_crossings"] = n
    return ensure_embedding(graph, copy=False, normalize=True)


### Hard certification
Every target family is certified to satisfy \(\Delta\le3\), to contain the same \(T(2,2m+1)\) constituent cycle, and to have the expected graph counts.

The expensive all-segment 3-D contact audit is **not** repeated for every \(m\).  Because the geometry is a translated repeated cell, it is run on representative values \(m=1,2,5,10\).  Each actual dataset row still receives the cheap exact combinatorial certification and the canonical projection is required to contain exactly \(q=2m+1\) crossings.

In [ ]:
def certify_connected(graph: nx.MultiGraph) -> None:
    if not nx.is_connected(nx.Graph(graph)):
        raise AssertionError("Graph is disconnected.")


def extract_constituent_cycle(graph: nx.MultiGraph) -> nx.MultiGraph:
    cycle = nx.MultiGraph()

    for node, data in graph.nodes(data=True):
        cycle.add_node(
            node,
            pos=np.asarray(data["pos"], dtype=float).copy(),
        )

    used_nodes = set()

    for u, v, key, data in graph.edges(keys=True, data=True):
        if data.get("role") != "constituent_cycle":
            continue

        cycle.add_edge(
            u,
            v,
            pts=np.asarray(data["pts"], dtype=float).copy(),
            role="constituent_cycle",
        )
        used_nodes.add(u)
        used_nodes.add(v)

    for node in list(cycle.nodes()):
        if node not in used_nodes:
            cycle.remove_node(node)

    return ensure_embedding(cycle, copy=False, normalize=True)


def certify_theta_derived_family(
    graph: nx.MultiGraph,
    *,
    require_cubic: bool,
    audit_3d_contacts: bool = False,
) -> None:
    certify_connected(graph)

    m = int(graph.graph["m"])
    q = int(graph.graph["q"])

    if q != 2 * m + 1:
        raise AssertionError("q != 2m+1.")

    if graph.number_of_nodes() != int(graph.graph["expected_vertices"]):
        raise AssertionError(
            f"Expected V={graph.graph['expected_vertices']}, "
            f"found {graph.number_of_nodes()}."
        )

    if graph.number_of_edges() != int(graph.graph["expected_edges"]):
        raise AssertionError(
            f"Expected E={graph.graph['expected_edges']}, "
            f"found {graph.number_of_edges()}."
        )

    role_counts = Counter(
        str(data.get("role", ""))
        for *_edge, data in graph.edges(data=True)
    )

    expected_role_counts = {
        "rung": int(graph.graph["expected_rungs"]),
        "bottom_pair_edge": int(graph.graph["expected_bottom_pair_edges"]),
        "top_pair_edge": int(graph.graph["expected_top_pair_edges"]),
    }

    for role, expected in expected_role_counts.items():
        actual = int(role_counts.get(role, 0))
        if actual != expected:
            raise AssertionError(
                f"Expected {expected} {role!r} edges, found {actual}."
            )

    degrees = [int(deg) for _, deg in graph.degree()]
    degree_histogram = dict(sorted(Counter(degrees).items()))

    if max(degrees) > 3:
        raise AssertionError(
            f"Subcubic requirement violated: Delta(G)={max(degrees)}."
        )

    expected_histogram = {
        int(k): int(v)
        for k, v in graph.graph["expected_degree_histogram"].items()
    }

    if degree_histogram != expected_histogram:
        raise AssertionError(
            "Degree histogram mismatch: "
            f"expected {expected_histogram}, found {degree_histogram}."
        )

    if require_cubic and any(deg != 3 for deg in degrees):
        raise AssertionError(
            f"Expected exactly cubic graph; histogram={degree_histogram}."
        )

    constituent = extract_constituent_cycle(graph)

    if not nx.is_connected(nx.Graph(constituent)):
        raise AssertionError("Constituent cycle is disconnected.")

    constituent_degrees = [int(deg) for _, deg in constituent.degree()]
    if any(deg != 2 for deg in constituent_degrees):
        raise AssertionError(
            "Constituent A∪B is not 2-regular; "
            f"histogram={dict(Counter(constituent_degrees))}."
        )

    # This zero-rotation constituent check is only used in representative preflight.
    if audit_3d_contacts:
        processor = PDCode(constituent)
        processor.compute(
            rotation_angles=(0.0, 0.0, 0.0),
            rotation_order=ROTATION_ORDER,
        )

        if len(processor.crossings) != q:
            raise AssertionError(
                f"Expected constituent to have q={q} canonical crossings, "
                f"detected {len(processor.crossings)}."
            )

        audit_piecewise_linear_embedding(graph)


# Representative expensive geometry audit only.
if RUN_REPRESENTATIVE_3D_CONTACT_AUDIT:
    for m in REPRESENTATIVE_AUDIT_M:
        for builder, cubic in [
            (crosslinked_theta_ladder, True),
            (bottom_paired_theta, False),
            (two_sided_paired_theta, True),
        ]:
            g = builder(m)
            certify_theta_derived_family(
                g,
                require_cubic=cubic,
                audit_3d_contacts=True,
            )

    print(
        "PASS: representative 3-D contact/constituent audits completed for m =",
        REPRESENTATIVE_AUDIT_M,
    )

# Cheap structural preflight at a few larger values.
for m in sorted(set([1, 2, 5, 10, min(25, MAX_DISCOVERY_M), MAX_DISCOVERY_M])):
    if m < 1:
        continue

    for builder, cubic in [
        (crosslinked_theta_ladder, True),
        (bottom_paired_theta, False),
        (two_sided_paired_theta, True),
    ]:
        g = builder(m)
        certify_theta_derived_family(
            g,
            require_cubic=cubic,
            audit_3d_contacts=False,
        )

for n in range(3, 21):
    dv = reference_dv_theta_graph(n)
    certify_connected(dv)

print("PASS: all three target constructors are certified Delta(G)<=3.")


### Canonical-projection regression at large \(m\)
Before the long invariant sweep, project representative large family members
(including the held-out-test scale) and require the **full graph** to contain
exactly \(q=2m+1\) crossings.  This catches embedding-constructor errors before
Yamada evaluation.

In [ ]:
if RUN_CANONICAL_PROJECTION_PREFLIGHT:
    builders = [
        ("crosslinked_theta_ladder", crosslinked_theta_ladder),
        ("bottom_paired_theta", bottom_paired_theta),
        ("two_sided_paired_theta", two_sided_paired_theta),
    ]

    for m in CANONICAL_PREFLIGHT_M:
        q = 2 * int(m) + 1

        for family_name, builder in builders:
            graph = builder(int(m))

            processor = PDCode(graph)
            processor.compute(
                rotation_angles=(0.0, 0.0, 0.0),
                rotation_order=ROTATION_ORDER,
            )

            actual = len(processor.crossings)

            if actual != q:
                raise AssertionError(
                    f"Canonical-projection preflight failed for "
                    f"{family_name}, m={m}: expected q={q}, got {actual}."
                )

            print(
                f"PASS canonical projection: "
                f"{family_name:30s} m={m:3d} crossings={actual:3d}"
            )

    print(
        "PASS: safe-return geometry preserves exactly q=2m+1 canonical "
        "crossings through m=200."
    )


### Fast canonical projection
These families are defined by the displayed canonical braid diagrams.  Therefore the dataset does **not** search over random rotations.  It computes the zero-rotation PD code once, asserts that the target families have exactly \(q=2m+1\) crossings, and evaluates Yamada.

This removes the largest avoidable wall-time component of the earlier implementation.  Optional frontier diagnostics can be re-enabled with `COMPUTE_FRONTIER_DIAGNOSTICS=True`.

In [ ]:
def factorized_frontier_width(processor) -> dict:
    """Optional diagnostic only; disabled by default for maximum throughput."""
    yamada = Yamada(
        vertices=list(processor.vertices.values()),
        crossings=list(processor.crossings.values()),
        arcs=list(processor.arcs.values()),
    )

    prepared = yamada._prepare_compact_state_builder()
    data = build_factorized_frontier(prepared)

    factor_count = len(data["factor_types"])
    ports_by_factor = [[] for _ in range(factor_count)]

    for port, factor in enumerate(data["port_factor"]):
        ports_by_factor[int(factor)].append(port)

    active = []
    processed = set()
    peak_live_ports = 0
    max_boundary_ports = 0

    for factor in data["factor_order"]:
        factor = int(factor)
        active.extend(ports_by_factor[factor])
        peak_live_ports = max(peak_live_ports, len(active))
        processed.add(factor)

        active = [
            port
            for port in active
            if int(
                data["port_factor"][
                    int(data["wire_partner"][port])
                ]
            )
            not in processed
        ]

        max_boundary_ports = max(
            max_boundary_ports,
            len(active),
        )

    if active:
        raise RuntimeError("Factorized frontier planner did not close.")

    return {
        "crossings_after_RII": len(prepared.crossing_ids),
        "peak_live_ports": int(peak_live_ports),
        "max_boundary_ports": int(max_boundary_ports),
        "factor_count": int(factor_count),
    }


def embedding_hash(graph: nx.MultiGraph) -> str:
    payload = []

    for node, data in sorted(
        graph.nodes(data=True),
        key=lambda item: repr(item[0]),
    ):
        payload.append(
            (
                "node",
                repr(node),
                np.asarray(data["pos"], dtype=float).round(12).tolist(),
            )
        )

    edge_payload = []

    for u, v, key, data in graph.edges(keys=True, data=True):
        edge_payload.append(
            (
                repr(u),
                repr(v),
                int(key),
                str(data.get("role", "")),
                np.asarray(data["pts"], dtype=float).round(12).tolist(),
            )
        )

    payload.extend(
        ("edge", *entry)
        for entry in sorted(
            edge_payload,
            key=lambda item: (item[0], item[1], item[2], item[3]),
        )
    )

    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()


def projection_cache_key(
    *,
    family: str,
    parameter_value: int,
    graph_hash: str,
) -> tuple[str, Path]:
    payload = {
        "schema": 2,
        "constructor_version": CONSTRUCTOR_VERSION,
        "git_commit": GIT_COMMIT,
        "family": family,
        "parameter_value": int(parameter_value),
        "graph_hash": graph_hash,
        "rotation_order": ROTATION_ORDER,
        "canonical_zero": True,
        "frontier_diagnostics": COMPUTE_FRONTIER_DIAGNOSTICS,
    }

    key = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()

    safe_family = family.replace("/", "_")

    path = (
        PROJECTION_CACHE_DIR
        / f"{safe_family}__p{parameter_value}__{key[:20]}.pkl"
    )

    return key, path


def _processor_payload(processor) -> dict:
    return {
        "vertices": dict(processor.vertices),
        "crossings": dict(processor.crossings),
        "arcs": dict(processor.arcs),
    }


def _processor_from_payload(payload: dict):
    return SimpleNamespace(
        vertices=dict(payload["vertices"]),
        crossings=dict(payload["crossings"]),
        arcs=dict(payload["arcs"]),
    )


def save_projection_cache(
    path: Path,
    *,
    cache_key: str,
    graph_hash: str,
    projection,
    frontier: dict,
) -> None:
    if not WRITE_PROJECTION_CACHE:
        return

    payload = {
        "schema": 2,
        "cache_key": cache_key,
        "graph_hash": graph_hash,
        "rotation_angles": projection.rotation_angles,
        "rotation_order": projection.rotation_order,
        "pd_code": projection.pd_code,
        "num_crossings": int(projection.num_crossings),
        "frontier": dict(frontier),
        **_processor_payload(projection.processor),
    }

    tmp = path.with_suffix(path.suffix + ".tmp")

    with tmp.open("wb") as handle:
        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
        handle.flush()
        os.fsync(handle.fileno())

    tmp.replace(path)


def load_projection_cache(
    path: Path,
    *,
    cache_key: str,
    graph_hash: str,
):
    if not REUSE_PROJECTION_CACHE or not path.exists():
        return None

    try:
        with path.open("rb") as handle:
            payload = pickle.load(handle)

        if payload.get("schema") != 2:
            return None
        if payload.get("cache_key") != cache_key:
            return None
        if payload.get("graph_hash") != graph_hash:
            return None

        processor = _processor_from_payload(payload)

        return {
            "processor": processor,
            "rotation_angles": payload["rotation_angles"],
            "rotation_order": payload["rotation_order"],
            "pd_code": payload["pd_code"],
            "num_crossings": int(payload["num_crossings"]),
            "frontier": dict(payload.get("frontier", {})),
            "source": "cache",
            "candidate_records": [],
        }

    except Exception as exc:
        print(
            f"Ignoring stale/broken projection cache {path.name}: "
            f"{type(exc).__name__}: {exc}"
        )
        return None


def choose_projection(
    graph: nx.MultiGraph,
    *,
    family: str,
    parameter_value: int,
):
    """
    Fast path: one canonical zero-rotation projection only.

    Projection independence is already covered by the repository sanity suite;
    here the canonical projection is part of the family definition.
    """
    graph_hash = embedding_hash(graph)

    cache_key, cache_path = projection_cache_key(
        family=family,
        parameter_value=parameter_value,
        graph_hash=graph_hash,
    )

    cached = load_projection_cache(
        cache_path,
        cache_key=cache_key,
        graph_hash=graph_hash,
    )

    if cached is not None:
        cached["cache_key"] = cache_key
        cached["graph_hash"] = graph_hash
        return cached

    start = time.perf_counter()

    projection = select_projection(
        graph,
        rotation_angles=(0.0, 0.0, 0.0),
        rotation_order=ROTATION_ORDER,
        num_rotation_samples=1,
    )

    if COMPUTE_FRONTIER_DIAGNOSTICS:
        frontier = factorized_frontier_width(projection.processor)
    else:
        frontier = {}

    projection_seconds = time.perf_counter() - start

    save_projection_cache(
        cache_path,
        cache_key=cache_key,
        graph_hash=graph_hash,
        projection=projection,
        frontier=frontier,
    )

    return {
        "processor": projection.processor,
        "rotation_angles": projection.rotation_angles,
        "rotation_order": projection.rotation_order,
        "pd_code": projection.pd_code,
        "num_crossings": int(projection.num_crossings),
        "frontier": dict(frontier),
        "candidate_records": [
            {
                "angles": [0.0, 0.0, 0.0],
                "crossings": int(projection.num_crossings),
                "canonical": True,
            }
        ],
        "source": "computed",
        "projection_seconds": projection_seconds,
        "cache_key": cache_key,
        "graph_hash": graph_hash,
    }


### Exact Laurent representation

In [ ]:
def laurent_coefficients_exact(
    expr: sp.Expr,
    variable: sp.Symbol,
) -> dict[int, int]:
    """
    Fast exact Laurent extraction.

    The Yamada result is already an expanded Laurent polynomial.  Avoiding
    sp.cancel() on the whole expression substantially reduces post-processing
    time for large m.
    """
    expr = sp.expand(expr)

    if expr == 0:
        return {}

    coefficients: dict[int, sp.Expr] = {}

    for term in sp.Add.make_args(expr):
        coefficient, exponent = term.as_coeff_exponent(variable)

        if exponent.is_Integer is not True:
            raise ValueError(
                f"Noninteger Laurent exponent in term {term!r}."
            )

        if variable in coefficient.free_symbols:
            raise ValueError(
                f"Could not isolate Laurent coefficient in term {term!r}."
            )

        exponent = int(exponent)
        coefficients[exponent] = (
            coefficients.get(exponent, sp.Integer(0))
            + coefficient
        )

    result: dict[int, int] = {}

    for exponent, coefficient in sorted(coefficients.items()):
        coefficient = sp.expand(coefficient)

        if coefficient == 0:
            continue

        if coefficient.is_Integer is not True:
            raise ValueError(
                f"Noninteger Laurent coefficient: {coefficient!r}."
            )

        result[int(exponent)] = int(coefficient)

    return result


def exact_same(left: sp.Expr, right: sp.Expr) -> bool:
    return sp.expand(left - right) == 0


### Family registry

In [ ]:
@dataclass(frozen=True)
class FamilySpec:
    key: str
    label: str
    parameter_name: str
    builder: Callable[[int], nx.MultiGraph]
    family_definition: str
    spatial_definition: str
    literature_role: str
    require_cubic: bool
    has_crossing_lower_bound: bool
    discovery_values: tuple[int, ...]
    heldout_values: tuple[int, ...]


FAMILIES = [
    FamilySpec(
        key="dv_theta",
        label="Dobrynin–Vesnin Theta(n)",
        parameter_name="n",
        builder=reference_dv_theta_graph,
        family_definition=(
            "Published Dobrynin–Vesnin two-strand Theta(n) spatial family."
        ),
        spatial_definition=(
            "Canonical n-crossing two-strand diagram plus exterior theta edge."
        ),
        literature_role="known_formula_positive_control",
        require_cubic=False,
        has_crossing_lower_bound=False,
        discovery_values=tuple(DV_DISCOVERY_N),
        heldout_values=tuple(DV_HELDOUT_N),
    ),
    FamilySpec(
        key="crosslinked_theta_ladder",
        label="Cross-linked theta ladder Lambda_m",
        parameter_name="m",
        builder=crosslinked_theta_ladder,
        family_definition=(
            "At every braid boundary, connect the upper and lower subdivision "
            "vertices by one rung."
        ),
        spatial_definition=(
            "Constituent T(2,2m+1) plus exterior theta edge plus 2m rungs."
        ),
        literature_role="new_formula_discovery_target",
        require_cubic=True,
        has_crossing_lower_bound=True,
        discovery_values=tuple(MAIN_DISCOVERY_M),
        heldout_values=tuple(MAIN_HELDOUT_M),
    ),
    FamilySpec(
        key="bottom_paired_theta",
        label="Bottom-paired theta family P_down_m",
        parameter_name="m",
        builder=bottom_paired_theta,
        family_definition=(
            "Pair consecutive lower braid-boundary vertices non-overlappingly: "
            "(1,2),(3,4),...,(2m-1,2m)."
        ),
        spatial_definition=(
            "Constituent T(2,2m+1) plus exterior theta edge plus m local "
            "bottom-side pairing arcs; maximum degree is 3."
        ),
        literature_role="new_formula_discovery_target",
        require_cubic=False,
        has_crossing_lower_bound=True,
        discovery_values=tuple(MAIN_DISCOVERY_M),
        heldout_values=tuple(MAIN_HELDOUT_M),
    ),
    FamilySpec(
        key="two_sided_paired_theta",
        label="Two-sided paired theta family P_updown_m",
        parameter_name="m",
        builder=two_sided_paired_theta,
        family_definition=(
            "Apply the same non-overlapping consecutive pairing on both the "
            "lower and upper braid-boundary vertices."
        ),
        spatial_definition=(
            "Constituent T(2,2m+1) plus exterior theta edge plus m lower and "
            "m upper local pairing arcs; the graph is cubic."
        ),
        literature_role="new_formula_discovery_target",
        require_cubic=True,
        has_crossing_lower_bound=True,
        discovery_values=tuple(MAIN_DISCOVERY_M),
        heldout_values=tuple(MAIN_HELDOUT_M),
    ),
]

SPEC_BY_KEY = {
    spec.key: spec
    for spec in FAMILIES
}


def requested_values(spec: FamilySpec) -> list[int]:
    values = set(spec.discovery_values)

    if COMPUTE_HELDOUT_NOW:
        values |= set(spec.heldout_values)

    return sorted(values)


def split_for(spec: FamilySpec, value: int) -> str:
    if value in spec.discovery_values:
        return "discovery"

    if value in spec.heldout_values:
        return "heldout_test"

    raise ValueError(
        f"{spec.key}, {spec.parameter_name}={value} belongs to no split."
    )


### Independent exact Yamada evaluation
Each row is independently constructed, projected, and evaluated by KnottedGraph's compiled production Yamada backend.  No earlier \(m\), recurrence, transfer matrix, or candidate formula is used.

The timing columns separate construction, certification, projection, optimized Yamada contraction, Laurent post-processing, and total wall time so any remaining bottleneck is visible.

In [ ]:
CSV_FIELDS = [
    "case_id",
    "split",
    "family",
    "family_label",
    "parameter_name",
    "parameter_value",
    "m",
    "q",
    "family_definition",
    "spatial_definition",
    "literature_role",
    "branch",
    "git_commit",
    "constructor_version",
    "run_config_sha256",
    "compute_config_sha256",
    "embedding_sha256",
    "projection_cache_key",
    "projection_source",
    "rotation_angles_json",
    "rotation_order",
    "candidate_projection_scores_json",
    "vertices",
    "edges",
    "components",
    "beta1",
    "max_degree",
    "is_subcubic",
    "is_cubic",
    "degree_histogram_json",
    "constituent_knot",
    "crossing_lower_bound",
    "selected_crossings",
    "canonical_crossing_count_pass",
    "crossing_lower_bound_pass",
    "crossings_after_RII",
    "factorized_peak_live_ports",
    "factorized_max_boundary_ports",
    "factorized_factor_count",
    "pd_code",
    "yamada_A_normalized",
    "laurent_min_exponent",
    "laurent_max_exponent",
    "laurent_span",
    "laurent_term_count",
    "laurent_coefficients_json",
    "construction_seconds",
    "certification_seconds",
    "projection_seconds",
    "yamada_seconds",
    "postprocess_seconds",
    "case_seconds",
    "wall_minus_yamada_seconds",
    "status",
    "error_type",
    "error_message",
]


def graph_summary(graph: nx.MultiGraph) -> dict:
    degrees = [int(deg) for _, deg in graph.degree()]
    components = nx.number_connected_components(nx.Graph(graph))
    beta1 = (
        graph.number_of_edges()
        - graph.number_of_nodes()
        + components
    )

    return {
        "vertices": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "components": components,
        "beta1": beta1,
        "max_degree": max(degrees),
        "is_subcubic": max(degrees) <= 3,
        "is_cubic": all(deg == 3 for deg in degrees),
        "degree_histogram_json": json.dumps(
            dict(sorted(Counter(degrees).items())),
            separators=(",", ":"),
        ),
    }


def _frontier_field(frontier: dict, key: str):
    value = frontier.get(key, "")
    return value if value != "" else ""


def evaluate_family_case(
    spec: FamilySpec,
    parameter_value: int,
) -> dict:
    total_start = time.perf_counter()
    value = int(parameter_value)

    # ---------------- construction ----------------
    stage = time.perf_counter()
    graph = spec.builder(value)
    construction_seconds = time.perf_counter() - stage

    # ---------------- cheap structural certification ----------------
    stage = time.perf_counter()
    certify_connected(graph)

    if spec.key in {
        "crosslinked_theta_ladder",
        "bottom_paired_theta",
        "two_sided_paired_theta",
    }:
        certify_theta_derived_family(
            graph,
            require_cubic=spec.require_cubic,
            audit_3d_contacts=False,
        )

    summary = graph_summary(graph)
    certification_seconds = time.perf_counter() - stage

    # ---------------- canonical projection ----------------
    stage = time.perf_counter()
    chosen = choose_projection(
        graph,
        family=spec.key,
        parameter_value=value,
    )
    projection_seconds = time.perf_counter() - stage

    canonical_crossing_count_pass = ""

    if spec.key in {
        "crosslinked_theta_ladder",
        "bottom_paired_theta",
        "two_sided_paired_theta",
    }:
        expected_crossings = int(graph.graph["expected_canonical_crossings"])
        canonical_crossing_count_pass = (
            int(chosen["num_crossings"])
            == expected_crossings
        )

        if not canonical_crossing_count_pass:
            raise AssertionError(
                f"{spec.key}, m={value}: canonical projection has "
                f"{chosen['num_crossings']} crossings; expected exactly "
                f"{expected_crossings}. This indicates an accidental extra "
                "crossing or a lost braid crossing."
            )

    lower_bound = (
        int(graph.graph["crossing_lower_bound"])
        if spec.has_crossing_lower_bound
        else ""
    )

    lower_bound_pass = ""

    if spec.has_crossing_lower_bound:
        lower_bound_pass = (
            int(chosen["num_crossings"])
            >= int(lower_bound)
        )

        if not lower_bound_pass:
            raise AssertionError(
                f"{spec.key}, {spec.parameter_name}={value}: selected projection "
                f"has {chosen['num_crossings']} crossings, below constituent-knot "
                f"lower bound {lower_bound}."
            )

    # ---------------- optimized exact Yamada ----------------
    stage = time.perf_counter()

    computer = Yamada(
        vertices=list(chosen["processor"].vertices.values()),
        crossings=list(chosen["processor"].crossings.values()),
        arcs=list(chosen["processor"].arcs.values()),
    )

    polynomial = sp.expand(
        computer.compute(
            A,
            normalize=NORMALIZE_YAMADA,
        )
    )

    yamada_seconds = time.perf_counter() - stage

    # ---------------- exact polynomial post-processing ----------------
    stage = time.perf_counter()

    coeffs = laurent_coefficients_exact(
        polynomial,
        A,
    )

    min_exp = min(coeffs) if coeffs else ""
    max_exp = max(coeffs) if coeffs else ""
    span = max_exp - min_exp if coeffs else 0

    polynomial_text = sp.sstr(polynomial)
    coefficients_text = json.dumps(
        coeffs,
        sort_keys=True,
        separators=(",", ":"),
    )

    postprocess_seconds = time.perf_counter() - stage
    case_seconds = time.perf_counter() - total_start

    frontier = chosen.get("frontier", {})
    q_value = (
        int(graph.graph.get("q", ""))
        if graph.graph.get("q", "") != ""
        else ""
    )
    m_value = value if spec.parameter_name == "m" else ""

    return {
        "case_id": f"{spec.key}__{spec.parameter_name}{value}",
        "split": split_for(spec, value),
        "family": spec.key,
        "family_label": spec.label,
        "parameter_name": spec.parameter_name,
        "parameter_value": value,
        "m": m_value,
        "q": q_value,
        "family_definition": spec.family_definition,
        "spatial_definition": spec.spatial_definition,
        "literature_role": spec.literature_role,
        "branch": CURRENT_BRANCH,
        "git_commit": GIT_COMMIT,
        "constructor_version": CONSTRUCTOR_VERSION,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "compute_config_sha256": COMPUTE_CONFIG_SHA256,
        "embedding_sha256": chosen["graph_hash"],
        "projection_cache_key": chosen["cache_key"],
        "projection_source": chosen["source"],
        "rotation_angles_json": json.dumps(
            (
                None
                if chosen["rotation_angles"] is None
                else [
                    float(x)
                    for x in chosen["rotation_angles"]
                ]
            ),
            separators=(",", ":"),
        ),
        "rotation_order": chosen["rotation_order"],
        "candidate_projection_scores_json": json.dumps(
            chosen.get("candidate_records", []),
            separators=(",", ":"),
        ),
        **summary,
        "constituent_knot": graph.graph.get("constituent_knot", ""),
        "crossing_lower_bound": lower_bound,
        "selected_crossings": int(chosen["num_crossings"]),
        "canonical_crossing_count_pass": canonical_crossing_count_pass,
        "crossing_lower_bound_pass": lower_bound_pass,
        "crossings_after_RII": _frontier_field(
            frontier,
            "crossings_after_RII",
        ),
        "factorized_peak_live_ports": _frontier_field(
            frontier,
            "peak_live_ports",
        ),
        "factorized_max_boundary_ports": _frontier_field(
            frontier,
            "max_boundary_ports",
        ),
        "factorized_factor_count": _frontier_field(
            frontier,
            "factor_count",
        ),
        "pd_code": chosen["pd_code"],
        "yamada_A_normalized": polynomial_text,
        "laurent_min_exponent": min_exp,
        "laurent_max_exponent": max_exp,
        "laurent_span": span,
        "laurent_term_count": len(coeffs),
        "laurent_coefficients_json": coefficients_text,
        "construction_seconds": construction_seconds,
        "certification_seconds": certification_seconds,
        "projection_seconds": projection_seconds,
        "yamada_seconds": yamada_seconds,
        "postprocess_seconds": postprocess_seconds,
        "case_seconds": case_seconds,
        "wall_minus_yamada_seconds": case_seconds - yamada_seconds,
        "status": "success",
        "error_type": "",
        "error_message": "",
    }


def read_csv(path: Path) -> list[dict]:
    if not path.exists():
        return []

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:
        return list(csv.DictReader(handle))


def atomic_write_rows(
    rows: list[dict],
    path: Path,
    *,
    fields: list[str],
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            extrasaction="ignore",
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(
                {
                    field: row.get(field, "")
                    for field in fields
                }
            )

        handle.flush()
        os.fsync(handle.fileno())

    tmp.replace(path)


family_rank = {
    spec.key: i
    for i, spec in enumerate(FAMILIES)
}


def sort_rows(rows: list[dict]) -> list[dict]:
    return sorted(
        rows,
        key=lambda row: (
            family_rank.get(
                row.get("family", ""),
                999,
            ),
            int(
                row.get(
                    "parameter_value",
                    0,
                )
            ),
        ),
    )


### Generate the discovery dataset
The full/share CSVs are checkpointed every `SAVE_EVERY_CASES` newly computed rows rather than after every case.  A manual `KeyboardInterrupt` always flushes all completed rows first.

In [ ]:
existing = (
    read_csv(FULL_CSV)
    if RESUME
    else []
)

rows_by_id = {
    row["case_id"]: row
    for row in existing
}

for case_id, row in list(rows_by_id.items()):
    if row.get("status") != "success":
        continue

    if row.get("branch") != CURRENT_BRANCH:
        raise RuntimeError(
            f"Existing row {case_id} is from another branch."
        )

    if row.get("git_commit") != GIT_COMMIT:
        raise RuntimeError(
            f"Existing row {case_id} is from commit "
            f"{row.get('git_commit')}; current commit is {GIT_COMMIT}. "
            "Rename/remove the old CSV."
        )

    if row.get("constructor_version") != CONSTRUCTOR_VERSION:
        raise RuntimeError(
            f"Existing row {case_id} uses another constructor version."
        )


SHARE_FIELDS = [
    "family",
    "family_label",
    "parameter_name",
    "parameter_value",
    "m",
    "q",
    "family_definition",
    "spatial_definition",
    "literature_role",
    "constituent_knot",
    "crossing_lower_bound",
    "selected_crossings",
    "max_degree",
    "is_subcubic",
    "is_cubic",
    "yamada_A_normalized",
    "laurent_min_exponent",
    "laurent_max_exponent",
    "laurent_span",
    "laurent_term_count",
    "laurent_coefficients_json",
]


def refresh_share_csv() -> list[dict]:
    discovery_rows = []

    for row in rows_by_id.values():
        if row.get("status") != "success":
            continue

        spec = SPEC_BY_KEY.get(
            row.get("family", "")
        )

        if spec is None:
            continue

        value = int(
            row["parameter_value"]
        )

        if value not in spec.discovery_values:
            continue

        discovery_rows.append(row)

    atomic_write_rows(
        sort_rows(discovery_rows),
        SHARE_CSV,
        fields=SHARE_FIELDS,
    )

    return discovery_rows


def flush_ledgers() -> list[dict]:
    atomic_write_rows(
        sort_rows(
            list(rows_by_id.values())
        ),
        FULL_CSV,
        fields=CSV_FIELDS,
    )

    return refresh_share_csv()


# Initial checkpoint.
flush_ledgers()

failures = []
new_cases_since_flush = 0

for spec in FAMILIES:
    print(
        f"\n=== {spec.label} ===",
        flush=True,
    )

    for value in requested_values(spec):
        case_id = (
            f"{spec.key}__"
            f"{spec.parameter_name}{value}"
        )

        previous = rows_by_id.get(
            case_id
        )

        reusable = (
            RESUME
            and previous is not None
            and previous.get("status") == "success"
            and previous.get("branch") == CURRENT_BRANCH
            and previous.get("git_commit") == GIT_COMMIT
            and previous.get("constructor_version") == CONSTRUCTOR_VERSION
        )

        if reusable:
            print(
                f"SKIP {case_id:46s} "
                f"crossings={previous['selected_crossings']} "
                f"Yamada={float(previous['yamada_seconds']):.4g}s "
                f"wall={float(previous['case_seconds']):.4g}s",
                flush=True,
            )
            continue

        try:
            row = evaluate_family_case(
                spec,
                value,
            )

            rows_by_id[case_id] = row
            new_cases_since_flush += 1

            family_done_values = [
                int(r["parameter_value"])
                for r in rows_by_id.values()
                if (
                    r.get("status") == "success"
                    and r.get("family") == spec.key
                    and int(r["parameter_value"]) in spec.discovery_values
                )
            ]

            max_done = max(
                family_done_values,
                default=None,
            )

            print(
                f"PASS {case_id:46s} "
                f"V={row['vertices']:4d} "
                f"crossings={row['selected_crossings']:4d} "
                f"terms={row['laurent_term_count']:4d} "
                f"proj={row['projection_seconds']:.4g}s "
                f"Yamada={row['yamada_seconds']:.4g}s "
                f"post={row['postprocess_seconds']:.4g}s "
                f"wall={row['case_seconds']:.4g}s "
                f"| max discovery parameter={max_done}",
                flush=True,
            )

            if new_cases_since_flush >= SAVE_EVERY_CASES:
                flush_ledgers()
                new_cases_since_flush = 0
                print(
                    "  checkpoint saved",
                    flush=True,
                )

        except KeyboardInterrupt:
            flush_ledgers()

            print(
                "\nInterrupted by user. All completed exact rows and "
                "the current share CSV have been saved.",
                flush=True,
            )

            raise

        except Exception as exc:
            error_row = {
                "case_id": case_id,
                "split": split_for(
                    spec,
                    value,
                ),
                "family": spec.key,
                "family_label": spec.label,
                "parameter_name": spec.parameter_name,
                "parameter_value": value,
                "m": (
                    value
                    if spec.parameter_name == "m"
                    else ""
                ),
                "q": (
                    2 * value + 1
                    if spec.parameter_name == "m"
                    else ""
                ),
                "family_definition": spec.family_definition,
                "spatial_definition": spec.spatial_definition,
                "literature_role": spec.literature_role,
                "branch": CURRENT_BRANCH,
                "git_commit": GIT_COMMIT,
                "constructor_version": CONSTRUCTOR_VERSION,
                "run_config_sha256": RUN_CONFIG_SHA256,
                "compute_config_sha256": COMPUTE_CONFIG_SHA256,
                "status": "error",
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }

            rows_by_id[case_id] = error_row
            failures.append(
                (
                    case_id,
                    type(exc).__name__,
                    str(exc),
                )
            )
            new_cases_since_flush += 1

            print(
                f"FAIL {case_id}: "
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )

            if new_cases_since_flush >= SAVE_EVERY_CASES:
                flush_ledgers()
                new_cases_since_flush = 0

            if FAIL_FAST:
                flush_ledgers()
                raise

    # Always checkpoint at a family boundary.
    flush_ledgers()
    new_cases_since_flush = 0

print("\nFull dataset:", FULL_CSV.resolve())
print("Current share data:", SHARE_CSV.resolve())

if failures:
    print(
        f"\nWARNING: {len(failures)} case(s) failed, but every successful "
        "discovery row has been checkpointed."
    )

    for case_id, kind, message in failures:
        print(
            f"  {case_id}: {kind}: {message}"
        )


### Export blind discovery / held-out files

In [ ]:
full_rows = read_csv(
    FULL_CSV
)

SHARE_FIELDS = [
    "family",
    "family_label",
    "parameter_name",
    "parameter_value",
    "m",
    "q",
    "family_definition",
    "spatial_definition",
    "literature_role",
    "constituent_knot",
    "crossing_lower_bound",
    "selected_crossings",
    "max_degree",
    "is_subcubic",
    "is_cubic",
    "yamada_A_normalized",
    "laurent_min_exponent",
    "laurent_max_exponent",
    "laurent_span",
    "laurent_term_count",
    "laurent_coefficients_json",
]

HELDOUT_FIELDS = SHARE_FIELDS + [
    "case_id",
    "branch",
    "git_commit",
    "embedding_sha256",
    "pd_code",
    "construction_seconds",
    "projection_seconds",
    "yamada_seconds",
    "postprocess_seconds",
    "case_seconds",
]

discovery_rows = []
heldout_rows = []

for row in full_rows:
    if row.get("status") != "success":
        continue

    spec = SPEC_BY_KEY.get(
        row.get("family", "")
    )

    if spec is None:
        continue

    value = int(
        row["parameter_value"]
    )

    if value in spec.discovery_values:
        discovery_rows.append(row)

    if (
        COMPUTE_HELDOUT_NOW
        and value in spec.heldout_values
    ):
        heldout_rows.append(row)

atomic_write_rows(
    sort_rows(discovery_rows),
    SHARE_CSV,
    fields=SHARE_FIELDS,
)

atomic_write_rows(
    sort_rows(heldout_rows),
    HELDOUT_CSV,
    fields=HELDOUT_FIELDS,
)

print(
    f"SHARE DATASET: {SHARE_CSV.resolve()} "
    f"({len(discovery_rows)} completed rows)"
)

print(
    f"HELD-OUT PRECOMPUTED: {len(heldout_rows)} "
    f"(COMPUTE_HELDOUT_NOW={COMPUTE_HELDOUT_NOW})"
)

for spec in FAMILIES:
    done = sorted(
        int(row["parameter_value"])
        for row in discovery_rows
        if row["family"] == spec.key
    )

    requested = list(
        spec.discovery_values
    )

    missing = [
        value
        for value in requested
        if value not in set(done)
    ]

    print(
        f"{spec.key:30s} "
        f"done={len(done):3d}/{len(requested):3d}; "
        f"max_done={max(done) if done else None}; "
        f"remaining={len(missing)}"
    )


### Export a formula-discovery dataset
After the discovery sweep has completed as far as practical, use the share CSV as the public fitting dataset. It contains the exact rows needed to search for Laurent-polynomial recurrences, factorizations, rational generating functions, parity sectors, or finite exponential-sum forms.

Freeze any machine-evaluable SymPy candidates before computing the held-out values. All three target families satisfy \(\Delta\le3\), share the constituent \(T(2,2m+1)\), and have canonical crossing count \(2m+1\).

## Held-out exact test after formulas are proposed

In [ ]:
# --------------------------------------------------------------------
# PASTE FROZEN FORMULA CANDIDATES HERE ONLY AFTER THE SHARE_CSV HAS BEEN EXPORTED.
# --------------------------------------------------------------------
#
# Example syntax only:
#
# FORMULA_CANDIDATES = {
#     "dv_theta": lambda n, A: ...,
#     "crosslinked_theta_ladder": lambda m, A: ...,
#     "bottom_paired_theta": lambda m, A: ...,
#     "two_sided_paired_theta": lambda m, A: ...,
# }

FORMULA_CANDIDATES = {
    "crosslinked_theta_ladder": lambda m, A: (
        (A**2 + 1)**(2*m + 1)
        - A**(2*m)
          * (A**4 + A**2 + 1)
          * (A**2 - A + 1)**(2*m)
        - A**(8*m + 2)
          * (A**4 + A**3 + A**2 + A + 1)
    ),

    "bottom_paired_theta": lambda m, A: (
        (A**2 + 1)**(m + 1)
        - A**(2*m)
          * (A**4 + A**2 + 1)
          * (A**2 - A + 1)**m
        + (-1)**(m + 1)
          * A**(7*m + 2)
          * (A**4 + A**3 + A**2 + A + 1)
    ),

    "two_sided_paired_theta": lambda m, A: (
        (A**2 + 1)**(m + 1)
        * (A**2 - A + 1)**m

        + (-1)**(m + 1)
          * A**(2*m)
          * (A**4 + A**2 + 1)
          * (A - 1)**m
          * (2*A**2 - A + 2)**m

        - A**(5*m + 2)
          * (A**4 + A**3 + A**2 + A + 1)
          * (A + 1)**m
          * (A**2 - A + 1)**m
    ),
}


### Strongest held-out test: compute reserved values for the first time
The default discovery run leaves the held-out file empty.

After all candidate formulas are frozen, set

```python
RECOMPUTE_HELDOUT_FRESH = True
```

and run the final cell.  KnottedGraph will then compute \(m=101,125,150,200\) for the new families for the first time.

In [ ]:
def parse_candidate(candidate, parameter_value: int) -> sp.Expr:
    if callable(candidate):
        return sp.sympify(candidate(int(parameter_value), A))
    if isinstance(candidate, str):
        return sp.sympify(
            candidate,
            locals={
                "A": A,
                "m": sp.Integer(int(parameter_value)),
                "n": sp.Integer(int(parameter_value)),
                "sigma": SIGMA,
            },
        )
    return sp.sympify(candidate)


def parse_actual_polynomial(text: str) -> sp.Expr:
    return sp.sympify(text, locals={"A": A})


RECOMPUTE_HELDOUT_FRESH = True

if RECOMPUTE_HELDOUT_FRESH:
    if not FORMULA_CANDIDATES:
        raise RuntimeError("Paste frozen formula candidates first.")

    fresh_results = []

    for spec in FAMILIES:
        if spec.key not in FORMULA_CANDIDATES:
            continue

        for value in spec.heldout_values:
            print(f"FRESH HELD-OUT {spec.key} {spec.parameter_name}={value}", flush=True)

            fresh_row = evaluate_family_case(spec, value)
            actual = parse_actual_polynomial(fresh_row["yamada_A_normalized"])
            predicted = parse_candidate(FORMULA_CANDIDATES[spec.key], value)

            passed = exact_same(sp.expand(actual), sp.expand(predicted))
            fresh_results.append(
                {
                    "family": spec.key,
                    "parameter_name": spec.parameter_name,
                    "parameter_value": value,
                    "pass": passed,
                    "fresh_yamada_seconds": fresh_row["yamada_seconds"],
                }
            )

            print(
                f"{'PASS' if passed else 'FAIL'} "
                f"{spec.key} {spec.parameter_name}={value}; "
                f"fresh Yamada={fresh_row['yamada_seconds']:.4g}s",
                flush=True,
            )

    if not all(result["pass"] for result in fresh_results):
        raise AssertionError("At least one frozen candidate failed fresh exact held-out recomputation.")

    print("\nPASS: every frozen candidate survived fresh exact KnottedGraph held-out extrapolation.")


### How to interpret the result
The intended result is a reusable three-family discovery example in which every target family lies in the subcubic regime:


$$
\boxed{
\Lambda_m,\qquad
\Lambda_m^\downarrow,\qquad
\Lambda_m^{\updownarrow},
\qquad \Delta\le3.
}
$$


The workflow is

$$
\text{define family}
\rightarrow
\text{independent exact KnottedGraph dataset}
\rightarrow
\text{formula conjecture}
\rightarrow
\text{freeze}
\rightarrow
\text{distant held-out exact verification}.
$$

Until a mathematical derivation is supplied, an exact held-out success should be described as a **computer-generated conjectured closed form with exact held-out verification**, not as a proved theorem.

# Part II — Abelian mixed theta words

## Goal

The three homogeneous families suggest a stronger test: instead of repeating
one motif, arrange the three motifs in an arbitrary word.  The computational
alphabet is

- `L`: cross-linked cell \(\Lambda\),
- `D`: lower-paired cell \(\downarrow\),
- `B`: two-sided-paired cell \(\updownarrow\).

For

$$
u=\tau_1\tau_2\cdots\tau_m,
\qquad
\tau_j\in\{\Lambda,\downarrow,\updownarrow\},
$$

define

$$
\mathbf n=(n_\Lambda,n_\downarrow,n_\updownarrow).
$$

The corresponding named family is \(\mathcal A_{\mathbf n}\).

## Why “Abelian”?

The hypothesis tested here is stronger than the three homogeneous formulas:
the Yamada polynomial should factor through the motif-count vector,

$$
u\longmapsto \mathbf n
\longmapsto \Upsilon(\mathcal A_{\mathbf n};Y).
$$

Thus two different orderings with the same counts should have exactly the same
Laurent polynomial.  The notebook first exhausts/stress-tests short words and
then freezes the count-only master formula before evaluating 20 extreme
previously unseen words at

$$
m=101,\ 125,\ 150,\ 200.
$$

At each extreme length, same-count words with deliberately different order are
included explicitly.


### Environment and provenance

In [ ]:

from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
from types import SimpleNamespace
import csv
import hashlib
import itertools
import json
import math
import os
import pickle
import random
import subprocess
import time
import warnings

import networkx as nx
import numpy as np
import sympy as sp

AUDITED_SOURCE_COMMIT = "49e34e47ca5c182f55ef5f0ea0906220df59befb"
STRICT_PUBLICATION_REGENERATION = (
    os.environ.get("KNOTTEDGRAPH_STRICT_PUBLICATION_REGENERATION", "0") == "1"
)
SOURCE_CONSTRUCTOR_VERSION = "2026-08-25.three-subcubic-theta-families.safe-return.v3"
CONSTRUCTOR_VERSION = "2026-08-27.mixed-theta-word-transfer-test.v1"
RESULT_SCHEMA_VERSION = 2

A = sp.Symbol("A")
SIGMA = A + 1 + A**-1

def _is_repo_root(path: Path) -> bool:
    return (
        (path / "pyproject.toml").is_file()
        and (path / "src" / "knotted_graph").is_dir()
        and (path / "User_guide" / "applications").is_dir()
    )

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if _is_repo_root(candidate):
            return candidate
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook/script from inside the repository checkout."
    )

ROOT = find_repo_root()
RESULTS_DIR = ROOT / "User_guide" / "applications" / "results" / "05_yamada_formula_discovery"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FULL_CSV = RESULTS_DIR / "05b_mixed_theta_transfer_words_full.csv"
CLASS_CSV = RESULTS_DIR / "05b_mixed_theta_transfer_permutation_classes.csv"
COUNTEREXAMPLE_CSV = RESULTS_DIR / "05b_mixed_theta_transfer_counterexamples.csv"
HANKEL_CSV = RESULTS_DIR / "05b_mixed_theta_transfer_hankel_rank.csv"
SHARE_CSV = RESULTS_DIR / "05b_mixed_theta_transfer_share.csv"
PROJECTION_CACHE_DIR = RESULTS_DIR / "05b_mixed_theta_transfer_pd_cache"
PROJECTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def git_text(*args: str) -> str:
    proc = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.returncode:
        raise RuntimeError(
            f"git {' '.join(args)} failed.\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc.stdout.strip()

CURRENT_BRANCH = git_text("rev-parse", "--abbrev-ref", "HEAD")
GIT_COMMIT = git_text("rev-parse", "HEAD")

AUDITED_SOURCE_PRESENT = subprocess.run(
    ["git", "merge-base", "--is-ancestor", AUDITED_SOURCE_COMMIT, "HEAD"],
    cwd=ROOT,
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

if STRICT_PUBLICATION_REGENERATION and not AUDITED_SOURCE_PRESENT:
    raise RuntimeError(
        "Strict publication regeneration requires audited source commit "
        f"{AUDITED_SOURCE_COMMIT}. Current HEAD is {GIT_COMMIT}."
    )
if not AUDITED_SOURCE_PRESENT:
    warnings.warn(
        "The audited source commit is not an ancestor of this checkout. "
        "Exploration may continue, but do not treat regenerated tables as "
        "publication-audited outputs.",
        RuntimeWarning,
    )

library_status = git_text("status", "--porcelain", "--", "src/knotted_graph")
if library_status:
    raise RuntimeError(
        "The KnottedGraph source tree has uncommitted changes. "
        "Commit or stash them before generating an auditable dataset:\n"
        + library_status
    )

import knotted_graph

KG_FILE = Path(knotted_graph.__file__).resolve()
if ROOT not in KG_FILE.parents:
    raise RuntimeError(
        "Python imported knotted_graph from outside this checkout.\n"
        f"Repository root: {ROOT}\nImported package: {KG_FILE}"
    )

from knotted_graph.core.embedding import ensure_embedding
from knotted_graph.projection import PDCode, select_projection
from knotted_graph.invariants.yamada.polynomial import Yamada
from knotted_graph.invariants.yamada.factorized_frontier import (
    build_factorized_frontier,
    native_factorized_available,
    factorized_import_error,
)

if not native_factorized_available():
    raise RuntimeError(
        "The optimized native factorized Yamada backend is unavailable. "
        "Rebuild this checkout before running the dataset.\n"
        f"Import error: {factorized_import_error()!r}"
    )

print("Repository :", ROOT)
print("Branch     :", CURRENT_BRANCH)
print("Commit     :", GIT_COMMIT)
print("Package    :", KG_FILE)
print("Native optimized Yamada backend: AVAILABLE")


### Experiment controls

In [ ]:

# ============================================================
# USER CONTROLS
# ============================================================

# Exhaust every word over {L,D,B} through this length.
# 1..5 gives 3+9+27+81+243 = 363 exact mixed-word cases.
MAX_EXHAUSTIVE_WORD_LENGTH = 5

# Additional longer permutation-class stress tests.
STRESS_LENGTHS = [6, 7]
STRESS_COMPOSITION_CLASSES_PER_LENGTH = 8
STRESS_PERMUTATIONS_PER_CLASS = 6
RNG_SEED = 20260827

# Correctness gates.
RUN_UPSTREAM_SANITY = True
RUN_HOMOGENEOUS_RECOVERY_PREFLIGHT = True
HOMOGENEOUS_RECOVERY_M = [1, 2, 3, 5]
RUN_REPRESENTATIVE_3D_CONTACT_AUDIT = True
REPRESENTATIVE_MIXED_WORDS = [
    "LD", "DL", "LB", "BL", "DB", "BD",
    "LDB", "DBL", "BLD", "LDBD",
]
RUN_CANONICAL_PROJECTION_PREFLIGHT = True
CANONICAL_PREFLIGHT_WORDS = [
    "L", "D", "B",
    "LD", "DL", "LB", "BL", "DB", "BD",
    "LDB", "DBL", "BLD", "LDBD",
]

# Runtime / persistence.
RESUME = True
FAIL_FAST = False
SAVE_EVERY_CASES = 10
NORMALIZE_YAMADA = True

# Geometry/projection settings copied from the audited source notebook.
ROTATION_ORDER = "ZYX"
COMPUTE_FRONTIER_DIAGNOSTICS = False
REUSE_PROJECTION_CACHE = True
WRITE_PROJECTION_CACHE = True
BRAID_SAMPLES_PER_CROSSING = 5
RETURN_SAMPLES = 31
SIDE_CONNECTION_SAMPLES = 9
SAFE_RETURN_Y = 1.85
SAFE_RETURN_RIGHT_MARGIN = 0.70
SAFE_RETURN_LEFT_X = -0.55
CONTACT_TOL = 1e-8

# If Eq. (4) fails but this exact short-word Hankel rank remains <=3,
# investigate noncommuting 3x3 transfer matrices next.
HANKEL_BASIS_MAX_WORD_LENGTH = 2
HANKEL_EVALUATION_POINTS = [2, 3, 5, 7]

ALPHABET = ("L", "D", "B")
MOTIF_NAMES = {
    "L": "Lambda / cross-linked cell",
    "D": "down / bottom-paired cell",
    "B": "both / two-sided-paired cell",
}

COMPUTE_CONFIG = {
    "schema": RESULT_SCHEMA_VERSION,
    "constructor_version": CONSTRUCTOR_VERSION,
    "source_constructor_version": SOURCE_CONSTRUCTOR_VERSION,
    "rotation_order": ROTATION_ORDER,
    "normalize": NORMALIZE_YAMADA,
    "braid_samples_per_crossing": BRAID_SAMPLES_PER_CROSSING,
    "return_samples": RETURN_SAMPLES,
    "side_connection_samples": SIDE_CONNECTION_SAMPLES,
    "safe_return_y": SAFE_RETURN_Y,
    "safe_return_right_margin": SAFE_RETURN_RIGHT_MARGIN,
    "safe_return_left_x": SAFE_RETURN_LEFT_X,
}
COMPUTE_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(COMPUTE_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

RUN_CONFIG = {
    **COMPUTE_CONFIG,
    "max_exhaustive_word_length": MAX_EXHAUSTIVE_WORD_LENGTH,
    "stress_lengths": STRESS_LENGTHS,
    "stress_composition_classes_per_length": STRESS_COMPOSITION_CLASSES_PER_LENGTH,
    "stress_permutations_per_class": STRESS_PERMUTATIONS_PER_CLASS,
    "rng_seed": RNG_SEED,
    "hankel_basis_max_word_length": HANKEL_BASIS_MAX_WORD_LENGTH,
    "hankel_evaluation_points": HANKEL_EVALUATION_POINTS,
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

print("Compute config :", COMPUTE_CONFIG_SHA256)
print("Run config     :", RUN_CONFIG_SHA256)
print("Full CSV       :", FULL_CSV)
print("Share CSV      :", SHARE_CSV)


### Upstream Yamada correctness gate

In [ ]:

if RUN_UPSTREAM_SANITY:
    sanity_script = ROOT / "dev" / "run_yamada_sanity_checks.py"
    if not sanity_script.is_file():
        raise RuntimeError(f"Missing sanity script: {sanity_script}")

    proc = subprocess.run(
        [os.sys.executable, str(sanity_script)],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    print(proc.stdout)

    if proc.returncode:
        raise RuntimeError(
            "Upstream Yamada sanity checks failed.\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )

    marker = "PASS: all published/independent Yamada sanity checks succeeded."
    if marker not in proc.stdout:
        raise RuntimeError("Expected upstream Yamada sanity success marker was not emitted.")

print("PASS: upstream Yamada sanity gate.")


### Audited geometry helpers copied from notebook 06

In [ ]:
def join_paths(*paths: np.ndarray) -> np.ndarray:
    pieces = []
    for path in paths:
        path = np.asarray(path, dtype=float)
        if len(path) == 0:
            continue
        if pieces and np.allclose(pieces[-1][-1], path[0]):
            path = path[1:]
        if len(path):
            pieces.append(path)

    if not pieces:
        return np.empty((0, 3), dtype=float)
    return np.vstack(pieces)


def line(a, b, samples: int = 2) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (1.0 - t) * a + t * b


def bezier_cubic(p0, p1, p2, p3, samples: int = 41) -> np.ndarray:
    p0, p1, p2, p3 = [np.asarray(p, dtype=float) for p in (p0, p1, p2, p3)]
    t = np.linspace(0.0, 1.0, int(samples))[:, None]
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t**2 * p2
        + t**3 * p3
    )


def segment_distance_3d(p1, q1, p2, q2) -> float:
    p1 = np.asarray(p1, dtype=float)
    q1 = np.asarray(q1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    q2 = np.asarray(q2, dtype=float)

    u = q1 - p1
    v = q2 - p2
    w = p1 - p2

    a = float(u @ u)
    b = float(u @ v)
    c = float(v @ v)
    d = float(u @ w)
    e = float(v @ w)

    eps = 1e-14

    if a <= eps and c <= eps:
        return float(np.linalg.norm(p1 - p2))
    if a <= eps:
        t = float(np.clip(e / c, 0.0, 1.0))
        return float(np.linalg.norm(p1 - (p2 + t * v)))
    if c <= eps:
        s = float(np.clip(-d / a, 0.0, 1.0))
        return float(np.linalg.norm((p1 + s * u) - p2))

    D = a * c - b * b
    sN = sD = D
    tN = tD = D

    if D < eps:
        sN = 0.0
        sD = 1.0
        tN = e
        tD = c
    else:
        sN = b * e - c * d
        tN = a * e - b * d

        if sN < 0.0:
            sN = 0.0
            tN = e
            tD = c
        elif sN > sD:
            sN = sD
            tN = e + b
            tD = c

    if tN < 0.0:
        tN = 0.0
        if -d < 0.0:
            sN = 0.0
        elif -d > a:
            sN = sD
        else:
            sN = -d
            sD = a
    elif tN > tD:
        tN = tD
        if (-d + b) < 0.0:
            sN = 0.0
        elif (-d + b) > a:
            sN = sD
        else:
            sN = -d + b
            sD = a

    sc = 0.0 if abs(sN) < eps else sN / sD
    tc = 0.0 if abs(tN) < eps else tN / tD

    return float(np.linalg.norm(w + sc * u - tc * v))


def audit_piecewise_linear_embedding(graph: nx.MultiGraph, *, tolerance: float = CONTACT_TOL) -> None:
    graph = ensure_embedding(graph, copy=True, normalize=True)

    segments = []
    for u, v, key, data in graph.edges(keys=True, data=True):
        pts = np.asarray(data["pts"], dtype=float)
        lengths = np.linalg.norm(np.diff(pts, axis=0), axis=1)
        if np.any(lengths <= tolerance):
            raise AssertionError(f"Degenerate segment in edge {(u, v, key)!r}.")
        for segment_index in range(len(pts) - 1):
            segments.append((u, v, key, segment_index, pts[segment_index], pts[segment_index + 1]))

    for i, first in enumerate(segments):
        u1, v1, k1, s1, p1, q1 = first
        for second in segments[i + 1:]:
            u2, v2, k2, s2, p2, q2 = second

            if (u1, v1, k1) == (u2, v2, k2) and abs(s1 - s2) <= 1:
                continue

            if (
                np.any(np.maximum(p1, q1) + tolerance < np.minimum(p2, q2))
                or np.any(np.maximum(p2, q2) + tolerance < np.minimum(p1, q1))
            ):
                continue

            allowed = False
            for node in {u1, v1}.intersection({u2, v2}):
                pos = np.asarray(graph.nodes[node]["pos"], dtype=float)
                first_touches = min(np.linalg.norm(p1 - pos), np.linalg.norm(q1 - pos)) <= tolerance
                second_touches = min(np.linalg.norm(p2 - pos), np.linalg.norm(q2 - pos)) <= tolerance
                if first_touches and second_touches:
                    allowed = True
                    break

            if allowed:
                continue

            if segment_distance_3d(p1, q1, p2, q2) <= tolerance:
                raise AssertionError(
                    "Unintended 3-D contact between "
                    f"{(u1, v1, k1, s1)!r} and {(u2, v2, k2, s2)!r}."
                )


In [ ]:
def positive_two_braid_geometry(
    q: int,
    *,
    samples_per_crossing: int = BRAID_SAMPLES_PER_CROSSING,
):
    q = int(q)
    if q <= 0 or q % 2 == 0:
        raise ValueError("q must be a positive odd integer.")
    samples_per_crossing = int(samples_per_crossing)
    if samples_per_crossing < 5 or samples_per_crossing % 2 == 0:
        raise ValueError("BRAID_SAMPLES_PER_CROSSING must be an odd integer >=5.")

    y0 = 0.62
    depth = 0.24

    U = np.array([0.0, y0, 0.0])
    V = np.array([0.0, -y0, 0.0])

    x_left = 0.90
    x_right = x_left + float(q)

    count = q * samples_per_crossing + 1
    t = np.linspace(0.0, 1.0, count)
    knots = np.linspace(0.0, 1.0, q + 1)
    lane_values = y0 * ((-1.0) ** np.arange(q + 1))

    y_a = np.interp(t, knots, lane_values)
    z_a = depth * np.sin(np.pi * q * t)

    x = x_left + (x_right - x_left) * t
    braid_a = np.column_stack([x, y_a, z_a])
    braid_b = np.column_stack([x, -y_a, -z_a])

    top_right = np.array([x_right, y0, 0.0])
    bottom_right = np.array([x_right, -y0, 0.0])

    if not np.allclose(braid_a[-1], bottom_right):
        raise AssertionError("Odd positive braid A did not end on bottom lane.")
    if not np.allclose(braid_b[-1], top_right):
        raise AssertionError("Odd positive braid B did not end on top lane.")

    # Safe outside return corridors.
    #
    # The previous long Bezier returns were visually compact but, as q grew,
    # their xy projection entered the same band as the local side-pair arcs.
    # This produced accidental crossings.  These PL returns immediately leave
    # the repeated motif to x>x_right, travel in a far upper/lower corridor,
    # return at x<0, and then connect to U/V.  Hence they are disjoint in the
    # canonical xy projection from every local side-pair edge for all m.
    x_far_right = x_right + SAFE_RETURN_RIGHT_MARGIN
    x_far_left = SAFE_RETURN_LEFT_X

    top_return = np.array(
        [
            top_right,
            [x_far_right, y0, 0.0],
            [x_far_right, SAFE_RETURN_Y, 0.0],
            [x_far_left, SAFE_RETURN_Y, 0.0],
            [x_far_left, y0, 0.0],
            U,
        ],
        dtype=float,
    )

    bottom_return = np.array(
        [
            bottom_right,
            [x_far_right, -y0, 0.0],
            [x_far_right, -SAFE_RETURN_Y, 0.0],
            [x_far_left, -SAFE_RETURN_Y, 0.0],
            [x_far_left, -y0, 0.0],
            V,
        ],
        dtype=float,
    )

    left_u = line(U, braid_a[0], 9)
    left_v = line(V, braid_b[0], 9)

    phi = np.linspace(np.pi / 2, 3 * np.pi / 2, RETURN_SAMPLES)
    exterior = np.column_stack(
        [
            -0.78 * np.cos(phi - np.pi),
            y0 * np.sin(phi),
            np.full_like(phi, 0.38),
        ]
    )
    exterior[0] = U
    exterior[-1] = V

    return {
        "U": U,
        "V": V,
        "braid_a": braid_a,
        "braid_b": braid_b,
        "left_u": left_u,
        "left_v": left_v,
        "top_return": top_return,
        "bottom_return": bottom_return,
        "exterior": exterior,
        "samples_per_crossing": samples_per_crossing,
    }



def lower_upper_nodes_for_boundary(graph: nx.MultiGraph, j: int):
    """Return the lower and upper subdivision node at braid boundary j."""
    a = ("a", int(j))
    b = ("b", int(j))

    ya = float(graph.nodes[a]["pos"][1])
    yb = float(graph.nodes[b]["pos"][1])

    if ya <= yb:
        return a, b
    return b, a


def side_connection_arc(
    p0: np.ndarray,
    p1: np.ndarray,
    *,
    direction: str,
    pair_index: int,
) -> np.ndarray:
    """
    Local outside-the-braid connection used by the degree<=3 paired families.

    Endpoints have the same y-coordinate in the canonical projection.  The arc
    bows away from the braid strip and receives a tiny alternating z-offset so
    distinct 3-D edges stay separated even before projection.
    """
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)

    if direction not in {"down", "up"}:
        raise ValueError("direction must be 'down' or 'up'.")

    sign = -1.0 if direction == "down" else 1.0
    span = float(p1[0] - p0[0])

    if span <= 0:
        raise ValueError("side_connection_arc expects p1 to lie to the right of p0.")

    bow = 0.48
    zoff = 0.06 * (1.0 if pair_index % 2 else -1.0)

    c1 = p0 + np.array([0.28 * span, sign * bow, zoff])
    c2 = p1 + np.array([-0.28 * span, sign * bow, -zoff])

    return bezier_cubic(
        p0,
        c1,
        c2,
        p1,
        samples=SIDE_CONNECTION_SAMPLES,
    )


def add_nonoverlapping_side_pairs(
    graph: nx.MultiGraph,
    *,
    add_bottom: bool,
    add_top: bool,
) -> None:
    """
    Pair boundaries (1,2), (3,4), ..., (2m-1,2m).

    Because q=2m+1, q-1=2m is even.  Every selected subdivision vertex therefore
    receives exactly ONE added edge, keeping its total degree <=3.
    """
    q = int(graph.graph["q"])

    if (q - 1) % 2:
        raise AssertionError("Expected q-1 to be even.")

    for j in range(1, q - 1, 2):
        jp = j + 1
        pair_index = (j + 1) // 2

        lower_j, upper_j = lower_upper_nodes_for_boundary(graph, j)
        lower_p, upper_p = lower_upper_nodes_for_boundary(graph, jp)

        if add_bottom:
            p0 = np.asarray(graph.nodes[lower_j]["pos"], dtype=float)
            p1 = np.asarray(graph.nodes[lower_p]["pos"], dtype=float)
            graph.add_edge(
                lower_j,
                lower_p,
                pts=side_connection_arc(
                    p0,
                    p1,
                    direction="down",
                    pair_index=pair_index,
                ),
                role="bottom_pair_edge",
                pair_index=pair_index,
            )

        if add_top:
            p0 = np.asarray(graph.nodes[upper_j]["pos"], dtype=float)
            p1 = np.asarray(graph.nodes[upper_p]["pos"], dtype=float)
            graph.add_edge(
                upper_j,
                upper_p,
                pts=side_connection_arc(
                    p0,
                    p1,
                    direction="up",
                    pair_index=pair_index,
                ),
                role="top_pair_edge",
                pair_index=pair_index,
            )


In [ ]:
def subdivided_theta_skeleton(m: int) -> nx.MultiGraph:
    """
    Shared skeleton with vertices U, V and boundary vertices a_j, b_j, j=1..q-1.
    Constituent-cycle edges are subdivided at every boundary between consecutive crossings.
    """
    m = int(m)
    if m < 1:
        raise ValueError("m must be >=1.")

    q = 2 * m + 1
    geom = positive_two_braid_geometry(q)
    U = geom["U"]
    V = geom["V"]
    braid_a = geom["braid_a"]
    braid_b = geom["braid_b"]
    s = int(geom["samples_per_crossing"])

    graph = nx.MultiGraph()
    graph.add_node("U", pos=U.copy())
    graph.add_node("V", pos=V.copy())

    for j in range(1, q):
        index = j * s
        graph.add_node(("a", j), pos=braid_a[index].copy())
        graph.add_node(("b", j), pos=braid_b[index].copy())

    # Path A: U -> a1 -> ... -> a_(q-1) -> V
    first_a = join_paths(geom["left_u"], braid_a[: s + 1])
    graph.add_edge("U", ("a", 1), pts=first_a, role="constituent_cycle", path_family="A")

    for j in range(1, q - 1):
        pts = braid_a[j * s : (j + 1) * s + 1].copy()
        graph.add_edge(("a", j), ("a", j + 1), pts=pts, role="constituent_cycle", path_family="A")

    last_a = join_paths(braid_a[(q - 1) * s :], geom["bottom_return"])
    graph.add_edge(("a", q - 1), "V", pts=last_a, role="constituent_cycle", path_family="A")

    # Path B: U -> b_(q-1) -> ... -> b_1 -> V
    first_b = join_paths(geom["top_return"][::-1], braid_b[(q - 1) * s :][::-1])
    graph.add_edge("U", ("b", q - 1), pts=first_b, role="constituent_cycle", path_family="B")

    for j in range(q - 1, 1, -1):
        pts = braid_b[(j - 1) * s : j * s + 1][::-1].copy()
        graph.add_edge(("b", j), ("b", j - 1), pts=pts, role="constituent_cycle", path_family="B")

    last_b = join_paths(braid_b[: s + 1][::-1], geom["left_v"][::-1])
    graph.add_edge(("b", 1), "V", pts=last_b, role="constituent_cycle", path_family="B")

    graph.add_edge("U", "V", pts=geom["exterior"].copy(), role="exterior_theta_edge")

    graph.graph.update(
        parameter_name="m",
        m=m,
        q=q,
        constituent_knot=f"T(2,{q})",
        crossing_lower_bound=q,
        expected_canonical_crossings=q,
        expected_vertices=2 * q,
    )

    return ensure_embedding(graph, copy=False, normalize=True)


In [ ]:
def crosslinked_theta_ladder(m: int) -> nx.MultiGraph:
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    for j in range(1, q):
        a = ("a", j)
        b = ("b", j)

        p = np.asarray(graph.nodes[a]["pos"], dtype=float)
        p_other = np.asarray(graph.nodes[b]["pos"], dtype=float)

        graph.add_edge(
            a,
            b,
            pts=line(p, p_other, 2),
            role="rung",
            rung_index=j,
        )

    graph.graph.update(
        family="crosslinked_theta_ladder",
        family_label="Cross-linked theta ladder Lambda_m",
        expected_edges=3 * q,
        expected_rungs=q - 1,
        expected_bottom_pair_edges=0,
        expected_top_pair_edges=0,
        expected_degree_histogram={3: 2 * q},
    )

    return ensure_embedding(graph, copy=False, normalize=True)


def bottom_paired_theta(m: int) -> nx.MultiGraph:
    """
    P_m^downarrow:
    pair consecutive LOWER boundary vertices non-overlappingly:
    (1,2), (3,4), ..., (2m-1,2m).
    """
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    add_nonoverlapping_side_pairs(
        graph,
        add_bottom=True,
        add_top=False,
    )

    # Base subdivided theta has 2q+1 edges; we add m=(q-1)/2 pair edges.
    pair_count = (q - 1) // 2

    graph.graph.update(
        family="bottom_paired_theta",
        family_label="Bottom-paired theta family P_down_m",
        expected_edges=2 * q + 1 + pair_count,
        expected_rungs=0,
        expected_bottom_pair_edges=pair_count,
        expected_top_pair_edges=0,
        # 2m lower boundary vertices + U,V are degree 3;
        # 2m upper boundary vertices remain degree 2.
        expected_degree_histogram={
            2: q - 1,
            3: q + 1,
        },
    )

    return ensure_embedding(graph, copy=False, normalize=True)


def two_sided_paired_theta(m: int) -> nx.MultiGraph:
    """
    P_m^{updownarrow}:
    add the same non-overlapping consecutive pairing on BOTH lower and upper sides.
    """
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    add_nonoverlapping_side_pairs(
        graph,
        add_bottom=True,
        add_top=True,
    )

    pair_count = (q - 1) // 2

    graph.graph.update(
        family="two_sided_paired_theta",
        family_label="Two-sided paired theta family P_updown_m",
        expected_edges=3 * q,
        expected_rungs=0,
        expected_bottom_pair_edges=pair_count,
        expected_top_pair_edges=pair_count,
        expected_degree_histogram={3: 2 * q},
    )

    return ensure_embedding(graph, copy=False, normalize=True)


### Structural and 3-D certification

In [ ]:
def certify_connected(graph: nx.MultiGraph) -> None:
    if not nx.is_connected(nx.Graph(graph)):
        raise AssertionError("Graph is disconnected.")


def extract_constituent_cycle(graph: nx.MultiGraph) -> nx.MultiGraph:
    cycle = nx.MultiGraph()

    for node, data in graph.nodes(data=True):
        cycle.add_node(
            node,
            pos=np.asarray(data["pos"], dtype=float).copy(),
        )

    used_nodes = set()

    for u, v, key, data in graph.edges(keys=True, data=True):
        if data.get("role") != "constituent_cycle":
            continue

        cycle.add_edge(
            u,
            v,
            pts=np.asarray(data["pts"], dtype=float).copy(),
            role="constituent_cycle",
        )
        used_nodes.add(u)
        used_nodes.add(v)

    for node in list(cycle.nodes()):
        if node not in used_nodes:
            cycle.remove_node(node)

    return ensure_embedding(cycle, copy=False, normalize=True)


def certify_theta_derived_family(
    graph: nx.MultiGraph,
    *,
    require_cubic: bool,
    audit_3d_contacts: bool = False,
) -> None:
    certify_connected(graph)

    m = int(graph.graph["m"])
    q = int(graph.graph["q"])

    if q != 2 * m + 1:
        raise AssertionError("q != 2m+1.")

    if graph.number_of_nodes() != int(graph.graph["expected_vertices"]):
        raise AssertionError(
            f"Expected V={graph.graph['expected_vertices']}, "
            f"found {graph.number_of_nodes()}."
        )

    if graph.number_of_edges() != int(graph.graph["expected_edges"]):
        raise AssertionError(
            f"Expected E={graph.graph['expected_edges']}, "
            f"found {graph.number_of_edges()}."
        )

    role_counts = Counter(
        str(data.get("role", ""))
        for *_edge, data in graph.edges(data=True)
    )

    expected_role_counts = {
        "rung": int(graph.graph["expected_rungs"]),
        "bottom_pair_edge": int(graph.graph["expected_bottom_pair_edges"]),
        "top_pair_edge": int(graph.graph["expected_top_pair_edges"]),
    }

    for role, expected in expected_role_counts.items():
        actual = int(role_counts.get(role, 0))
        if actual != expected:
            raise AssertionError(
                f"Expected {expected} {role!r} edges, found {actual}."
            )

    degrees = [int(deg) for _, deg in graph.degree()]
    degree_histogram = dict(sorted(Counter(degrees).items()))

    if max(degrees) > 3:
        raise AssertionError(
            f"Subcubic requirement violated: Delta(G)={max(degrees)}."
        )

    expected_histogram = {
        int(k): int(v)
        for k, v in graph.graph["expected_degree_histogram"].items()
    }

    if degree_histogram != expected_histogram:
        raise AssertionError(
            "Degree histogram mismatch: "
            f"expected {expected_histogram}, found {degree_histogram}."
        )

    if require_cubic and any(deg != 3 for deg in degrees):
        raise AssertionError(
            f"Expected exactly cubic graph; histogram={degree_histogram}."
        )

    constituent = extract_constituent_cycle(graph)

    if not nx.is_connected(nx.Graph(constituent)):
        raise AssertionError("Constituent cycle is disconnected.")

    constituent_degrees = [int(deg) for _, deg in constituent.degree()]
    if any(deg != 2 for deg in constituent_degrees):
        raise AssertionError(
            "Constituent A∪B is not 2-regular; "
            f"histogram={dict(Counter(constituent_degrees))}."
        )

    # This zero-rotation constituent check is only used in representative preflight.
    if audit_3d_contacts:
        processor = PDCode(constituent)
        processor.compute(
            rotation_angles=(0.0, 0.0, 0.0),
            rotation_order=ROTATION_ORDER,
        )

        if len(processor.crossings) != q:
            raise AssertionError(
                f"Expected constituent to have q={q} canonical crossings, "
                f"detected {len(processor.crossings)}."
            )

        audit_piecewise_linear_embedding(graph)


### Canonical projection/cache and exact Laurent utilities

In [ ]:
def factorized_frontier_width(processor) -> dict:
    """Optional diagnostic only; disabled by default for maximum throughput."""
    yamada = Yamada(
        vertices=list(processor.vertices.values()),
        crossings=list(processor.crossings.values()),
        arcs=list(processor.arcs.values()),
    )

    prepared = yamada._prepare_compact_state_builder()
    data = build_factorized_frontier(prepared)

    factor_count = len(data["factor_types"])
    ports_by_factor = [[] for _ in range(factor_count)]

    for port, factor in enumerate(data["port_factor"]):
        ports_by_factor[int(factor)].append(port)

    active = []
    processed = set()
    peak_live_ports = 0
    max_boundary_ports = 0

    for factor in data["factor_order"]:
        factor = int(factor)
        active.extend(ports_by_factor[factor])
        peak_live_ports = max(peak_live_ports, len(active))
        processed.add(factor)

        active = [
            port
            for port in active
            if int(
                data["port_factor"][
                    int(data["wire_partner"][port])
                ]
            )
            not in processed
        ]

        max_boundary_ports = max(
            max_boundary_ports,
            len(active),
        )

    if active:
        raise RuntimeError("Factorized frontier planner did not close.")

    return {
        "crossings_after_RII": len(prepared.crossing_ids),
        "peak_live_ports": int(peak_live_ports),
        "max_boundary_ports": int(max_boundary_ports),
        "factor_count": int(factor_count),
    }


def embedding_hash(graph: nx.MultiGraph) -> str:
    payload = []

    for node, data in sorted(
        graph.nodes(data=True),
        key=lambda item: repr(item[0]),
    ):
        payload.append(
            (
                "node",
                repr(node),
                np.asarray(data["pos"], dtype=float).round(12).tolist(),
            )
        )

    edge_payload = []

    for u, v, key, data in graph.edges(keys=True, data=True):
        edge_payload.append(
            (
                repr(u),
                repr(v),
                int(key),
                str(data.get("role", "")),
                np.asarray(data["pts"], dtype=float).round(12).tolist(),
            )
        )

    payload.extend(
        ("edge", *entry)
        for entry in sorted(
            edge_payload,
            key=lambda item: (item[0], item[1], item[2], item[3]),
        )
    )

    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()


def projection_cache_key(
    *,
    family: str,
    parameter_value: int,
    graph_hash: str,
) -> tuple[str, Path]:
    payload = {
        "schema": 2,
        "constructor_version": CONSTRUCTOR_VERSION,
        "git_commit": GIT_COMMIT,
        "family": family,
        "parameter_value": int(parameter_value),
        "graph_hash": graph_hash,
        "rotation_order": ROTATION_ORDER,
        "canonical_zero": True,
        "frontier_diagnostics": COMPUTE_FRONTIER_DIAGNOSTICS,
    }

    key = hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()

    safe_family = family.replace("/", "_")

    path = (
        PROJECTION_CACHE_DIR
        / f"{safe_family}__p{parameter_value}__{key[:20]}.pkl"
    )

    return key, path


def _processor_payload(processor) -> dict:
    return {
        "vertices": dict(processor.vertices),
        "crossings": dict(processor.crossings),
        "arcs": dict(processor.arcs),
    }


def _processor_from_payload(payload: dict):
    return SimpleNamespace(
        vertices=dict(payload["vertices"]),
        crossings=dict(payload["crossings"]),
        arcs=dict(payload["arcs"]),
    )


def save_projection_cache(
    path: Path,
    *,
    cache_key: str,
    graph_hash: str,
    projection,
    frontier: dict,
) -> None:
    if not WRITE_PROJECTION_CACHE:
        return

    payload = {
        "schema": 2,
        "cache_key": cache_key,
        "graph_hash": graph_hash,
        "rotation_angles": projection.rotation_angles,
        "rotation_order": projection.rotation_order,
        "pd_code": projection.pd_code,
        "num_crossings": int(projection.num_crossings),
        "frontier": dict(frontier),
        **_processor_payload(projection.processor),
    }

    tmp = path.with_suffix(path.suffix + ".tmp")

    with tmp.open("wb") as handle:
        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
        handle.flush()
        os.fsync(handle.fileno())

    tmp.replace(path)


def load_projection_cache(
    path: Path,
    *,
    cache_key: str,
    graph_hash: str,
):
    if not REUSE_PROJECTION_CACHE or not path.exists():
        return None

    try:
        with path.open("rb") as handle:
            payload = pickle.load(handle)

        if payload.get("schema") != 2:
            return None
        if payload.get("cache_key") != cache_key:
            return None
        if payload.get("graph_hash") != graph_hash:
            return None

        processor = _processor_from_payload(payload)

        return {
            "processor": processor,
            "rotation_angles": payload["rotation_angles"],
            "rotation_order": payload["rotation_order"],
            "pd_code": payload["pd_code"],
            "num_crossings": int(payload["num_crossings"]),
            "frontier": dict(payload.get("frontier", {})),
            "source": "cache",
            "candidate_records": [],
        }

    except Exception as exc:
        print(
            f"Ignoring stale/broken projection cache {path.name}: "
            f"{type(exc).__name__}: {exc}"
        )
        return None


def choose_projection(
    graph: nx.MultiGraph,
    *,
    family: str,
    parameter_value: int,
):
    """
    Fast path: one canonical zero-rotation projection only.

    Projection independence is already covered by the repository sanity suite;
    here the canonical projection is part of the family definition.
    """
    graph_hash = embedding_hash(graph)

    cache_key, cache_path = projection_cache_key(
        family=family,
        parameter_value=parameter_value,
        graph_hash=graph_hash,
    )

    cached = load_projection_cache(
        cache_path,
        cache_key=cache_key,
        graph_hash=graph_hash,
    )

    if cached is not None:
        cached["cache_key"] = cache_key
        cached["graph_hash"] = graph_hash
        return cached

    start = time.perf_counter()

    projection = select_projection(
        graph,
        rotation_angles=(0.0, 0.0, 0.0),
        rotation_order=ROTATION_ORDER,
        num_rotation_samples=1,
    )

    if COMPUTE_FRONTIER_DIAGNOSTICS:
        frontier = factorized_frontier_width(projection.processor)
    else:
        frontier = {}

    projection_seconds = time.perf_counter() - start

    save_projection_cache(
        cache_path,
        cache_key=cache_key,
        graph_hash=graph_hash,
        projection=projection,
        frontier=frontier,
    )

    return {
        "processor": projection.processor,
        "rotation_angles": projection.rotation_angles,
        "rotation_order": projection.rotation_order,
        "pd_code": projection.pd_code,
        "num_crossings": int(projection.num_crossings),
        "frontier": dict(frontier),
        "candidate_records": [
            {
                "angles": [0.0, 0.0, 0.0],
                "crossings": int(projection.num_crossings),
                "canonical": True,
            }
        ],
        "source": "computed",
        "projection_seconds": projection_seconds,
        "cache_key": cache_key,
        "graph_hash": graph_hash,
    }


In [ ]:
def laurent_coefficients_exact(
    expr: sp.Expr,
    variable: sp.Symbol,
) -> dict[int, int]:
    """
    Fast exact Laurent extraction.

    The Yamada result is already an expanded Laurent polynomial.  Avoiding
    sp.cancel() on the whole expression substantially reduces post-processing
    time for large m.
    """
    expr = sp.expand(expr)

    if expr == 0:
        return {}

    coefficients: dict[int, sp.Expr] = {}

    for term in sp.Add.make_args(expr):
        coefficient, exponent = term.as_coeff_exponent(variable)

        if exponent.is_Integer is not True:
            raise ValueError(
                f"Noninteger Laurent exponent in term {term!r}."
            )

        if variable in coefficient.free_symbols:
            raise ValueError(
                f"Could not isolate Laurent coefficient in term {term!r}."
            )

        exponent = int(exponent)
        coefficients[exponent] = (
            coefficients.get(exponent, sp.Integer(0))
            + coefficient
        )

    result: dict[int, int] = {}

    for exponent, coefficient in sorted(coefficients.items()):
        coefficient = sp.expand(coefficient)

        if coefficient == 0:
            continue

        if coefficient.is_Integer is not True:
            raise ValueError(
                f"Noninteger Laurent coefficient: {coefficient!r}."
            )

        result[int(exponent)] = int(coefficient)

    return result


def exact_same(left: sp.Expr, right: sp.Expr) -> bool:
    return sp.expand(left - right) == 0


In [ ]:
def graph_summary(graph: nx.MultiGraph) -> dict:
    degrees = [int(deg) for _, deg in graph.degree()]
    components = nx.number_connected_components(nx.Graph(graph))
    beta1 = (
        graph.number_of_edges()
        - graph.number_of_nodes()
        + components
    )

    return {
        "vertices": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "components": components,
        "beta1": beta1,
        "max_degree": max(degrees),
        "is_subcubic": max(degrees) <= 3,
        "is_cubic": all(deg == 3 for deg in degrees),
        "degree_histogram_json": json.dumps(
            dict(sorted(Counter(degrees).items())),
            separators=(",", ":"),
        ),
    }



def read_csv(path: Path) -> list[dict]:
    if not path.exists():
        return []

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:
        return list(csv.DictReader(handle))


def atomic_write_rows(
    rows: list[dict],
    path: Path,
    *,
    fields: list[str],
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            extrasaction="ignore",
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(
                {
                    field: row.get(field, "")
                    for field in fields
                }
            )

        handle.flush()
        os.fsync(handle.fileno())

    tmp.replace(path)


### Arbitrary mixed-word constructor and master conjecture

In [ ]:

# ============================================================
# Mixed-word constructor
# ============================================================

def normalize_word(word: str) -> str:
    word = str(word).strip().upper().replace("Λ", "L")
    if not word:
        raise ValueError("word must be nonempty.")
    bad = sorted(set(word) - set(ALPHABET))
    if bad:
        raise ValueError(f"Unknown motif symbols {bad}; use only {ALPHABET}.")
    return word


def word_counts(word: str) -> dict[str, int]:
    word = normalize_word(word)
    counts = Counter(word)
    return {symbol: int(counts.get(symbol, 0)) for symbol in ALPHABET}


def word_signature(word: str) -> str:
    counts = word_counts(word)
    return f"L{counts['L']}_D{counts['D']}_B{counts['B']}"


def expected_permutation_count(word: str) -> int:
    counts = word_counts(word)
    m = len(normalize_word(word))
    denom = math.prod(math.factorial(counts[s]) for s in ALPHABET)
    return math.factorial(m) // denom


def _add_rung_at_boundary(graph: nx.MultiGraph, j: int, *, cell_index: int) -> None:
    a_node = ("a", int(j))
    b_node = ("b", int(j))
    p = np.asarray(graph.nodes[a_node]["pos"], dtype=float)
    p_other = np.asarray(graph.nodes[b_node]["pos"], dtype=float)
    graph.add_edge(
        a_node,
        b_node,
        pts=line(p, p_other, 2),
        role="rung",
        rung_index=int(j),
        motif_cell=int(cell_index),
        motif_symbol="L",
    )


def _add_side_pair_for_cell(
    graph: nx.MultiGraph,
    j: int,
    jp: int,
    *,
    direction: str,
    cell_index: int,
    motif_symbol: str,
) -> None:
    lower_j, upper_j = lower_upper_nodes_for_boundary(graph, j)
    lower_p, upper_p = lower_upper_nodes_for_boundary(graph, jp)

    if direction == "down":
        node0, node1 = lower_j, lower_p
        role = "bottom_pair_edge"
    elif direction == "up":
        node0, node1 = upper_j, upper_p
        role = "top_pair_edge"
    else:
        raise ValueError("direction must be 'down' or 'up'.")

    p0 = np.asarray(graph.nodes[node0]["pos"], dtype=float)
    p1 = np.asarray(graph.nodes[node1]["pos"], dtype=float)

    graph.add_edge(
        node0,
        node1,
        pts=side_connection_arc(
            p0,
            p1,
            direction=direction,
            pair_index=int(cell_index),
        ),
        role=role,
        pair_index=int(cell_index),
        motif_cell=int(cell_index),
        motif_symbol=str(motif_symbol),
    )


def mixed_theta_word(word: str) -> nx.MultiGraph:
    """
    One letter occupies one m-cell = boundaries (2k-1, 2k).

      L: two rungs, one at each boundary;
      D: one lower pairing between those boundaries;
      B: one lower + one upper pairing.

    Therefore L^m, D^m and B^m exactly recover the three homogeneous families.
    """
    word = normalize_word(word)
    m = len(word)
    graph = subdivided_theta_skeleton(m)
    q = int(graph.graph["q"])

    for k, motif in enumerate(word, start=1):
        j = 2 * k - 1
        jp = 2 * k

        if motif == "L":
            _add_rung_at_boundary(graph, j, cell_index=k)
            _add_rung_at_boundary(graph, jp, cell_index=k)

        elif motif == "D":
            _add_side_pair_for_cell(
                graph, j, jp,
                direction="down",
                cell_index=k,
                motif_symbol=motif,
            )

        elif motif == "B":
            _add_side_pair_for_cell(
                graph, j, jp,
                direction="down",
                cell_index=k,
                motif_symbol=motif,
            )
            _add_side_pair_for_cell(
                graph, j, jp,
                direction="up",
                cell_index=k,
                motif_symbol=motif,
            )

        else:
            raise AssertionError(f"Unhandled motif {motif!r}.")

    counts = word_counts(word)
    nL, nD, nB = counts["L"], counts["D"], counts["B"]

    graph.graph.update(
        family="mixed_theta_word",
        family_label=f"Mixed theta word {word}",
        word=word,
        word_length=m,
        word_signature=word_signature(word),
        expected_edges=(2 * q + 1) + 2 * nL + nD + 2 * nB,
        expected_rungs=2 * nL,
        expected_bottom_pair_edges=nD + nB,
        expected_top_pair_edges=nB,
        expected_degree_histogram={
            **({2: 2 * nD} if nD else {}),
            3: 2 + 4 * nL + 2 * nD + 4 * nB,
        },
    )

    return ensure_embedding(graph, copy=False, normalize=True)


# ============================================================
# Conjectured common three-channel formula
# ============================================================

a = A**2 + 1
b = A**2 - A + 1
c = A**4 + A**2 + 1
phi = A**4 + A**3 + A**2 + A + 1
d = 2 * A**2 - A + 2

CHANNEL_FACTORS = {
    "L": (a**2, A**2 * b**2, A**8),
    "D": (a, A**2 * b, -A**7),
    "B": (a * b, -A**2 * (A - 1) * d, A**5 * (A + 1) * b),
}
CLOSURE_WEIGHTS = (a, -c, -A**2 * phi)


def master_prediction(word: str) -> sp.Expr:
    # Never used in graph construction or actual Yamada evaluation.
    word = normalize_word(word)
    products = [sp.Integer(1), sp.Integer(1), sp.Integer(1)]

    for motif in word:
        factors = CHANNEL_FACTORS[motif]
        for channel in range(3):
            products[channel] *= factors[channel]

    return sp.expand(
        sum(
            CLOSURE_WEIGHTS[channel] * products[channel]
            for channel in range(3)
        )
    )


def known_homogeneous_formula(motif: str, m: int) -> sp.Expr:
    motif = normalize_word(motif)
    if len(motif) != 1:
        raise ValueError("motif must be a single symbol.")
    m = int(m)

    if motif == "L":
        return sp.expand(
            (A**2 + 1)**(2*m + 1)
            - A**(2*m) * (A**4 + A**2 + 1) * (A**2 - A + 1)**(2*m)
            - A**(8*m + 2) * (A**4 + A**3 + A**2 + A + 1)
        )
    if motif == "D":
        return sp.expand(
            (A**2 + 1)**(m + 1)
            - A**(2*m) * (A**4 + A**2 + 1) * (A**2 - A + 1)**m
            + (-1)**(m + 1) * A**(7*m + 2) * (A**4 + A**3 + A**2 + A + 1)
        )
    if motif == "B":
        return sp.expand(
            (A**2 + 1)**(m + 1) * (A**2 - A + 1)**m
            + (-1)**(m + 1)
              * A**(2*m) * (A**4 + A**2 + 1)
              * (A - 1)**m * (2*A**2 - A + 2)**m
            - A**(5*m + 2)
              * (A**4 + A**3 + A**2 + A + 1)
              * (A + 1)**m * (A**2 - A + 1)**m
        )
    raise AssertionError("unreachable")


def coeffs_json(coeffs: dict[int, int]) -> str:
    return json.dumps(coeffs, sort_keys=True, separators=(",", ":"))


def coeffs_hash(coeffs: dict[int, int]) -> str:
    return hashlib.sha256(coeffs_json(coeffs).encode()).hexdigest()


def subtract_coeff_dicts(
    left: dict[int, int],
    right: dict[int, int],
) -> dict[int, int]:
    powers = set(left) | set(right)
    out = {}
    for power in sorted(powers):
        value = int(left.get(power, 0)) - int(right.get(power, 0))
        if value:
            out[int(power)] = int(value)
    return out


### Mixed-word preflights, exact evaluator, and word plan

In [ ]:

def homogeneous_builder_for_symbol(symbol: str):
    return {
        "L": crosslinked_theta_ladder,
        "D": bottom_paired_theta,
        "B": two_sided_paired_theta,
    }[symbol]


def run_mixed_preflights() -> None:
    if RUN_HOMOGENEOUS_RECOVERY_PREFLIGHT:
        for symbol in ALPHABET:
            builder = homogeneous_builder_for_symbol(symbol)
            for m in HOMOGENEOUS_RECOVERY_M:
                mixed = mixed_theta_word(symbol * int(m))
                original = builder(int(m))

                if embedding_hash(mixed) != embedding_hash(original):
                    raise AssertionError(
                        f"Homogeneous recovery failed for {symbol}^{m}: "
                        "mixed-word embedding differs from original constructor."
                    )

                predicted = master_prediction(symbol * int(m))
                known = known_homogeneous_formula(symbol, int(m))
                if not exact_same(predicted, known):
                    raise AssertionError(
                        f"Master formula does not algebraically reduce to the "
                        f"known {symbol}^{m} homogeneous formula."
                    )

        print(
            "PASS: mixed-word constructor exactly recovers all three original "
            "homogeneous embeddings and formulas on the preflight values."
        )

    if RUN_REPRESENTATIVE_3D_CONTACT_AUDIT:
        for word in REPRESENTATIVE_MIXED_WORDS:
            graph = mixed_theta_word(word)
            certify_theta_derived_family(
                graph,
                require_cubic=False,
                audit_3d_contacts=True,
            )
        print(
            "PASS: representative mixed words passed the 3-D contact and "
            "constituent-cycle audit."
        )

    if RUN_CANONICAL_PROJECTION_PREFLIGHT:
        for word in CANONICAL_PREFLIGHT_WORDS:
            graph = mixed_theta_word(word)
            certify_theta_derived_family(
                graph,
                require_cubic=False,
                audit_3d_contacts=False,
            )
            processor = PDCode(graph)
            processor.compute(
                rotation_angles=(0.0, 0.0, 0.0),
                rotation_order=ROTATION_ORDER,
            )
            expected = int(graph.graph["expected_canonical_crossings"])
            actual = len(processor.crossings)
            if actual != expected:
                raise AssertionError(
                    f"Mixed canonical projection failed for word={word}: "
                    f"expected {expected}, got {actual}."
                )
            print(
                f"PASS canonical mixed word {word:8s}: crossings={actual}",
                flush=True,
            )


FULL_FIELDS = [
    "case_id","split","word","word_length","n_L","n_D","n_B","signature",
    "expected_permutation_count","reverse_word","is_homogeneous",
    "branch","git_commit","constructor_version","compute_config_sha256",
    "run_config_sha256","embedding_sha256","projection_source",
    "projection_cache_key","rotation_order","vertices","edges","components",
    "beta1","max_degree","is_subcubic","is_cubic","degree_histogram_json",
    "constituent_knot","selected_crossings","expected_canonical_crossings",
    "canonical_crossing_count_pass","pd_code","actual_yamada_A_normalized",
    "predicted_master_A","actual_coefficients_json",
    "predicted_coefficients_json","difference_coefficients_json",
    "difference_term_count","actual_polynomial_sha256",
    "predicted_polynomial_sha256","master_formula_pass",
    "laurent_min_exponent","laurent_max_exponent","laurent_span",
    "laurent_term_count","construction_seconds","certification_seconds",
    "projection_seconds","yamada_seconds","prediction_seconds",
    "postprocess_seconds","case_seconds","status","error_type","error_message",
]


def evaluate_mixed_word(word: str, *, split: str) -> dict:
    total_start = time.perf_counter()
    word = normalize_word(word)
    m = len(word)
    counts = word_counts(word)

    stage = time.perf_counter()
    graph = mixed_theta_word(word)
    construction_seconds = time.perf_counter() - stage

    stage = time.perf_counter()
    certify_theta_derived_family(
        graph,
        require_cubic=False,
        audit_3d_contacts=False,
    )
    summary = graph_summary(graph)
    certification_seconds = time.perf_counter() - stage

    stage = time.perf_counter()
    chosen = choose_projection(
        graph,
        family=f"mixed_theta_word_{word}",
        parameter_value=m,
    )
    projection_seconds = time.perf_counter() - stage

    expected_crossings = int(graph.graph["expected_canonical_crossings"])
    actual_crossings = int(chosen["num_crossings"])
    crossing_pass = actual_crossings == expected_crossings
    if not crossing_pass:
        raise AssertionError(
            f"word={word}: canonical projection has {actual_crossings} crossings; "
            f"expected exactly {expected_crossings}."
        )

    # Actual Yamada is computed first, independently.
    stage = time.perf_counter()
    computer = Yamada(
        vertices=list(chosen["processor"].vertices.values()),
        crossings=list(chosen["processor"].crossings.values()),
        arcs=list(chosen["processor"].arcs.values()),
    )
    actual = sp.expand(
        computer.compute(
            A,
            normalize=NORMALIZE_YAMADA,
        )
    )
    yamada_seconds = time.perf_counter() - stage

    # Only after the actual result exists do we evaluate the conjecture.
    stage = time.perf_counter()
    predicted = master_prediction(word)
    prediction_seconds = time.perf_counter() - stage

    stage = time.perf_counter()
    actual_coeffs = laurent_coefficients_exact(actual, A)
    predicted_coeffs = laurent_coefficients_exact(predicted, A)
    difference_coeffs = subtract_coeff_dicts(actual_coeffs, predicted_coeffs)
    formula_pass = actual_coeffs == predicted_coeffs

    min_exp = min(actual_coeffs) if actual_coeffs else ""
    max_exp = max(actual_coeffs) if actual_coeffs else ""
    span = max_exp - min_exp if actual_coeffs else 0

    actual_hash = coeffs_hash(actual_coeffs)
    predicted_hash = coeffs_hash(predicted_coeffs)

    postprocess_seconds = time.perf_counter() - stage
    case_seconds = time.perf_counter() - total_start

    return {
        "case_id": f"mixed_theta_word__{word}",
        "split": str(split),
        "word": word,
        "word_length": m,
        "n_L": counts["L"],
        "n_D": counts["D"],
        "n_B": counts["B"],
        "signature": word_signature(word),
        "expected_permutation_count": expected_permutation_count(word),
        "reverse_word": word[::-1],
        "is_homogeneous": len(set(word)) == 1,
        "branch": CURRENT_BRANCH,
        "git_commit": GIT_COMMIT,
        "constructor_version": CONSTRUCTOR_VERSION,
        "compute_config_sha256": COMPUTE_CONFIG_SHA256,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "embedding_sha256": chosen["graph_hash"],
        "projection_source": chosen["source"],
        "projection_cache_key": chosen["cache_key"],
        "rotation_order": chosen["rotation_order"],
        **summary,
        "constituent_knot": graph.graph.get("constituent_knot", ""),
        "selected_crossings": actual_crossings,
        "expected_canonical_crossings": expected_crossings,
        "canonical_crossing_count_pass": crossing_pass,
        "pd_code": chosen["pd_code"],
        "actual_yamada_A_normalized": sp.sstr(actual),
        "predicted_master_A": sp.sstr(predicted),
        "actual_coefficients_json": coeffs_json(actual_coeffs),
        "predicted_coefficients_json": coeffs_json(predicted_coeffs),
        "difference_coefficients_json": coeffs_json(difference_coeffs),
        "difference_term_count": len(difference_coeffs),
        "actual_polynomial_sha256": actual_hash,
        "predicted_polynomial_sha256": predicted_hash,
        "master_formula_pass": formula_pass,
        "laurent_min_exponent": min_exp,
        "laurent_max_exponent": max_exp,
        "laurent_span": span,
        "laurent_term_count": len(actual_coeffs),
        "construction_seconds": construction_seconds,
        "certification_seconds": certification_seconds,
        "projection_seconds": projection_seconds,
        "yamada_seconds": yamada_seconds,
        "prediction_seconds": prediction_seconds,
        "postprocess_seconds": postprocess_seconds,
        "case_seconds": case_seconds,
        "status": "success",
        "error_type": "",
        "error_message": "",
    }


def all_words_of_length(m: int) -> list[str]:
    return [
        "".join(chars)
        for chars in itertools.product(ALPHABET, repeat=int(m))
    ]


def all_count_signatures(m: int) -> list[tuple[int, int, int]]:
    out = []
    for nL in range(m + 1):
        for nD in range(m - nL + 1):
            nB = m - nL - nD
            if sum(value > 0 for value in (nL, nD, nB)) < 2:
                continue
            out.append((nL, nD, nB))
    return out


def words_for_counts(
    counts: tuple[int, int, int],
    *,
    limit: int,
    rng: random.Random,
) -> list[str]:
    nL, nD, nB = counts
    base = "L" * nL + "D" * nD + "B" * nB
    unique = sorted({"".join(p) for p in itertools.permutations(base)})

    if len(unique) <= limit:
        return unique

    selected = {unique[0], unique[-1], base, base[::-1]}
    remaining = [word for word in unique if word not in selected]
    need = max(0, int(limit) - len(selected))
    if need:
        selected.update(rng.sample(remaining, min(need, len(remaining))))

    return sorted(selected)[:limit]


def build_word_plan() -> dict[str, str]:
    plan: dict[str, str] = {}

    for m in range(1, MAX_EXHAUSTIVE_WORD_LENGTH + 1):
        for word in all_words_of_length(m):
            plan[word] = "exhaustive"

    rng = random.Random(RNG_SEED)

    for m in STRESS_LENGTHS:
        signatures = all_count_signatures(int(m))
        signatures.sort(
            key=lambda counts: (
                -(math.factorial(m)
                  // math.prod(math.factorial(v) for v in counts)),
                counts,
            )
        )
        chosen_signatures = signatures[:STRESS_COMPOSITION_CLASSES_PER_LENGTH]

        for counts in chosen_signatures:
            for word in words_for_counts(
                counts,
                limit=STRESS_PERMUTATIONS_PER_CLASS,
                rng=rng,
            ):
                plan.setdefault(word, "stress")

    return dict(sorted(plan.items(), key=lambda item: (len(item[0]), item[0])))


def sort_full_rows(rows: list[dict]) -> list[dict]:
    return sorted(
        rows,
        key=lambda row: (
            int(row.get("word_length", 0) or 0),
            str(row.get("word", "")),
        ),
    )


### Generate CSVs and print the decisive audit summary

In [ ]:

def validate_reusable_row(row: dict) -> bool:
    return (
        row.get("status") == "success"
        and row.get("branch") == CURRENT_BRANCH
        and row.get("git_commit") == GIT_COMMIT
        and row.get("constructor_version") == CONSTRUCTOR_VERSION
        and row.get("compute_config_sha256") == COMPUTE_CONFIG_SHA256
        and row.get("run_config_sha256") == RUN_CONFIG_SHA256
    )


def flush_full(rows_by_word: dict[str, dict]) -> None:
    atomic_write_rows(
        sort_full_rows(list(rows_by_word.values())),
        FULL_CSV,
        fields=FULL_FIELDS,
    )


run_mixed_preflights()

WORD_PLAN = build_word_plan()
print(
    f"Planned exact mixed-word cases: {len(WORD_PLAN)} "
    f"(exhaustive through length {MAX_EXHAUSTIVE_WORD_LENGTH}, "
    f"stress lengths={STRESS_LENGTHS})"
)

existing_rows = read_csv(FULL_CSV) if RESUME else []
rows_by_word: dict[str, dict] = {}

for row in existing_rows:
    word = row.get("word", "")
    if not word:
        continue
    if row.get("status") == "success" and not validate_reusable_row(row):
        raise RuntimeError(
            f"Existing successful row for word={word!r} was generated under "
            "another branch/commit/config. Rename or remove the old full CSV."
        )
    rows_by_word[word] = row

failures = []
new_cases_since_flush = 0

for index, (word, split) in enumerate(WORD_PLAN.items(), start=1):
    previous = rows_by_word.get(word)

    if RESUME and previous is not None and validate_reusable_row(previous):
        print(
            f"SKIP [{index:4d}/{len(WORD_PLAN):4d}] {word:8s} "
            f"crossings={previous['selected_crossings']} "
            f"pass={previous['master_formula_pass']}",
            flush=True,
        )
        continue

    try:
        row = evaluate_mixed_word(word, split=split)
        rows_by_word[word] = row
        new_cases_since_flush += 1

        verdict = "PASS" if row["master_formula_pass"] else "FAIL"
        print(
            f"{verdict} [{index:4d}/{len(WORD_PLAN):4d}] {word:8s} "
            f"m={row['word_length']} "
            f"sig={row['signature']:12s} "
            f"crossings={row['selected_crossings']:3d} "
            f"terms={row['laurent_term_count']:4d} "
            f"Yamada={row['yamada_seconds']:.4g}s "
            f"wall={row['case_seconds']:.4g}s",
            flush=True,
        )

        if new_cases_since_flush >= SAVE_EVERY_CASES:
            flush_full(rows_by_word)
            new_cases_since_flush = 0
            print("  checkpoint saved", flush=True)

    except KeyboardInterrupt:
        flush_full(rows_by_word)
        print(
            "\nInterrupted. Every completed exact word has been checkpointed.",
            flush=True,
        )
        raise

    except Exception as exc:
        counts = word_counts(word)
        error_row = {
            "case_id": f"mixed_theta_word__{word}",
            "split": split,
            "word": word,
            "word_length": len(word),
            "n_L": counts["L"],
            "n_D": counts["D"],
            "n_B": counts["B"],
            "signature": word_signature(word),
            "expected_permutation_count": expected_permutation_count(word),
            "reverse_word": word[::-1],
            "is_homogeneous": len(set(word)) == 1,
            "branch": CURRENT_BRANCH,
            "git_commit": GIT_COMMIT,
            "constructor_version": CONSTRUCTOR_VERSION,
            "compute_config_sha256": COMPUTE_CONFIG_SHA256,
            "run_config_sha256": RUN_CONFIG_SHA256,
            "status": "error",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }
        rows_by_word[word] = error_row
        failures.append((word, type(exc).__name__, str(exc)))
        new_cases_since_flush += 1

        print(
            f"ERROR [{index:4d}/{len(WORD_PLAN):4d}] {word}: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )

        if new_cases_since_flush >= SAVE_EVERY_CASES:
            flush_full(rows_by_word)
            new_cases_since_flush = 0

        if FAIL_FAST:
            flush_full(rows_by_word)
            raise

flush_full(rows_by_word)

print(f"\nFull exact ledger: {FULL_CSV.resolve()}")
if failures:
    print(f"WARNING: {len(failures)} word(s) failed computationally.")


def csv_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes"}


success_rows = [
    row
    for row in rows_by_word.values()
    if row.get("status") == "success"
    and row.get("word") in WORD_PLAN
]
success_rows = sort_full_rows(success_rows)

by_signature: dict[str, list[dict]] = defaultdict(list)
for row in success_rows:
    by_signature[row["signature"]].append(row)

CLASS_FIELDS = [
    "signature","word_length","n_L","n_D","n_B",
    "tested_permutations","expected_permutations","complete_class_coverage",
    "distinct_actual_polynomials","order_independent_on_tested_words",
    "master_formula_failures","all_master_formula_pass","representative_word",
    "representative_actual_yamada_A","representative_predicted_master_A",
    "distinct_actual_hashes_json","words_json",
]

class_rows = []
class_by_signature = {}
order_witnesses = []

for signature, group in sorted(
    by_signature.items(),
    key=lambda item: (int(item[1][0]["word_length"]), item[0]),
):
    group = sorted(group, key=lambda row: row["word"])
    first = group[0]
    expected = int(first["expected_permutation_count"])
    hashes = sorted({row["actual_polynomial_sha256"] for row in group})
    failure_count = sum(not csv_bool(row["master_formula_pass"]) for row in group)
    order_independent = len(hashes) == 1

    class_row = {
        "signature": signature,
        "word_length": int(first["word_length"]),
        "n_L": int(first["n_L"]),
        "n_D": int(first["n_D"]),
        "n_B": int(first["n_B"]),
        "tested_permutations": len(group),
        "expected_permutations": expected,
        "complete_class_coverage": len(group) == expected,
        "distinct_actual_polynomials": len(hashes),
        "order_independent_on_tested_words": order_independent,
        "master_formula_failures": failure_count,
        "all_master_formula_pass": failure_count == 0,
        "representative_word": first["word"],
        "representative_actual_yamada_A": first["actual_yamada_A_normalized"],
        "representative_predicted_master_A": first["predicted_master_A"],
        "distinct_actual_hashes_json": json.dumps(hashes, separators=(",", ":")),
        "words_json": json.dumps([row["word"] for row in group], separators=(",", ":")),
    }
    class_rows.append(class_row)
    class_by_signature[signature] = class_row

    if not order_independent:
        by_hash = defaultdict(list)
        for row in group:
            by_hash[row["actual_polynomial_sha256"]].append(row)
        hash_keys = sorted(by_hash)
        left = sorted(by_hash[hash_keys[0]], key=lambda r: r["word"])[0]
        right = sorted(by_hash[hash_keys[1]], key=lambda r: r["word"])[0]
        order_witnesses.append((class_row, left, right))

atomic_write_rows(class_rows, CLASS_CSV, fields=CLASS_FIELDS)

COUNTER_FIELDS = [
    "failure_type","word_length","signature","word_a","word_b",
    "actual_a","actual_b","predicted_master","difference_coefficients_json",
    "actual_hash_a","actual_hash_b",
]
counter_rows = []

for row in success_rows:
    if csv_bool(row["master_formula_pass"]):
        continue
    counter_rows.append({
        "failure_type": "master_formula_failure",
        "word_length": row["word_length"],
        "signature": row["signature"],
        "word_a": row["word"],
        "word_b": "",
        "actual_a": row["actual_yamada_A_normalized"],
        "actual_b": "",
        "predicted_master": row["predicted_master_A"],
        "difference_coefficients_json": row["difference_coefficients_json"],
        "actual_hash_a": row["actual_polynomial_sha256"],
        "actual_hash_b": "",
    })

for class_row, left, right in order_witnesses:
    counter_rows.append({
        "failure_type": "order_dependence_witness",
        "word_length": class_row["word_length"],
        "signature": class_row["signature"],
        "word_a": left["word"],
        "word_b": right["word"],
        "actual_a": left["actual_yamada_A_normalized"],
        "actual_b": right["actual_yamada_A_normalized"],
        "predicted_master": left["predicted_master_A"],
        "difference_coefficients_json": "",
        "actual_hash_a": left["actual_polynomial_sha256"],
        "actual_hash_b": right["actual_polynomial_sha256"],
    })

counter_rows.sort(
    key=lambda row: (
        int(row["word_length"]),
        row["failure_type"],
        row["signature"],
        row["word_a"],
        row["word_b"],
    )
)
atomic_write_rows(counter_rows, COUNTEREXAMPLE_CSV, fields=COUNTER_FIELDS)


def parse_coeffs(text: str) -> dict[int, int]:
    raw = json.loads(text)
    return {int(k): int(v) for k, v in raw.items()}


def evaluate_coeffs_at(coeffs: dict[int, int], value: int) -> sp.Rational:
    x = sp.Integer(int(value))
    total = sp.Integer(0)
    for power, coefficient in coeffs.items():
        total += sp.Integer(coefficient) * x**int(power)
    return sp.cancel(total)


actual_coeffs_by_word = {
    row["word"]: parse_coeffs(row["actual_coefficients_json"])
    for row in success_rows
}

basis_words = []
for length in range(1, HANKEL_BASIS_MAX_WORD_LENGTH + 1):
    basis_words.extend(all_words_of_length(length))

HANKEL_FIELDS = [
    "evaluation_A","basis_word_count","basis_max_word_length",
    "required_max_concatenated_length","complete","missing_words_json",
    "actual_hankel_rank","rank_le_3","interpretation",
]
hankel_rows = []

for eval_A in HANKEL_EVALUATION_POINTS:
    missing = sorted(
        {
            u + v
            for u in basis_words
            for v in basis_words
            if (u + v) not in actual_coeffs_by_word
        },
        key=lambda word: (len(word), word),
    )

    if missing:
        rank = ""
        rank_le_3 = ""
        interpretation = (
            "Incomplete: increase MAX_EXHAUSTIVE_WORD_LENGTH so every "
            "basis concatenation is present."
        )
    else:
        matrix_values = [
            [
                evaluate_coeffs_at(actual_coeffs_by_word[u + v], eval_A)
                for v in basis_words
            ]
            for u in basis_words
        ]
        rank = int(sp.Matrix(matrix_values).rank())
        rank_le_3 = rank <= 3
        if rank > 3:
            interpretation = (
                "FALSIFIES any 3-state linear word-transfer realization "
                "for this tested Hankel block at this evaluation point."
            )
        else:
            interpretation = (
                "Compatible with a <=3-state word-transfer realization on "
                "this finite Hankel block; not by itself a proof."
            )

    hankel_rows.append({
        "evaluation_A": int(eval_A),
        "basis_word_count": len(basis_words),
        "basis_max_word_length": HANKEL_BASIS_MAX_WORD_LENGTH,
        "required_max_concatenated_length": 2 * HANKEL_BASIS_MAX_WORD_LENGTH,
        "complete": not missing,
        "missing_words_json": json.dumps(missing, separators=(",", ":")),
        "actual_hankel_rank": rank,
        "rank_le_3": rank_le_3,
        "interpretation": interpretation,
    })

atomic_write_rows(hankel_rows, HANKEL_CSV, fields=HANKEL_FIELDS)


SHARE_FIELDS = [
    "word","word_length","n_L","n_D","n_B","signature","split",
    "selected_crossings","max_degree","is_subcubic",
    "actual_yamada_A_normalized","predicted_master_A","master_formula_pass",
    "difference_coefficients_json","actual_polynomial_sha256",
    "tested_permutations_in_class","expected_permutations_in_class",
    "complete_class_coverage","distinct_actual_polynomials_in_class",
    "order_independent_on_tested_words","branch","git_commit",
    "constructor_version",
]

share_rows = []
for row in success_rows:
    class_row = class_by_signature[row["signature"]]
    share_rows.append({
        **row,
        "tested_permutations_in_class": class_row["tested_permutations"],
        "expected_permutations_in_class": class_row["expected_permutations"],
        "complete_class_coverage": class_row["complete_class_coverage"],
        "distinct_actual_polynomials_in_class": class_row["distinct_actual_polynomials"],
        "order_independent_on_tested_words": class_row["order_independent_on_tested_words"],
    })

atomic_write_rows(share_rows, SHARE_CSV, fields=SHARE_FIELDS)


formula_failures = [
    row for row in success_rows
    if not csv_bool(row["master_formula_pass"])
]
order_dependent_classes = [
    row for row in class_rows
    if not csv_bool(row["order_independent_on_tested_words"])
]
complete_classes = [
    row for row in class_rows
    if csv_bool(row["complete_class_coverage"])
]

print("\n================ CONJECTURE AUDIT SUMMARY ================")
print(f"Successful exact words           : {len(success_rows)} / {len(WORD_PLAN)}")
print(f"Direct master-formula failures   : {len(formula_failures)}")
print(f"Permutation classes tested       : {len(class_rows)}")
print(f"Completely exhausted classes     : {len(complete_classes)}")
print(f"Order-dependent tested classes   : {len(order_dependent_classes)}")

for row in hankel_rows:
    print(
        f"Hankel @ A={row['evaluation_A']}: "
        f"rank={row['actual_hankel_rank']} "
        f"complete={row['complete']}"
    )

if formula_failures:
    first = sorted(
        formula_failures,
        key=lambda row: (int(row["word_length"]), row["word"]),
    )[0]
    print(
        "\nSMALLEST DIRECT COUNTEREXAMPLE:",
        first["word"],
        "signature=", first["signature"],
    )
else:
    print("\nNO DIRECT COUNTEREXAMPLE FOUND in the completed test set.")

if order_witnesses:
    class_row, left, right = sorted(
        order_witnesses,
        key=lambda item: (
            int(item[0]["word_length"]),
            item[0]["signature"],
        ),
    )[0]
    print(
        "SMALLEST ORDER-DEPENDENCE WITNESS:",
        left["word"], "vs", right["word"],
        "signature=", class_row["signature"],
    )
else:
    print("NO ORDER-DEPENDENCE WITNESS FOUND in the tested permutation classes.")

print("\nSaved outputs:")
print("  SHARE          :", SHARE_CSV.resolve())
print("  CLASS SUMMARY  :", CLASS_CSV.resolve())
print("  COUNTEREXAMPLES:", COUNTEREXAMPLE_CSV.resolve())
print("  HANKEL RANK    :", HANKEL_CSV.resolve())
print("  FULL LEDGER    :", FULL_CSV.resolve())


### Frozen count-only prediction: extreme out-of-range validation
The count-only master formula is frozen before these graphs are generated.

This validation uses **exactly 20 previously uncomputed mixed words**:

- \(m=101\): 5 words
- \(m=125\): 5 words
- \(m=150\): 5 words
- \(m=200\): 5 words

At each length the five words are chosen to test two different consequences:

1. **Order-independence:** two distinct words have exactly the same near-balanced
   motif counts \((n_L,n_D,n_B)\), but very different orderings.
2. **Composition extrapolation:** three further words use strongly \(L\)-heavy,
   \(D\)-heavy, and \(B\)-heavy compositions.

Thus the test probes lengths far outside the original discovery/stress range
(\(m\le 7\)), while simultaneously testing both the count-only formula and its
prediction that ordering is irrelevant.

Each case is checkpointed immediately so the notebook can be safely resumed.


In [ ]:

# ============================================================
# FROZEN COUNT-ONLY PREDICTION
# EXTREME OUT-OF-RANGE TEST: m = 101, 125, 150, 200
# EXACTLY FIVE WORDS PER LENGTH
# ============================================================

COUNT_FORMULA_VERSION = "2026-08-27.count-only-master.v1"
COUNT_FORMULA_TEXT = r"""
Upsilon_w(A) =
(A^2+1)^(m+n_L+1) (A^2-A+1)^n_B
+ (-1)^(n_B+1) A^(2m) (A^4+A^2+1)
  (A^2-A+1)^(2n_L+n_D) (A-1)^n_B (2A^2-A+2)^n_B
+ (-1)^(n_D+1) A^(8n_L+7n_D+5n_B+2)
  (A^4+A^3+A^2+A+1) (A+1)^n_B (A^2-A+1)^n_B,
m=n_L+n_D+n_B.
""".strip()

COUNT_FORMULA_SHA256 = hashlib.sha256(
    COUNT_FORMULA_TEXT.encode("utf-8")
).hexdigest()

UNSEEN_LENGTHS = [101, 125, 150, 200]
UNSEEN_WORDS_PER_LENGTH = 5

# Large cases are expensive enough that every completed graph should be
# persisted immediately.
UNSEEN_RESUME = True
UNSEEN_SAVE_EVERY_CASES = 1

UNSEEN_CSV = RESULTS_DIR / "05b_mixed_theta_count_formula_extreme_unseen.csv"
UNSEEN_CLASS_CSV = RESULTS_DIR / "05b_mixed_theta_count_formula_extreme_unseen_classes.csv"
UNSEEN_COUNTEREXAMPLE_CSV = (
    RESULTS_DIR / "05b_mixed_theta_count_formula_extreme_unseen_counterexamples.csv"
)


def count_only_prediction_from_counts(
    n_L: int,
    n_D: int,
    n_B: int,
) -> sp.Expr:
    """
    Frozen explicit prediction.

    This function depends only on motif counts.  It never uses word ordering
    or any computed Yamada value.
    """
    n_L = int(n_L)
    n_D = int(n_D)
    n_B = int(n_B)

    if min(n_L, n_D, n_B) < 0:
        raise ValueError("Motif counts must be nonnegative.")

    m = n_L + n_D + n_B

    aa = A**2 + 1
    bb = A**2 - A + 1
    cc = A**4 + A**2 + 1
    ph = A**4 + A**3 + A**2 + A + 1
    delta = 2*A**2 - A + 2

    term0 = aa**(m + n_L + 1) * bb**n_B

    term1 = (
        (-1)**(n_B + 1)
        * A**(2*m)
        * cc
        * bb**(2*n_L + n_D)
        * (A - 1)**n_B
        * delta**n_B
    )

    term2 = (
        (-1)**(n_D + 1)
        * A**(8*n_L + 7*n_D + 5*n_B + 2)
        * ph
        * (A + 1)**n_B
        * bb**n_B
    )

    return sp.expand(term0 + term1 + term2)


def count_only_prediction(word: str) -> sp.Expr:
    counts = word_counts(word)
    return count_only_prediction_from_counts(
        counts["L"],
        counts["D"],
        counts["B"],
    )


# Internal algebraic guard.  This checks only equivalence to the already-frozen
# three-channel formula; it does NOT use any held-out graph data.
for _probe_word in [
    "L", "D", "B",
    "LD", "DB", "BL",
    "LDB", "LLDBB", "LDBLDB",
]:
    if not exact_same(
        count_only_prediction(_probe_word),
        master_prediction(_probe_word),
    ):
        raise AssertionError(
            "Frozen count-only formula is inconsistent with the "
            f"pre-existing channel formula on {_probe_word!r}."
        )

print("PASS: frozen count-only formula agrees algebraically with the channel form.")


# ============================================================
# Five deliberately different words at each target length
# ============================================================

def near_balanced_counts(m: int) -> tuple[int, int, int]:
    """
    Split m as evenly as possible across L,D,B.
    """
    q, r = divmod(int(m), 3)
    counts = [q, q, q]
    for i in range(r):
        counts[i] += 1
    return tuple(counts)


def heavy_counts(m: int, dominant: str) -> tuple[int, int, int]:
    """
    About 60% dominant motif and 20% of each other motif, adjusted exactly to m.
    Returns counts in (L,D,B) order.
    """
    m = int(m)
    dominant_count = int(round(0.60 * m))
    remainder = m - dominant_count
    secondary_1 = remainder // 2
    secondary_2 = remainder - secondary_1

    if dominant == "L":
        return (dominant_count, secondary_1, secondary_2)
    if dominant == "D":
        return (secondary_1, dominant_count, secondary_2)
    if dominant == "B":
        return (secondary_1, secondary_2, dominant_count)
    raise ValueError("dominant must be L, D, or B")


def block_word(counts: tuple[int, int, int]) -> str:
    nL, nD, nB = map(int, counts)
    return "L"*nL + "D"*nD + "B"*nB


def cyclic_interleave_word(
    counts: tuple[int, int, int],
    cycle: tuple[str, str, str] = ("L", "D", "B"),
) -> str:
    """
    Deterministically interleave motifs as much as possible while preserving
    the exact requested counts.
    """
    remaining = {
        "L": int(counts[0]),
        "D": int(counts[1]),
        "B": int(counts[2]),
    }
    out = []

    while sum(remaining.values()) > 0:
        made_progress = False
        for symbol in cycle:
            if remaining[symbol] > 0:
                out.append(symbol)
                remaining[symbol] -= 1
                made_progress = True
        if not made_progress:
            raise AssertionError("Interleaving stalled unexpectedly.")

    return "".join(out)


def spread_dominant_word(
    counts: tuple[int, int, int],
    dominant: str,
) -> str:
    """
    A deterministic non-block ordering for a heavy composition.
    It alternates the dominant motif with whichever minority motif is currently
    most abundant, then appends any residual symbols in a cyclic order.
    """
    remaining = {
        "L": int(counts[0]),
        "D": int(counts[1]),
        "B": int(counts[2]),
    }
    minorities = [s for s in ALPHABET if s != dominant]
    out = []

    while sum(remaining.values()) > 0:
        if remaining[dominant] > 0:
            out.append(dominant)
            remaining[dominant] -= 1

        available_minorities = [
            s for s in minorities if remaining[s] > 0
        ]
        if available_minorities:
            chosen = max(
                available_minorities,
                key=lambda s: (remaining[s], s),
            )
            out.append(chosen)
            remaining[chosen] -= 1

        # If the dominant motif is exhausted first, consume the rest cyclically.
        if remaining[dominant] == 0:
            for s in ("L", "D", "B"):
                while remaining[s] > 0:
                    out.append(s)
                    remaining[s] -= 1

    return "".join(out)


def make_five_unseen_words(m: int) -> list[dict]:
    """
    Exactly five words:
      1. balanced counts, block ordering
      2. same balanced counts, strongly interleaved ordering
      3. L-heavy composition
      4. D-heavy composition
      5. B-heavy composition
    """
    balanced = near_balanced_counts(m)
    l_heavy = heavy_counts(m, "L")
    d_heavy = heavy_counts(m, "D")
    b_heavy = heavy_counts(m, "B")

    cases = [
        {
            "test_role": "balanced_block",
            "counts": balanced,
            "word": block_word(balanced),
            "order_pair_id": f"m{m}_balanced",
        },
        {
            "test_role": "balanced_interleaved",
            "counts": balanced,
            "word": cyclic_interleave_word(
                balanced,
                cycle=("L", "B", "D"),
            ),
            "order_pair_id": f"m{m}_balanced",
        },
        {
            "test_role": "L_heavy",
            "counts": l_heavy,
            "word": spread_dominant_word(l_heavy, "L"),
            "order_pair_id": "",
        },
        {
            "test_role": "D_heavy",
            "counts": d_heavy,
            "word": spread_dominant_word(d_heavy, "D"),
            "order_pair_id": "",
        },
        {
            "test_role": "B_heavy",
            "counts": b_heavy,
            "word": spread_dominant_word(b_heavy, "B"),
            "order_pair_id": "",
        },
    ]

    words = [case["word"] for case in cases]

    if len(cases) != UNSEEN_WORDS_PER_LENGTH:
        raise AssertionError("Expected exactly five unseen cases per length.")
    if len(set(words)) != UNSEEN_WORDS_PER_LENGTH:
        raise AssertionError(
            f"m={m}: the five designed held-out words are not all distinct."
        )
    if any(len(word) != int(m) for word in words):
        raise AssertionError(f"m={m}: a held-out word has the wrong length.")

    for case in cases:
        observed = word_counts(case["word"])
        expected = {
            "L": int(case["counts"][0]),
            "D": int(case["counts"][1]),
            "B": int(case["counts"][2]),
        }
        if observed != expected:
            raise AssertionError(
                f"m={m}, role={case['test_role']}: "
                f"count mismatch {observed} != {expected}"
            )

    return cases


def build_extreme_unseen_plan() -> dict[str, dict]:
    plan = {}

    for m in UNSEEN_LENGTHS:
        cases = make_five_unseen_words(m)
        for within_length_index, case in enumerate(cases, start=1):
            word = case["word"]
            counts = word_counts(word)

            plan[word] = {
                "target_m": int(m),
                "within_length_index": int(within_length_index),
                "test_role": case["test_role"],
                "order_pair_id": case["order_pair_id"],
                "designed_n_L": counts["L"],
                "designed_n_D": counts["D"],
                "designed_n_B": counts["B"],
            }

    # --------------------------------------------------------
    # Leakage / design guards
    # --------------------------------------------------------
    original_words = set(WORD_PLAN)
    overlap = sorted(set(plan) & original_words)
    if overlap:
        raise AssertionError(
            "UNSEEN LEAKAGE: extreme held-out words overlap WORD_PLAN."
        )

    if len(plan) != len(UNSEEN_LENGTHS) * UNSEEN_WORDS_PER_LENGTH:
        raise AssertionError(
            f"Expected exactly {len(UNSEEN_LENGTHS) * UNSEEN_WORDS_PER_LENGTH} "
            f"held-out words; generated {len(plan)}."
        )

    for m in UNSEEN_LENGTHS:
        at_m = [word for word in plan if len(word) == int(m)]
        if len(at_m) != UNSEEN_WORDS_PER_LENGTH:
            raise AssertionError(
                f"m={m}: expected exactly 5 words, found {len(at_m)}."
            )

    if min(len(word) for word in plan) <= max(len(word) for word in original_words):
        raise AssertionError(
            "Held-out lengths are not strictly beyond the original WORD_PLAN."
        )

    return dict(
        sorted(
            plan.items(),
            key=lambda item: (
                len(item[0]),
                int(item[1]["within_length_index"]),
            ),
        )
    )


UNSEEN_PLAN = build_extreme_unseen_plan()

UNSEEN_CONFIG = {
    "prediction_formula_version": COUNT_FORMULA_VERSION,
    "prediction_formula_sha256": COUNT_FORMULA_SHA256,
    "target_lengths": UNSEEN_LENGTHS,
    "words_per_length": UNSEEN_WORDS_PER_LENGTH,
    "selection_design": [
        "balanced_block",
        "balanced_interleaved_same_counts",
        "L_heavy",
        "D_heavy",
        "B_heavy",
    ],
    "constructor_version": CONSTRUCTOR_VERSION,
    "compute_config_sha256": COMPUTE_CONFIG_SHA256,
}

UNSEEN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(
        UNSEEN_CONFIG,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

print("Frozen formula SHA256 :", COUNT_FORMULA_SHA256)
print("Unseen config SHA256  :", UNSEEN_CONFIG_SHA256)
print("Held-out target lengths:", UNSEEN_LENGTHS)
print("Held-out words/length :", UNSEEN_WORDS_PER_LENGTH)
print("Total held-out words  :", len(UNSEEN_PLAN))
print("Overlap with WORD_PLAN: 0")

print("\nHeld-out design:")
for m in UNSEEN_LENGTHS:
    rows = [
        (word, meta)
        for word, meta in UNSEEN_PLAN.items()
        if len(word) == int(m)
    ]
    print(f"\nm={m}")
    for word, meta in rows:
        counts = word_counts(word)
        print(
            f"  {meta['within_length_index']}. "
            f"{meta['test_role']:20s} "
            f"counts=(L={counts['L']},D={counts['D']},B={counts['B']}) "
            f"prefix={word[:30]}..."
        )


# ============================================================
# Exact held-out evaluator
# ============================================================

UNSEEN_FIELDS = [
    "case_id",
    "split",
    "target_m",
    "within_length_index",
    "test_role",
    "order_pair_id",
    "word",
    "word_length",
    "n_L",
    "n_D",
    "n_B",
    "signature",
    "branch",
    "git_commit",
    "constructor_version",
    "compute_config_sha256",
    "unseen_config_sha256",
    "prediction_formula_version",
    "prediction_formula_sha256",
    "prediction_frozen_before_holdout",
    "selected_crossings",
    "max_degree",
    "is_subcubic",
    "actual_yamada_A_normalized",
    "predicted_count_formula_A",
    "actual_coefficients_json",
    "predicted_count_coefficients_json",
    "difference_coefficients_json",
    "difference_term_count",
    "actual_polynomial_sha256",
    "predicted_polynomial_sha256",
    "count_formula_pass",
    "construction_seconds",
    "certification_seconds",
    "projection_seconds",
    "yamada_seconds",
    "prediction_seconds",
    "case_seconds",
    "status",
    "error_type",
    "error_message",
]


def evaluate_extreme_unseen_word(word: str, metadata: dict) -> dict:
    """
    Actual Yamada is computed independently first by evaluate_mixed_word().
    The frozen count-only prediction is evaluated only afterwards.
    """
    row = evaluate_mixed_word(word, split="extreme_unseen")

    prediction_start = time.perf_counter()
    prediction = count_only_prediction(word)
    prediction_seconds = time.perf_counter() - prediction_start

    actual_coeffs = parse_coeffs(row["actual_coefficients_json"])
    predicted_coeffs = laurent_coefficients_exact(prediction, A)
    difference = subtract_coeff_dicts(
        actual_coeffs,
        predicted_coeffs,
    )

    return {
        **row,
        **metadata,
        "unseen_config_sha256": UNSEEN_CONFIG_SHA256,
        "prediction_formula_version": COUNT_FORMULA_VERSION,
        "prediction_formula_sha256": COUNT_FORMULA_SHA256,
        "prediction_frozen_before_holdout": True,
        "predicted_count_formula_A": sp.sstr(prediction),
        "predicted_count_coefficients_json": coeffs_json(predicted_coeffs),
        "difference_coefficients_json": coeffs_json(difference),
        "difference_term_count": len(difference),
        "predicted_polynomial_sha256": coeffs_hash(predicted_coeffs),
        "count_formula_pass": actual_coeffs == predicted_coeffs,
        "prediction_seconds": prediction_seconds,
    }


def validate_reusable_unseen_row(row: dict) -> bool:
    return (
        row.get("status") == "success"
        and row.get("branch") == CURRENT_BRANCH
        and row.get("git_commit") == GIT_COMMIT
        and row.get("constructor_version") == CONSTRUCTOR_VERSION
        and row.get("compute_config_sha256") == COMPUTE_CONFIG_SHA256
        and row.get("unseen_config_sha256") == UNSEEN_CONFIG_SHA256
        and row.get("prediction_formula_sha256") == COUNT_FORMULA_SHA256
        and csv_bool(row.get("prediction_frozen_before_holdout"))
    )


def flush_extreme_unseen(rows_by_word: dict[str, dict]) -> None:
    rows = sorted(
        rows_by_word.values(),
        key=lambda row: (
            int(row.get("target_m", 0) or 0),
            int(row.get("within_length_index", 0) or 0),
        ),
    )
    atomic_write_rows(
        rows,
        UNSEEN_CSV,
        fields=UNSEEN_FIELDS,
    )


existing_unseen = read_csv(UNSEEN_CSV) if UNSEEN_RESUME else []
unseen_rows_by_word = {}

for row in existing_unseen:
    word = row.get("word", "")
    if not word:
        continue

    if row.get("status") == "success" and not validate_reusable_unseen_row(row):
        raise RuntimeError(
            f"Existing held-out row for word length {len(word)} was generated "
            "under another formula/commit/config. Rename or remove the extreme "
            "unseen CSV before continuing."
        )

    unseen_rows_by_word[word] = row


unseen_errors = []
new_since_flush = 0

for index, (word, metadata) in enumerate(UNSEEN_PLAN.items(), start=1):
    previous = unseen_rows_by_word.get(word)

    if (
        UNSEEN_RESUME
        and previous is not None
        and validate_reusable_unseen_row(previous)
    ):
        print(
            f"SKIP EXTREME [{index:2d}/{len(UNSEEN_PLAN):2d}] "
            f"m={len(word):3d} "
            f"{metadata['test_role']:20s} "
            f"pass={previous['count_formula_pass']}",
            flush=True,
        )
        continue

    try:
        row = evaluate_extreme_unseen_word(word, metadata)
        unseen_rows_by_word[word] = row
        new_since_flush += 1

        verdict = "PASS" if row["count_formula_pass"] else "FAIL"
        print(
            f"{verdict} EXTREME [{index:2d}/{len(UNSEEN_PLAN):2d}] "
            f"m={len(word):3d} "
            f"{metadata['test_role']:20s} "
            f"sig={row['signature']:17s} "
            f"crossings={int(row['selected_crossings']):4d} "
            f"terms={int(row['laurent_term_count']):5d} "
            f"Yamada={float(row['yamada_seconds']):.5g}s "
            f"wall={float(row['case_seconds']):.5g}s",
            flush=True,
        )

        if new_since_flush >= UNSEEN_SAVE_EVERY_CASES:
            flush_extreme_unseen(unseen_rows_by_word)
            new_since_flush = 0
            print("  extreme-unseen checkpoint saved", flush=True)

    except KeyboardInterrupt:
        flush_extreme_unseen(unseen_rows_by_word)
        print(
            "\nInterrupted: every completed extreme held-out case is saved.",
            flush=True,
        )
        raise

    except Exception as exc:
        counts = word_counts(word)
        error_row = {
            "case_id": f"mixed_theta_word__{word}",
            "split": "extreme_unseen",
            **metadata,
            "word": word,
            "word_length": len(word),
            "n_L": counts["L"],
            "n_D": counts["D"],
            "n_B": counts["B"],
            "signature": word_signature(word),
            "branch": CURRENT_BRANCH,
            "git_commit": GIT_COMMIT,
            "constructor_version": CONSTRUCTOR_VERSION,
            "compute_config_sha256": COMPUTE_CONFIG_SHA256,
            "unseen_config_sha256": UNSEEN_CONFIG_SHA256,
            "prediction_formula_version": COUNT_FORMULA_VERSION,
            "prediction_formula_sha256": COUNT_FORMULA_SHA256,
            "prediction_frozen_before_holdout": True,
            "status": "error",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }

        unseen_rows_by_word[word] = error_row
        unseen_errors.append((word, type(exc).__name__, str(exc)))
        new_since_flush += 1

        print(
            f"ERROR EXTREME [{index:2d}/{len(UNSEEN_PLAN):2d}] "
            f"m={len(word):3d} "
            f"{metadata['test_role']:20s}: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )

        flush_extreme_unseen(unseen_rows_by_word)
        new_since_flush = 0

        if FAIL_FAST:
            raise

flush_extreme_unseen(unseen_rows_by_word)


# ============================================================
# Exact order-independence audit for the balanced pair
# ============================================================

unseen_success = [
    row
    for word, row in unseen_rows_by_word.items()
    if word in UNSEEN_PLAN and row.get("status") == "success"
]

UNSEEN_CLASS_FIELDS = [
    "target_m",
    "order_pair_id",
    "n_L",
    "n_D",
    "n_B",
    "tested_orderings",
    "word_1_role",
    "word_2_role",
    "word_1_sha256",
    "word_2_sha256",
    "distinct_actual_polynomials",
    "order_independence_pass",
    "both_count_formula_pass",
]

class_rows = []

for m in UNSEEN_LENGTHS:
    pair_id = f"m{m}_balanced"
    pair = sorted(
        [
            row
            for row in unseen_success
            if row.get("order_pair_id") == pair_id
        ],
        key=lambda row: int(row["within_length_index"]),
    )

    if len(pair) == 2:
        hashes = {
            pair[0]["actual_polynomial_sha256"],
            pair[1]["actual_polynomial_sha256"],
        }

        class_rows.append({
            "target_m": int(m),
            "order_pair_id": pair_id,
            "n_L": int(pair[0]["n_L"]),
            "n_D": int(pair[0]["n_D"]),
            "n_B": int(pair[0]["n_B"]),
            "tested_orderings": 2,
            "word_1_role": pair[0]["test_role"],
            "word_2_role": pair[1]["test_role"],
            "word_1_sha256": pair[0]["actual_polynomial_sha256"],
            "word_2_sha256": pair[1]["actual_polynomial_sha256"],
            "distinct_actual_polynomials": len(hashes),
            "order_independence_pass": len(hashes) == 1,
            "both_count_formula_pass": (
                csv_bool(pair[0]["count_formula_pass"])
                and csv_bool(pair[1]["count_formula_pass"])
            ),
        })

atomic_write_rows(
    class_rows,
    UNSEEN_CLASS_CSV,
    fields=UNSEEN_CLASS_FIELDS,
)


# ============================================================
# Counterexample ledger
# ============================================================

UNSEEN_COUNTER_FIELDS = [
    "failure_type",
    "target_m",
    "within_length_index",
    "test_role",
    "word",
    "signature",
    "actual_yamada_A_normalized",
    "predicted_count_formula_A",
    "difference_coefficients_json",
    "actual_polynomial_sha256",
    "predicted_polynomial_sha256",
]

counter_rows = []

for row in unseen_success:
    if not csv_bool(row["count_formula_pass"]):
        counter_rows.append({
            "failure_type": "count_formula_failure",
            "target_m": row["target_m"],
            "within_length_index": row["within_length_index"],
            "test_role": row["test_role"],
            "word": row["word"],
            "signature": row["signature"],
            "actual_yamada_A_normalized": row["actual_yamada_A_normalized"],
            "predicted_count_formula_A": row["predicted_count_formula_A"],
            "difference_coefficients_json": row["difference_coefficients_json"],
            "actual_polynomial_sha256": row["actual_polynomial_sha256"],
            "predicted_polynomial_sha256": row["predicted_polynomial_sha256"],
        })

atomic_write_rows(
    counter_rows,
    UNSEEN_COUNTEREXAMPLE_CSV,
    fields=UNSEEN_COUNTER_FIELDS,
)


# ============================================================
# Final summary
# ============================================================

formula_failures = [
    row
    for row in unseen_success
    if not csv_bool(row["count_formula_pass"])
]

order_failures = [
    row
    for row in class_rows
    if not csv_bool(row["order_independence_pass"])
]

print("\n================ EXTREME BLIND VALIDATION ================")
print(f"Frozen formula SHA256        : {COUNT_FORMULA_SHA256}")
print(f"Target lengths               : {UNSEEN_LENGTHS}")
print(f"Requested words per length   : {UNSEEN_WORDS_PER_LENGTH}")
print(f"Planned held-out cases       : {len(UNSEEN_PLAN)}")
print(f"Successful exact evaluations : {len(unseen_success)}")
print(f"Computational errors         : {len(unseen_errors)}")
print(f"Formula failures             : {len(formula_failures)}")
print(f"Balanced order-pair failures : {len(order_failures)}")

for m in UNSEEN_LENGTHS:
    rows_m = [
        row
        for row in unseen_success
        if int(row["target_m"]) == int(m)
    ]
    passes_m = sum(
        csv_bool(row["count_formula_pass"])
        for row in rows_m
    )
    print(
        f"m={m:3d}: completed={len(rows_m)}/5, "
        f"formula_pass={passes_m}/{len(rows_m)}"
    )

if formula_failures:
    first = sorted(
        formula_failures,
        key=lambda row: (
            int(row["target_m"]),
            int(row["within_length_index"]),
        ),
    )[0]
    print(
        "\nSMALLEST EXTREME COUNTEREXAMPLE:",
        f"m={first['target_m']}",
        first["test_role"],
        first["signature"],
    )
else:
    print("\nNO EXTREME HELD-OUT COUNTEREXAMPLE FOUND among completed cases.")

print("\nSaved outputs:")
print("  EXACT 20-CASE TEST :", UNSEEN_CSV.resolve())
print("  ORDER-PAIR AUDIT   :", UNSEEN_CLASS_CSV.resolve())
print("  COUNTEREXAMPLES    :", UNSEEN_COUNTEREXAMPLE_CSV.resolve())


# Part III — Non-Abelian ordered pure-braid words

## Goal

The final family asks whether order can survive rather than collapse to motif
counts.  The abstract spatial graph is fixed: an \(8\)-vertex, \(12\)-edge
cubic multigraph carries three distinguished braid edges.  Their embedding is
modified by the pure-braid alphabet

$$
A=\sigma_1^2,\qquad B=\sigma_2^2.
$$

A graph is therefore indexed by an ordered word

$$
w=w_1\cdots w_m\in\{A,B\}^{*},
$$

and is denoted \(\mathcal N_w\).

## Exact transfer reconstruction

Short-word raw Yamada data define the residual Hankel representation

$$
H_{u,v}(Y)=R_{uv}(Y),\qquad
(H_A)_{u,v}(Y)=R_{uAv}(Y),\qquad
(H_B)_{u,v}(Y)=R_{uBv}(Y),
$$

from which

$$
T_A=H^{-1}H_A,\qquad
T_B=H^{-1}H_B.
$$

The reconstructed transfers satisfy

$$
(T_X-Y^2I)(T_X-Y^{-2}I)(T_X-Y^{-4}I)=0,
\qquad X=A,B,
$$

leading to exact spectral projectors \(P_X^{(2)},P_X^{(-2)},P_X^{(-4)}\)
and the ordered-run formula

$$
\Upsilon_w(Y)
=
(-1)^{-\mu(w)}Y^{-\mu(w)}
L^T
\prod_{\text{ordered runs }X^n\subset w}
\left[
Y^{2n}P_X^{(2)}
+Y^{-2n}P_X^{(-2)}
+Y^{-4n}P_X^{(-4)}
\right]R.
$$

This part freezes the complete symbolic formula from short words, predicts the
**entire Laurent polynomial** for 20 unseen long words, and only then computes
the corresponding spatial graphs independently.  Equality is checked
coefficient-by-coefficient rather than at sampled numerical values of \(Y\).


In [ ]:
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import csv
import hashlib
import itertools
import json
import os
import subprocess
import time
import warnings

import networkx as nx
import numpy as np
import sympy as sp

AUDITED_SOURCE_COMMIT = "49e34e47ca5c182f55ef5f0ea0906220df59befb"
STRICT_PUBLICATION_REGENERATION = (
    os.environ.get("KNOTTEDGRAPH_STRICT_PUBLICATION_REGENERATION", "0") == "1"
)
CONSTRUCTOR_VERSION = "2026-08-27.fixed-cubic-pure-braid-word.v2"

A = sp.Symbol("A")

def _is_repo_root(path: Path) -> bool:
    return (
        (path / "pyproject.toml").is_file()
        and (path / "src" / "knotted_graph").is_dir()
        and (path / "User_guide" / "applications").is_dir()
    )

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if _is_repo_root(candidate):
            return candidate
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

ROOT = find_repo_root()
RESULTS_DIR = ROOT / "User_guide" / "applications" / "results" / "05_yamada_formula_discovery"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SHORT_RAW_CSV = RESULTS_DIR / "05c_symbolic_short_raw_training.csv"
FORMULA_JSON = RESULTS_DIR / "05c_symbolic_frozen_formula.json"
HELDOUT_PLAN_CSV = RESULTS_DIR / "05c_symbolic_heldout_plan.csv"
PREDICTIONS_CSV = RESULTS_DIR / "05c_symbolic_full_polynomial_predictions.csv"
ACTUAL_CSV = RESULTS_DIR / "05c_symbolic_heldout_actual.csv"
CHECKS_CSV = RESULTS_DIR / "05c_symbolic_coefficient_checks.csv"
SUMMARY_JSON = RESULTS_DIR / "05c_symbolic_heldout_summary.json"

def git_text(*args):
    proc = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.returncode:
        raise RuntimeError(
            f"git {' '.join(args)} failed.\n{proc.stdout}\n{proc.stderr}"
        )
    return proc.stdout.strip()

CURRENT_BRANCH = git_text("rev-parse", "--abbrev-ref", "HEAD")
GIT_COMMIT = git_text("rev-parse", "HEAD")

AUDITED_SOURCE_PRESENT = subprocess.run(
    ["git", "merge-base", "--is-ancestor", AUDITED_SOURCE_COMMIT, "HEAD"],
    cwd=ROOT,
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

if STRICT_PUBLICATION_REGENERATION and not AUDITED_SOURCE_PRESENT:
    raise RuntimeError(
        "Strict publication regeneration requires audited source commit "
        f"{AUDITED_SOURCE_COMMIT}. Current HEAD is {GIT_COMMIT}."
    )
if not AUDITED_SOURCE_PRESENT:
    warnings.warn(
        "The audited source commit is not an ancestor of this checkout. "
        "Exploration may continue, but do not treat regenerated tables as "
        "publication-audited outputs.",
        RuntimeWarning,
    )

library_status = git_text("status", "--porcelain", "--", "src/knotted_graph")
if library_status:
    raise RuntimeError(
        "Commit/stash changes under src/knotted_graph before an audited run:\n"
        + library_status
    )

import knotted_graph
KG_FILE = Path(knotted_graph.__file__).resolve()
if ROOT not in KG_FILE.parents:
    message = (
        "Python imported knotted_graph from outside this checkout: "
        f"{KG_FILE}. Exploration may continue, but strict publication "
        "regeneration requires the audited editable checkout."
    )
    if STRICT_PUBLICATION_REGENERATION:
        raise RuntimeError(message)
    warnings.warn(message, RuntimeWarning)

from knotted_graph.core.embedding import ensure_embedding
from knotted_graph.projection import PDCode
from knotted_graph.invariants.yamada.polynomial import Yamada
from knotted_graph.invariants.yamada.factorized_frontier import (
    native_factorized_available,
    factorized_import_error,
)

if not native_factorized_available():
    message = (
        "The optimized factorized Yamada backend is unavailable. "
        f"Heavy formula-discovery cells cannot run: {factorized_import_error()!r}"
    )
    if STRICT_PUBLICATION_REGENERATION:
        raise RuntimeError(message)
    warnings.warn(message, RuntimeWarning)

print("Repository :", ROOT)
print("Branch     :", CURRENT_BRANCH)
print("Commit     :", GIT_COMMIT)
print("Package    :", KG_FILE)

from sympy.polys.matrices import DomainMatrix


### Fixed family, symbolic variable, and transfer basis

In [ ]:
# The family found by notebook 08b.
FAMILY_NAME = "pure_squares"
BLOCKS = {
    "A": (+1, +1),  # sigma_1^2
    "B": (+2, +2),  # sigma_2^2
}

NORMALIZE_YAMADA = False  # spectral formula is tested on the raw Laurent polynomial

BRAID_SAMPLES_PER_GENERATOR = 7
LANE_SPACING = 0.90
CROSSING_Z = 0.32
MIDDLE_SPLIT = 0.45

HELDOUT_LENGTHS = [101, 125, 150, 200]
HELDOUT_WORDS_PER_LENGTH = 5

# A 15-word residual basis.  All entries needed for H, H_A, H_B
# have length <= 9; none of the extreme held-out words enters reconstruction.
TRANSFER_BASIS = [
    "",
    "A", "B",
    "AA", "AB", "BA", "BB",
    "AAB", "ABA", "ABB", "BAA", "BAB",
    "AABA", "ABAA", "ABAB",
]

if len(TRANSFER_BASIS) != 15 or len(set(TRANSFER_BASIS)) != 15:
    raise AssertionError("TRANSFER_BASIS must contain 15 distinct words.")

print("Family:", FAMILY_NAME, BLOCKS)
print("Transfer basis size:", len(TRANSFER_BASIS))
print("Formula test mode: full symbolic coefficient identity (no sampled Y-values)")
print("Held-out lengths:", HELDOUT_LENGTHS)

# Use Y in the theorem/formula.  The direct evaluator may internally use A;
# coefficient dictionaries are variable-independent.
Y = sp.Symbol("Y")

print("SYMBOLIC audit: no numerical Y-specializations are used in the formula test.")


In [ ]:
def _polyline(points, samples_per_segment=5):
    points = [np.asarray(p, dtype=float) for p in points]
    out = []
    for i in range(len(points) - 1):
        t = np.linspace(0.0, 1.0, int(samples_per_segment))
        seg = (
            points[i][None, :] * (1.0 - t[:, None])
            + points[i+1][None, :] * t[:, None]
        )
        if i:
            seg = seg[1:]
        out.extend(seg.tolist())
    return np.asarray(out, dtype=float)


def _block_permutation(block):
    occ = [0, 1, 2]
    for g in block:
        i = abs(int(g)) - 1
        if i not in (0, 1):
            raise ValueError(g)
        occ[i], occ[i+1] = occ[i+1], occ[i]
    # occ[lane] is starting identity now at lane.
    inverse = [None, None, None]
    for lane, identity in enumerate(occ):
        inverse[identity] = lane
    return tuple(inverse)


# Validate only the frozen alphabet used in this symbolic audit.
# No dependency on the exploratory PURE_BLOCK_LIBRARY catalogue.
for _name, _block in BLOCKS.items():
    if _block_permutation(_block) != (0, 1, 2):
        raise AssertionError(
            f"Frozen block {_name} is not pure: "
            f"permutation={_block_permutation(_block)}"
        )

print("PASS: all catalogue blocks have identity strand permutation.")


def expand_word(word, blocks):
    expanded = []
    for symbol in str(word):
        expanded.extend(blocks[symbol])
    return tuple(int(g) for g in expanded)


def _braid_body_paths(braid):
    """
    Return three polylines, one for each strand identity.
    Pure input blocks ensure the final lane of each identity equals its start lane,
    but this routine itself handles any 3-braid sequence.
    """
    y_lanes = np.array([LANE_SPACING, 0.0, -LANE_SPACING], dtype=float)
    n = len(braid)

    if n == 0:
        n_body = 1.0
        return [
            np.array([
                [0.0, y_lanes[i], 0.0],
                [n_body, y_lanes[i], 0.0],
            ], dtype=float)
            for i in range(3)
        ], n_body

    points = {
        edge_id: [[0.0, y_lanes[edge_id], 0.0]]
        for edge_id in range(3)
    }
    occupant = [0, 1, 2]

    for step, generator in enumerate(braid):
        pair = abs(int(generator)) - 1
        upper_lane = pair
        lower_lane = pair + 1

        upper_edge = occupant[upper_lane]
        lower_edge = occupant[lower_lane]

        over_edge = upper_edge if generator > 0 else lower_edge
        under_edge = lower_edge if generator > 0 else upper_edge

        lane_of_edge = {
            occupant[lane]: lane
            for lane in range(3)
        }

        t = np.linspace(0.0, 1.0, BRAID_SAMPLES_PER_GENERATOR)
        smooth = 0.5 - 0.5 * np.cos(np.pi * t)
        bump = np.sin(np.pi * t)

        for edge_id in range(3):
            lane0 = lane_of_edge[edge_id]
            if edge_id == upper_edge:
                lane1 = lower_lane
            elif edge_id == lower_edge:
                lane1 = upper_lane
            else:
                lane1 = lane0

            xs = step + t
            ys = (
                (1.0 - smooth) * y_lanes[lane0]
                + smooth * y_lanes[lane1]
            )
            if edge_id == over_edge:
                zs = CROSSING_Z * bump
            elif edge_id == under_edge:
                zs = -CROSSING_Z * bump
            else:
                zs = np.zeros_like(t)

            seg = np.column_stack([xs, ys, zs])
            points[edge_id].extend(seg[1:].tolist())

        occupant[upper_lane], occupant[lower_lane] = (
            occupant[lower_lane], occupant[upper_lane]
        )

    final_lane = {
        occupant[lane]: lane
        for lane in range(3)
    }
    if tuple(final_lane[i] for i in range(3)) != (0, 1, 2):
        raise AssertionError(
            "Expanded word is not pure; endpoint lanes changed."
        )

    return [
        np.asarray(points[i], dtype=float)
        for i in range(3)
    ], float(n)


def fixed_cubic_pure_braid_graph(word, blocks):
    word = str(word).strip().upper()
    if any(ch not in {"A", "B"} for ch in word):
        raise ValueError("Word must use A/B only.")

    braid = expand_word(word, blocks)
    body_paths, xR = _braid_body_paths(braid)

    s = LANE_SPACING
    d = MIDDLE_SPLIT

    # Left boundary.
    LT  = np.array([0.0, +s, 0.0])
    LMB = np.array([0.0,  0.0, 0.0])
    LMC = np.array([-d,   0.0, 0.0])
    LB  = np.array([0.0, -s, 0.0])

    # Right boundary.
    RT  = np.array([xR, +s, 0.0])
    RMB = np.array([xR,  0.0, 0.0])
    RMC = np.array([xR + d, 0.0, 0.0])
    RB  = np.array([xR, -s, 0.0])

    pos = {
        "LT": LT, "LMB": LMB, "LMC": LMC, "LB": LB,
        "RT": RT, "RMB": RMB, "RMC": RMC, "RB": RB,
    }

    g = nx.MultiGraph()
    for node, p in pos.items():
        g.add_node(node, pos=p.copy())

    # Three braid-carrying graph edges.
    g.add_edge("LT",  "RT",  pts=body_paths[0], role="braid_top")
    g.add_edge("LMB", "RMB", pts=body_paths[1], role="braid_middle")
    g.add_edge("LB",  "RB",  pts=body_paths[2], role="braid_bottom")

    # Middle split/spacer edges.
    g.add_edge(
        "LMB", "LMC",
        pts=_polyline([LMB, LMC], 3),
        role="middle_left_spacer",
    )
    g.add_edge(
        "RMB", "RMC",
        pts=_polyline([RMB, RMC], 3),
        role="middle_right_spacer",
    )

    # Four side couplings.  No one of these is a graph bridge because
    # each adjacent cycle pair is coupled on BOTH left and right.
    g.add_edge(
        "LT", "LMB",
        pts=_polyline([LT, LMB], 3),
        role="top_middle_left",
    )
    g.add_edge(
        "RT", "RMB",
        pts=_polyline([RT, RMB], 3),
        role="top_middle_right",
    )
    g.add_edge(
        "LMC", "LB",
        pts=_polyline([LMC, LB], 3),
        role="middle_bottom_left",
    )
    g.add_edge(
        "RMC", "RB",
        pts=_polyline([RMC, RB], 3),
        role="middle_bottom_right",
    )

    # Three nested external closures.  Their x-margins decrease with height,
    # so the projected closure polylines are pairwise nonintersecting.
    top_route = [
        RT,
        [xR + 3.0, +s, 0.0],
        [xR + 3.0, 5.0, 0.0],
        [-3.0, 5.0, 0.0],
        [-3.0, +s, 0.0],
        LT,
    ]
    middle_route = [
        RMC,
        [xR + 2.0, 0.0, 0.0],
        [xR + 2.0, 4.0, 0.0],
        [-2.0, 4.0, 0.0],
        [-2.0, 0.0, 0.0],
        LMC,
    ]
    bottom_route = [
        RB,
        [xR + 1.0, -s, 0.0],
        [xR + 1.0, 3.0, 0.0],
        [-1.0, 3.0, 0.0],
        [-1.0, -s, 0.0],
        LB,
    ]

    g.add_edge("RT",  "LT",  pts=_polyline(top_route),    role="closure_top")
    g.add_edge("RMC", "LMC", pts=_polyline(middle_route), role="closure_middle")
    g.add_edge("RB",  "LB",  pts=_polyline(bottom_route), role="closure_bottom")

    g.graph.update(
        family="fixed_cubic_pure_braid_word",
        word=word,
        word_length=len(word),
        expanded_braid=list(braid),
        expected_braid_crossings=len(braid),
        expected_vertices=8,
        expected_edges=12,
    )

    g = ensure_embedding(g, copy=False, normalize=True)

    degrees = sorted(int(dg) for _, dg in g.degree())
    if degrees != [3] * 8:
        raise AssertionError(f"Expected cubic degree sequence; got {degrees}.")
    if g.number_of_edges() != 12:
        raise AssertionError(
            f"Expected 12 graph edges; got {g.number_of_edges()}."
        )
    if not nx.is_connected(nx.Graph(g)):
        raise AssertionError("Graph unexpectedly disconnected.")

    # Reject accidental cut edges: Yamada/flow-type invariants can collapse on them.
    simple = nx.Graph(g)
    bridges = list(nx.bridges(simple))
    if bridges:
        raise AssertionError(f"Unexpected graph bridges: {bridges}")

    return g


# Structural preflight: the abstract graph must literally be identical.
reference = fixed_cubic_pure_braid_graph(
    "",
    {"A": (+1,+1), "B": (+2,+2)},
)

ref_edges = sorted(
    (min(u,v), max(u,v))
    for u, v in nx.Graph(reference).edges()
)

for _word in ["A", "B", "AB", "BA", "AABB", "ABBA"]:
    _g = fixed_cubic_pure_braid_graph(
        _word,
        {"A": (+1,+1), "B": (+2,+2)},
    )
    edges = sorted(
        (min(u,v), max(u,v))
        for u, v in nx.Graph(_g).edges()
    )
    if edges != ref_edges:
        raise AssertionError(
            f"Abstract graph changed for word {_word!r}."
        )

print("PASS: fixed 8-vertex/12-edge cubic abstract graph for all preflight words.")


### Exact Laurent helpers and independent direct evaluator

In [ ]:
def laurent_coefficients_exact(expr, variable):
    expr = sp.expand(expr)
    if expr == 0:
        return {}
    out = {}
    for term in sp.Add.make_args(expr):
        coeff, exponent = term.as_coeff_exponent(variable)
        if exponent.is_Integer is not True:
            raise ValueError(term)
        if variable in coeff.free_symbols:
            raise ValueError(term)
        exponent = int(exponent)
        coeff = sp.expand(coeff)
        if coeff.is_Integer is not True:
            raise ValueError(coeff)
        out[exponent] = int(out.get(exponent, 0) + int(coeff))
    return {k:v for k,v in sorted(out.items()) if v}


def coeffs_json(coeffs):
    return json.dumps(coeffs, sort_keys=True, separators=(",", ":"))


def coeffs_hash(coeffs):
    return hashlib.sha256(coeffs_json(coeffs).encode()).hexdigest()


def atomic_write_rows(rows, path, fields):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, "") for k in fields})
        f.flush()
        os.fsync(f.fileno())
    tmp.replace(path)


def all_words_exact(n):
    return [
        "".join(chars)
        for chars in itertools.product(("A","B"), repeat=int(n))
    ] if n else [""]


def all_words_upto(n):
    out = []
    for k in range(int(n)+1):
        out.extend(all_words_exact(k))
    return out


def evaluate_coeffs_at(coeff_json, value):
    raw = json.loads(coeff_json)
    x = sp.Integer(value)
    return sp.expand(sum(
        sp.Integer(int(c)) * x**int(p)
        for p, c in raw.items()
    ))


In [ ]:
def evaluate_coeff_dict_at(coeffs, value):
    x = sp.Integer(int(value))
    return sp.expand(sum(
        sp.Integer(int(c)) * x**int(power)
        for power, c in coeffs.items()
    ))


def normalize_raw_coeffs(raw_coeffs):
    """
    Exact KnottedGraph normalization:
      shift the lowest Laurent exponent to zero,
      multiply by (-1)^displacement.
    """
    if not raw_coeffs:
        return {}, 0

    lowest = min(int(p) for p in raw_coeffs)
    displacement = -lowest
    sign = -1 if displacement % 2 else 1

    normalized = {
        int(power) + displacement: sign * int(coefficient)
        for power, coefficient in raw_coeffs.items()
        if int(coefficient)
    }
    normalized = {
        p: c for p, c in sorted(normalized.items()) if c
    }
    return normalized, lowest


def evaluate_word_raw_direct(word):
    """
    Independently construct the full spatial graph and compute its exact
    unnormalized Yamada Laurent polynomial using the production library.
    """
    start = time.perf_counter()

    graph = fixed_cubic_pure_braid_graph(word, BLOCKS)

    projection_start = time.perf_counter()
    pd = PDCode(graph)
    pd.compute(
        rotation_angles=(0.0, 0.0, 0.0),
        rotation_order="ZYX",
    )
    projection_seconds = time.perf_counter() - projection_start

    expected_crossings = len(expand_word(word, BLOCKS))
    actual_crossings = len(pd.crossings)
    if actual_crossings != expected_crossings:
        raise AssertionError(
            f"word length {len(word)}: expected {expected_crossings} braid crossings, "
            f"but canonical projection has {actual_crossings}."
        )

    yamada_start = time.perf_counter()
    computer = Yamada(
        vertices=list(pd.vertices.values()),
        crossings=list(pd.crossings.values()),
        arcs=list(pd.arcs.values()),
    )
    raw_expr = sp.expand(computer.compute(A, normalize=False))
    yamada_seconds = time.perf_counter() - yamada_start

    raw_coeffs = laurent_coefficients_exact(raw_expr, A)
    normalized_coeffs, mu = normalize_raw_coeffs(raw_coeffs)

    return {
        "family": FAMILY_NAME,
        "word": word,
        "word_length": len(word),
        "n_A": word.count("A"),
        "n_B": word.count("B"),
        "count_signature": f"A{word.count('A')}_B{word.count('B')}",
        "braid_crossings": expected_crossings,
        "selected_crossings": actual_crossings,
        "vertices": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "max_degree": max(int(d) for _, d in graph.degree()),
        "graph_bridge_count": len(list(nx.bridges(nx.Graph(graph)))),
        "raw_coefficients_json": coeffs_json(raw_coeffs),
        "raw_polynomial_sha256": coeffs_hash(raw_coeffs),
        "raw_laurent_terms": len(raw_coeffs),
        "raw_min_exponent": min(raw_coeffs) if raw_coeffs else "",
        "raw_max_exponent": max(raw_coeffs) if raw_coeffs else "",
        "mu": mu,
        "normalized_coefficients_json": coeffs_json(normalized_coeffs),
        "normalized_polynomial_sha256": coeffs_hash(normalized_coeffs),
        "normalized_terms": len(normalized_coeffs),
        "projection_seconds": projection_seconds,
        "yamada_seconds": yamada_seconds,
        "wall_seconds": time.perf_counter() - start,
        "branch": CURRENT_BRANCH,
        "git_commit": GIT_COMMIT,
        "constructor_version": CONSTRUCTOR_VERSION,
        "status": "success",
    }


### Stage 1 — short exact raw Yamada data only
For the fixed 15-word residual basis \(\mathcal B\), compute

$$
H_{u,v}(Y)=R_{uv}(Y),\quad
(H_A)_{u,v}(Y)=R_{uAv}(Y),\quad
(H_B)_{u,v}(Y)=R_{uBv}(Y).
$$

Only short words required for these matrices are evaluated.  No extreme word
is used in the reconstruction.

In [ ]:
required_short_words = sorted(
    {
        word
        for u in TRANSFER_BASIS
        for v in TRANSFER_BASIS
        for word in (u + v, u + "A" + v, u + "B" + v)
    },
    key=lambda w: (len(w), w),
)

print("Required short words:", len(required_short_words))
print("Maximum short-word length:", max(map(len, required_short_words)))

SHORT_FIELDS = [
    "family", "word", "word_length", "n_A", "n_B", "count_signature",
    "braid_crossings", "selected_crossings",
    "vertices", "edges", "max_degree", "graph_bridge_count",
    "raw_coefficients_json", "raw_polynomial_sha256",
    "raw_laurent_terms", "raw_min_exponent", "raw_max_exponent",
    "mu", "normalized_coefficients_json", "normalized_polynomial_sha256",
    "normalized_terms", "projection_seconds", "yamada_seconds", "wall_seconds",
    "branch", "git_commit", "constructor_version", "status",
]

short_rows_by_word = {}

# Reuse only rows generated by the same commit/constructor if the checkpoint exists.
if SHORT_RAW_CSV.exists():
    with SHORT_RAW_CSV.open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            if (
                row.get("git_commit") == GIT_COMMIT
                and row.get("constructor_version") == CONSTRUCTOR_VERSION
                and row.get("word") in required_short_words
                and row.get("status") == "success"
            ):
                short_rows_by_word[row["word"]] = row

for index, word in enumerate(required_short_words, start=1):
    if word in short_rows_by_word:
        print(
            f"REUSE [{index:3d}/{len(required_short_words):3d}] "
            f"{word or '<empty>':10s}",
            flush=True,
        )
        continue

    row = evaluate_word_raw_direct(word)
    short_rows_by_word[word] = row

    print(
        f"PASS  [{index:3d}/{len(required_short_words):3d}] "
        f"{word or '<empty>':10s} "
        f"crossings={row['selected_crossings']:2d} "
        f"terms={row['raw_laurent_terms']:4d} "
        f"Yamada={row['yamada_seconds']:.4g}s",
        flush=True,
    )

    atomic_write_rows(
        [
            short_rows_by_word[w]
            for w in required_short_words
            if w in short_rows_by_word
        ],
        SHORT_RAW_CSV,
        SHORT_FIELDS,
    )

missing = [w for w in required_short_words if w not in short_rows_by_word]
if missing:
    raise RuntimeError(f"Short training ledger incomplete: {missing[:10]}")

print("Short raw training CSV:", SHORT_RAW_CSV.resolve())


### Stage 2 — reconstruct the symbolic matrices (kept in the exact rational-function DomainMatrix backend) \(T_A(Y),T_B(Y)\)
We work over the exact rational-function field \(\mathbb Q(Y)\).  The
reconstruction is

$$
T_A=H^{-1}H_A,\qquad T_B=H^{-1}H_B.
$$

Then we explicitly verify the symbolic cubic identities

$$
(T_X-Y^2I)(T_X-Y^{-2}I)(T_X-Y^{-4}I)=0,
\qquad X=A,B.
$$

In [ ]:
def short_raw_coeffs(word):
    return {
        int(k): int(v)
        for k, v in json.loads(
            short_rows_by_word[word]["raw_coefficients_json"]
        ).items()
    }


def coeff_dict_to_expr(coeffs, variable=Y):
    if not coeffs:
        return sp.Integer(0)
    return sp.Add(
        *[
            sp.Integer(int(c)) * variable**int(p)
            for p, c in sorted(coeffs.items())
        ],
        evaluate=True,
    )


def raw_expr(word):
    return coeff_dict_to_expr(short_raw_coeffs(word), Y)


print("Building symbolic 15 x 15 Hankel matrices ...", flush=True)

H = sp.Matrix([
    [raw_expr(u + v) for v in TRANSFER_BASIS]
    for u in TRANSFER_BASIS
])
HA = sp.Matrix([
    [raw_expr(u + "A" + v) for v in TRANSFER_BASIS]
    for u in TRANSFER_BASIS
])
HB = sp.Matrix([
    [raw_expr(u + "B" + v) for v in TRANSFER_BASIS]
    for u in TRANSFER_BASIS
])

print("Converting once to exact rational-function DomainMatrix ...", flush=True)

H_dm = DomainMatrix.from_Matrix(H).to_field()
F = H_dm.domain

HA_dm = DomainMatrix.from_Matrix(HA).convert_to(F)
HB_dm = DomainMatrix.from_Matrix(HB).convert_to(F)

print("Exact field:", F, flush=True)
print("Inverting H in the exact field ...", flush=True)

H_inv_dm = H_dm.inv()

print("Constructing T_A and T_B without converting back to SymPy Matrix ...", flush=True)

TA_dm = H_inv_dm.matmul(HA_dm)
TB_dm = H_inv_dm.matmul(HB_dm)

I15_dm = DomainMatrix.eye((15, 15), F)


def fscalar(expr):
    """Convert a scalar expression to the exact matrix coefficient field."""
    return F.convert(expr)


def dm_shift(T, eigenvalue):
    """T - eigenvalue * I, entirely inside DomainMatrix."""
    return T.sub(I15_dm.scalarmul(fscalar(eigenvalue)))


print("Checking symbolic cubic law for A in DomainMatrix ...", flush=True)
cubic_A_dm = (
    dm_shift(TA_dm, Y**2)
    .matmul(dm_shift(TA_dm, Y**-2))
    .matmul(dm_shift(TA_dm, Y**-4))
)
if not cubic_A_dm.is_zero_matrix:
    raise AssertionError(
        "SYMBOLIC cubic identity failed for T_A. "
        "Do not proceed to held-out testing."
    )
print("  A: PASS", flush=True)

print("Checking symbolic cubic law for B in DomainMatrix ...", flush=True)
cubic_B_dm = (
    dm_shift(TB_dm, Y**2)
    .matmul(dm_shift(TB_dm, Y**-2))
    .matmul(dm_shift(TB_dm, Y**-4))
)
if not cubic_B_dm.is_zero_matrix:
    raise AssertionError(
        "SYMBOLIC cubic identity failed for T_B. "
        "Do not proceed to held-out testing."
    )
print("  B: PASS", flush=True)

empty_index = TRANSFER_BASIS.index("")

# L^T is the empty-prefix Hankel row; R is the empty-suffix basis vector.
ell_dm = H_dm.extract([empty_index], list(range(15)))
rvec_dm = I15_dm.extract(list(range(15)), [empty_index])

print("Checking symbolic noncommutativity in DomainMatrix ...", flush=True)
commutator_dm = TA_dm.matmul(TB_dm).sub(TB_dm.matmul(TA_dm))
if commutator_dm.is_zero_matrix:
    raise AssertionError("Unexpected symbolic commutativity: [T_A,T_B]=0.")

print("PASS: symbolic cubic law holds for both generators.")
print("PASS: symbolic commutator is nonzero.")


### Stage 3 — construct the symbolic spectral projectors
For \(X=A,B\),

$$
P_X^{(2)}
=
\frac{(T_X-Y^{-2}I)(T_X-Y^{-4}I)}
{(Y^2-Y^{-2})(Y^2-Y^{-4})},
$$

$$
P_X^{(-2)}
=
\frac{(T_X-Y^{2}I)(T_X-Y^{-4}I)}
{(Y^{-2}-Y^{2})(Y^{-2}-Y^{-4})},
$$

$$
P_X^{(-4)}
=
\frac{(T_X-Y^{2}I)(T_X-Y^{-2}I)}
{(Y^{-4}-Y^{2})(Y^{-4}-Y^{-2})}.
$$

We verify symbolically that they are orthogonal idempotents summing to \(I\).

In [ ]:
def dm_divide_by_scalar(M, scalar_expr):
    scalar = fscalar(scalar_expr)
    if scalar == F.zero:
        raise ZeroDivisionError("Projector denominator vanished symbolically.")
    return M.scalarmul(F.one / scalar)


def domain_spectral_projectors(T):
    P2_num = (
        dm_shift(T, Y**-2)
        .matmul(dm_shift(T, Y**-4))
    )
    P2 = dm_divide_by_scalar(
        P2_num,
        (Y**2 - Y**-2) * (Y**2 - Y**-4),
    )

    Pm2_num = (
        dm_shift(T, Y**2)
        .matmul(dm_shift(T, Y**-4))
    )
    Pm2 = dm_divide_by_scalar(
        Pm2_num,
        (Y**-2 - Y**2) * (Y**-2 - Y**-4),
    )

    Pm4_num = (
        dm_shift(T, Y**2)
        .matmul(dm_shift(T, Y**-2))
    )
    Pm4 = dm_divide_by_scalar(
        Pm4_num,
        (Y**-4 - Y**2) * (Y**-4 - Y**-2),
    )

    return {"2": P2, "-2": Pm2, "-4": Pm4}


print("Constructing exact spectral projectors in DomainMatrix ...", flush=True)
PA_dm = domain_spectral_projectors(TA_dm)
PB_dm = domain_spectral_projectors(TB_dm)

for letter, P in [("A", PA_dm), ("B", PB_dm)]:
    total = P["2"].add(P["-2"]).add(P["-4"])
    if not total.sub(I15_dm).is_zero_matrix:
        raise AssertionError(f"{letter}: projectors do not sum to I.")

    for label in ("2", "-2", "-4"):
        if not P[label].matmul(P[label]).sub(P[label]).is_zero_matrix:
            raise AssertionError(
                f"{letter}: projector {label} is not idempotent."
            )

    labels = ("2", "-2", "-4")
    for i, left in enumerate(labels):
        for right in labels[i+1:]:
            if not P[left].matmul(P[right]).is_zero_matrix:
                raise AssertionError(
                    f"{letter}: projectors {left},{right} are not orthogonal."
                )

print("PASS: both projector triples are exact orthogonal resolutions of I.")


### Stage 4 — verify the displayed run formula symbolically
For any positive integer \(n\), define

$$
S_X(n)
=
Y^{2n}P_X^{(2)}
+Y^{-2n}P_X^{(-2)}
+Y^{-4n}P_X^{(-4)}.
$$

Because the projectors are exact spectral projectors, this is the displayed
formula's local factor.  We explicitly verify representative symbolic powers
against \(T_X^n\), and the cubic identity proves the equality for arbitrary
\(n\).

In [ ]:
_RUN_MATRIX_CACHE = {}


def domain_run_matrix(letter, n):
    """
    The displayed formula's run factor

      S_X(n) = Y^(2n) P_X^(2)
             + Y^(-2n) P_X^(-2)
             + Y^(-4n) P_X^(-4)

    represented entirely in the exact rational-function DomainMatrix field.
    """
    key = (str(letter), int(n))
    cached = _RUN_MATRIX_CACHE.get(key)
    if cached is not None:
        return cached

    n = int(n)
    P = PA_dm if letter == "A" else PB_dm

    S = (
        P["2"].scalarmul(fscalar(Y**(2*n)))
        .add(P["-2"].scalarmul(fscalar(Y**(-2*n))))
        .add(P["-4"].scalarmul(fscalar(Y**(-4*n))))
    )
    _RUN_MATRIX_CACHE[key] = S
    return S


print("Checking displayed spectral run formula without generic SymPy expansion ...", flush=True)

for letter, T in [("A", TA_dm), ("B", TB_dm)]:
    power = I15_dm
    for n in range(1, 7):
        power = power.matmul(T)
        spectral = domain_run_matrix(letter, n)
        if not power.sub(spectral).is_zero_matrix:
            raise AssertionError(
                f"Displayed spectral run formula failed symbolically "
                f"for {letter}^{n}."
            )

print("PASS: displayed spectral run formula agrees exactly with T_X^n.")
print("The cubic identity establishes the same equality for arbitrary n.")


### Stage 5 — verify the symbolic transfer realization on every short training word
This catches basis-orientation errors before freezing the formula.

In [ ]:
state_cache_dm = {"": ell_dm}
short_symbolic_failures = []

for word in sorted(required_short_words, key=lambda w: (len(w), w)):
    if word:
        prefix = word[:-1]
        letter = word[-1]

        if prefix not in state_cache_dm:
            raise RuntimeError(f"Missing cached prefix {prefix!r}.")

        T = TA_dm if letter == "A" else TB_dm
        state_cache_dm[word] = state_cache_dm[prefix].matmul(T)

    predicted_field = (
        state_cache_dm[word]
        .matmul(rvec_dm)
        .to_list()[0][0]
    )
    actual_field = fscalar(raw_expr(word))

    if predicted_field != actual_field:
        short_symbolic_failures.append(word)

if short_symbolic_failures:
    raise AssertionError(
        "Symbolic transfer realization failed on short exact words: "
        + repr(short_symbolic_failures[:10])
    )

print(
    f"PASS: exact-field transfer realization reproduces "
    f"{len(required_short_words)}/{len(required_short_words)} "
    f"short raw polynomials."
)


### Stage 6 — freeze the exact symbolic formula
The complete symbolic matrices (kept in the exact rational-function DomainMatrix backend) and projectors are serialized **before** any
extreme held-out Yamada invariant is evaluated.

In [ ]:
def dm_payload(M):
    """
    Canonical text serialization of exact DomainMatrix field elements.
    Avoids conversion of all 225 entries to generic SymPy expressions.
    """
    return [
        [str(value) for value in row]
        for row in M.to_list()
    ]


formula_payload = {
    "formula_name": "full_symbolic_ordered_pure_braid_three_channel_formula",
    "family": FAMILY_NAME,
    "blocks": {k: list(v) for k, v in BLOCKS.items()},
    "basis": TRANSFER_BASIS,
    "coefficient_field": str(F),
    "git_commit": GIT_COMMIT,
    "constructor_version": CONSTRUCTOR_VERSION,
    "raw_formula": (
        "R_w(Y)=L^T prod_ordered_runs_X^n "
        "[Y^(2n) P_X^(2)+Y^(-2n) P_X^(-2)+Y^(-4n) P_X^(-4)] R"
    ),
    "normalized_formula": (
        "Upsilon_w(Y)=(-1)^(-mu(w)) Y^(-mu(w)) R_w(Y)"
    ),
    "T_A": dm_payload(TA_dm),
    "T_B": dm_payload(TB_dm),
    "P_A_2": dm_payload(PA_dm["2"]),
    "P_A_m2": dm_payload(PA_dm["-2"]),
    "P_A_m4": dm_payload(PA_dm["-4"]),
    "P_B_2": dm_payload(PB_dm["2"]),
    "P_B_m2": dm_payload(PB_dm["-2"]),
    "P_B_m4": dm_payload(PB_dm["-4"]),
    "ell": dm_payload(ell_dm),
    "r": dm_payload(rvec_dm),
}

formula_canonical = json.dumps(
    formula_payload,
    sort_keys=True,
    separators=(",", ":"),
)
FORMULA_SHA256 = hashlib.sha256(
    formula_canonical.encode("utf-8")
).hexdigest()

formula_payload["formula_sha256"] = FORMULA_SHA256

FORMULA_JSON.write_text(
    json.dumps(formula_payload, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("SYMBOLIC FORMULA FROZEN.")
print("Formula SHA256:", FORMULA_SHA256)
print("Formula file  :", FORMULA_JSON.resolve())


### Stage 7 — freeze 20 extreme held-out words
Five deterministic designs are used at each of

$$
m=101,\ 125,\ 150,\ 200.
$$

The balanced-block and balanced-interleaved words have identical \(n_A,n_B\)
but very different order.

In [ ]:
def repeat_pattern(pattern, m):
    return (pattern * ((m + len(pattern) - 1) // len(pattern)))[:m]


def deterministic_irregular_word(m):
    chars = []
    for i in range(int(m)):
        digest = hashlib.sha256(
            f"ordered-yamada-extreme-v1:{m}:{i}".encode("utf-8")
        ).digest()
        chars.append("A" if (digest[0] & 1) == 0 else "B")
    return "".join(chars)


def heldout_words(m):
    m = int(m)
    nA = (m + 1) // 2
    nB = m // 2

    designs = [
        ("balanced_block", "A"*nA + "B"*nB),
        ("balanced_interleaved", repeat_pattern("AB", m)),
        ("A_heavy", repeat_pattern("AAAB", m)),
        ("B_heavy", repeat_pattern("ABBB", m)),
        ("deterministic_irregular", deterministic_irregular_word(m)),
    ]

    if any(len(word) != m for _, word in designs):
        raise AssertionError(f"Held-out length construction failed for m={m}.")
    if len({word for _, word in designs}) != HELDOUT_WORDS_PER_LENGTH:
        raise AssertionError(f"Held-out designs collided for m={m}.")

    # Explicitly require the same-count pair.
    lookup = dict(designs)
    b1 = lookup["balanced_block"]
    b2 = lookup["balanced_interleaved"]
    if (b1.count("A"), b1.count("B")) != (b2.count("A"), b2.count("B")):
        raise AssertionError(f"Balanced same-count pair failed for m={m}.")

    return designs


heldout_plan_rows = []
training_word_set = set(required_short_words)

for m in HELDOUT_LENGTHS:
    for within_index, (design, word) in enumerate(heldout_words(m), start=1):
        if word in training_word_set:
            raise AssertionError("Held-out word overlaps short reconstruction data.")

        heldout_plan_rows.append({
            "family": FAMILY_NAME,
            "target_length": m,
            "within_length_index": within_index,
            "design": design,
            "word": word,
            "n_A": word.count("A"),
            "n_B": word.count("B"),
            "count_signature": f"A{word.count('A')}_B{word.count('B')}",
            "braid_crossings": len(expand_word(word, BLOCKS)),
            "formula_sha256": FORMULA_SHA256,
            "status": "FROZEN_BEFORE_HELDOUT_YAMADA",
        })

HELDOUT_FIELDS = [
    "family", "target_length", "within_length_index", "design", "word",
    "n_A", "n_B", "count_signature", "braid_crossings",
    "formula_sha256", "status",
]

atomic_write_rows(
    heldout_plan_rows,
    HELDOUT_PLAN_CSV,
    HELDOUT_FIELDS,
)

plan_json = json.dumps(
    heldout_plan_rows,
    sort_keys=True,
    separators=(",", ":"),
)
HELDOUT_PLAN_SHA256 = hashlib.sha256(plan_json.encode("utf-8")).hexdigest()

print("Held-out plan frozen:", HELDOUT_PLAN_CSV.resolve())
print("Held-out plan SHA256:", HELDOUT_PLAN_SHA256)
print("Number of unseen graphs:", len(heldout_plan_rows))


### Stage 8 — predict the **entire Laurent polynomial** from the frozen displayed formula
This occurs before direct evaluation of any held-out graph.

In [ ]:
def ordered_runs(word):
    if not word:
        return []

    runs = []
    current = word[0]
    count = 1

    for ch in word[1:]:
        if ch == current:
            count += 1
        else:
            runs.append((current, count))
            current = ch
            count = 1

    runs.append((current, count))
    return runs


def field_element_to_laurent_coeffs(value):
    """
    Decode a final exact F=Q(Y) / Z(Y) field element directly.

    For a genuine Laurent polynomial the reduced denominator must be one
    monomial Y^k (with coefficient 1).  We then shift numerator exponents by k.

    This avoids sp.cancel/sp.expand on the very large m=101..200 predictions.
    """
    if value == F.zero:
        return {}

    denominator_terms = list(value.denom.terms())
    if len(denominator_terms) != 1:
        raise AssertionError(
            "Predicted formula did not reduce to a Laurent polynomial: "
            f"denominator={value.denom}"
        )

    (den_exp_tuple, den_coeff_raw) = denominator_terms[0]
    if len(den_exp_tuple) != 1:
        raise AssertionError(
            f"Unexpected multivariate denominator: {value.denom}"
        )

    den_coeff = sp.Rational(str(den_coeff_raw))
    if den_coeff != 1:
        raise AssertionError(
            "Predicted result has a non-unit scalar denominator: "
            f"{value.denom}"
        )

    denominator_power = int(den_exp_tuple[0])

    coeffs = {}
    for (num_exp_tuple, coeff_raw) in value.numer.terms():
        if len(num_exp_tuple) != 1:
            raise AssertionError(
                f"Unexpected multivariate numerator: {value.numer}"
            )

        coeff_q = sp.Rational(str(coeff_raw))
        if coeff_q.q != 1:
            raise AssertionError(
                "Predicted Laurent coefficient is not integral: "
                f"{coeff_q}"
            )

        exponent = int(num_exp_tuple[0]) - denominator_power
        coefficient = int(coeff_q)
        if coefficient:
            coeffs[exponent] = coeffs.get(exponent, 0) + coefficient

    return {
        p: c for p, c in sorted(coeffs.items())
        if c
    }


def predict_raw_field_from_displayed_formula(word):
    """
    Literally propagate the displayed ordered-run formula in the exact field.

      L^T prod S_X(n) R

    No generic SymPy Matrix, simplify, cancel, or expand is used.
    """
    state = ell_dm

    for letter, n in ordered_runs(word):
        state = state.matmul(domain_run_matrix(letter, n))

    return state.matmul(rvec_dm).to_list()[0][0]


PREDICTION_FIELDS = [
    "target_length", "within_length_index", "design", "word",
    "n_A", "n_B", "count_signature",
    "predicted_raw_coefficients_json",
    "predicted_raw_polynomial_sha256",
    "predicted_raw_terms",
    "predicted_raw_min_exponent",
    "predicted_raw_max_exponent",
    "predicted_mu",
    "predicted_normalized_coefficients_json",
    "predicted_normalized_polynomial_sha256",
    "predicted_normalized_terms",
    "formula_sha256", "heldout_plan_sha256",
    "status",
]

prediction_rows = []
prediction_by_key = {}

for i, plan in enumerate(heldout_plan_rows, start=1):
    word = plan["word"]

    print(
        f"PREDICT [{i:2d}/{len(heldout_plan_rows):2d}] "
        f"m={plan['target_length']:3d} {plan['design']:23s} "
        f"runs={len(ordered_runs(word)):3d}",
        flush=True,
    )

    predicted_field = predict_raw_field_from_displayed_formula(word)
    raw_coeffs = field_element_to_laurent_coeffs(predicted_field)
    normalized_coeffs, mu = normalize_raw_coeffs(raw_coeffs)

    row = {
        "target_length": plan["target_length"],
        "within_length_index": plan["within_length_index"],
        "design": plan["design"],
        "word": word,
        "n_A": plan["n_A"],
        "n_B": plan["n_B"],
        "count_signature": plan["count_signature"],
        "predicted_raw_coefficients_json": coeffs_json(raw_coeffs),
        "predicted_raw_polynomial_sha256": coeffs_hash(raw_coeffs),
        "predicted_raw_terms": len(raw_coeffs),
        "predicted_raw_min_exponent": min(raw_coeffs) if raw_coeffs else "",
        "predicted_raw_max_exponent": max(raw_coeffs) if raw_coeffs else "",
        "predicted_mu": mu,
        "predicted_normalized_coefficients_json": coeffs_json(normalized_coeffs),
        "predicted_normalized_polynomial_sha256": coeffs_hash(normalized_coeffs),
        "predicted_normalized_terms": len(normalized_coeffs),
        "formula_sha256": FORMULA_SHA256,
        "heldout_plan_sha256": HELDOUT_PLAN_SHA256,
        "status": "FULL_SYMBOLIC_POLYNOMIAL_PREDICTED_BEFORE_DIRECT_YAMADA",
    }

    prediction_rows.append(row)
    prediction_by_key[
        (int(plan["target_length"]), int(plan["within_length_index"]))
    ] = row

    atomic_write_rows(
        prediction_rows,
        PREDICTIONS_CSV,
        PREDICTION_FIELDS,
    )

print("\nALL 20 FULL SYMBOLIC POLYNOMIAL PREDICTIONS FROZEN.")
print("Predictions CSV:", PREDICTIONS_CSV.resolve())


### Stage 9 — independently compute the 20 exact held-out Yamada polynomials
Only now are the extreme spatial graphs sent to KnottedGraph's production
Yamada evaluator.

In [ ]:
ACTUAL_FIELDS = [
    "family", "target_length", "within_length_index", "design",
    "word", "word_length", "n_A", "n_B", "count_signature",
    "braid_crossings", "selected_crossings", "vertices", "edges",
    "max_degree", "graph_bridge_count",
    "raw_coefficients_json", "raw_polynomial_sha256",
    "raw_laurent_terms", "raw_min_exponent", "raw_max_exponent",
    "mu", "normalized_coefficients_json", "normalized_polynomial_sha256",
    "normalized_terms", "projection_seconds", "yamada_seconds", "wall_seconds",
    "formula_sha256", "heldout_plan_sha256",
    "branch", "git_commit", "constructor_version", "status",
]

actual_rows_by_key = {}

if ACTUAL_CSV.exists():
    with ACTUAL_CSV.open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            key = (int(row["target_length"]), int(row["within_length_index"]))
            if (
                row.get("git_commit") == GIT_COMMIT
                and row.get("constructor_version") == CONSTRUCTOR_VERSION
                and row.get("formula_sha256") == FORMULA_SHA256
                and row.get("heldout_plan_sha256") == HELDOUT_PLAN_SHA256
                and row.get("status") == "success"
            ):
                actual_rows_by_key[key] = row

for graph_index, plan in enumerate(heldout_plan_rows, start=1):
    key = (int(plan["target_length"]), int(plan["within_length_index"]))

    if key in actual_rows_by_key:
        row = actual_rows_by_key[key]
        print(
            f"REUSE [{graph_index:2d}/{len(heldout_plan_rows):2d}] "
            f"m={plan['target_length']:3d} {plan['design']:23s} "
            f"terms={row['raw_laurent_terms']}",
            flush=True,
        )
        continue

    row = evaluate_word_raw_direct(plan["word"])
    row.update({
        "target_length": plan["target_length"],
        "within_length_index": plan["within_length_index"],
        "design": plan["design"],
        "formula_sha256": FORMULA_SHA256,
        "heldout_plan_sha256": HELDOUT_PLAN_SHA256,
    })
    actual_rows_by_key[key] = row

    print(
        f"PASS  [{graph_index:2d}/{len(heldout_plan_rows):2d}] "
        f"m={plan['target_length']:3d} {plan['design']:23s} "
        f"crossings={row['selected_crossings']:3d} "
        f"raw_terms={row['raw_laurent_terms']:5d} "
        f"Yamada={row['yamada_seconds']:.4g}s",
        flush=True,
    )

    atomic_write_rows(
        [
            actual_rows_by_key[
                (int(p["target_length"]), int(p["within_length_index"]))
            ]
            for p in heldout_plan_rows
            if (
                int(p["target_length"]),
                int(p["within_length_index"]),
            ) in actual_rows_by_key
        ],
        ACTUAL_CSV,
        ACTUAL_FIELDS,
    )

print("Exact held-out actual CSV:", ACTUAL_CSV.resolve())


### Stage 10 — explicit coefficient-by-coefficient formula test
A graph passes iff both

$$
R_w^{\rm formula}(Y)=R_w^{\rm direct}(Y)
$$

and

$$
\Upsilon_w^{\rm formula}(Y)=\Upsilon_w^{\rm direct}(Y)
$$

as complete Laurent polynomials.

No sampling of \(Y\) is used here.

In [ ]:
CHECK_FIELDS = [
    "target_length", "within_length_index", "design", "word",
    "n_A", "n_B",
    "predicted_raw_sha256", "actual_raw_sha256",
    "raw_coefficient_exact_match",
    "predicted_normalized_sha256", "actual_normalized_sha256",
    "normalized_coefficient_exact_match",
    "predicted_mu", "actual_mu", "mu_exact_match",
    "formula_sha256", "heldout_plan_sha256",
]

check_rows = []
passed_graphs = 0

for plan in heldout_plan_rows:
    key = (
        int(plan["target_length"]),
        int(plan["within_length_index"]),
    )
    pred = prediction_by_key[key]
    actual = actual_rows_by_key[key]

    pred_raw = {
        int(k): int(v)
        for k, v in json.loads(
            pred["predicted_raw_coefficients_json"]
        ).items()
    }
    act_raw = {
        int(k): int(v)
        for k, v in json.loads(
            actual["raw_coefficients_json"]
        ).items()
    }

    pred_norm = {
        int(k): int(v)
        for k, v in json.loads(
            pred["predicted_normalized_coefficients_json"]
        ).items()
    }
    act_norm = {
        int(k): int(v)
        for k, v in json.loads(
            actual["normalized_coefficients_json"]
        ).items()
    }

    raw_match = pred_raw == act_raw
    norm_match = pred_norm == act_norm
    mu_match = int(pred["predicted_mu"]) == int(actual["mu"])

    if raw_match and norm_match and mu_match:
        passed_graphs += 1

    check_rows.append({
        "target_length": key[0],
        "within_length_index": key[1],
        "design": plan["design"],
        "word": plan["word"],
        "n_A": plan["n_A"],
        "n_B": plan["n_B"],
        "predicted_raw_sha256": pred["predicted_raw_polynomial_sha256"],
        "actual_raw_sha256": actual["raw_polynomial_sha256"],
        "raw_coefficient_exact_match": raw_match,
        "predicted_normalized_sha256":
            pred["predicted_normalized_polynomial_sha256"],
        "actual_normalized_sha256":
            actual["normalized_polynomial_sha256"],
        "normalized_coefficient_exact_match": norm_match,
        "predicted_mu": pred["predicted_mu"],
        "actual_mu": actual["mu"],
        "mu_exact_match": mu_match,
        "formula_sha256": FORMULA_SHA256,
        "heldout_plan_sha256": HELDOUT_PLAN_SHA256,
    })

atomic_write_rows(
    check_rows,
    CHECKS_CSV,
    CHECK_FIELDS,
)

# Same-count order-sensitivity audit.
same_count_pairs = []
for m in HELDOUT_LENGTHS:
    plans = {
        p["design"]: p
        for p in heldout_plan_rows
        if int(p["target_length"]) == int(m)
    }
    p_block = plans["balanced_block"]
    p_inter = plans["balanced_interleaved"]

    k_block = (m, int(p_block["within_length_index"]))
    k_inter = (m, int(p_inter["within_length_index"]))

    a_block = actual_rows_by_key[k_block]
    a_inter = actual_rows_by_key[k_inter]

    same_count_pairs.append({
        "m": m,
        "same_counts": (
            p_block["n_A"], p_block["n_B"]
        ) == (
            p_inter["n_A"], p_inter["n_B"]
        ),
        "normalized_yamada_different": (
            a_block["normalized_polynomial_sha256"]
            != a_inter["normalized_polynomial_sha256"]
        ),
    })

summary = {
    "family": FAMILY_NAME,
    "git_commit": GIT_COMMIT,
    "constructor_version": CONSTRUCTOR_VERSION,
    "formula_sha256": FORMULA_SHA256,
    "heldout_plan_sha256": HELDOUT_PLAN_SHA256,
    "heldout_lengths": HELDOUT_LENGTHS,
    "heldout_graphs": len(heldout_plan_rows),
    "graphs_passing_full_raw_coefficient_identity": sum(
        bool(r["raw_coefficient_exact_match"])
        for r in check_rows
    ),
    "graphs_passing_full_normalized_coefficient_identity": sum(
        bool(r["normalized_coefficient_exact_match"])
        for r in check_rows
    ),
    "graphs_passing_every_requirement": passed_graphs,
    "same_count_order_pairs": same_count_pairs,
    "all_pass": passed_graphs == len(heldout_plan_rows),
}

SUMMARY_JSON.write_text(
    json.dumps(summary, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("\n================ FULL SYMBOLIC EXTREME AUDIT ================")
print(
    "Raw Laurent coefficient identities     : "
    f"{summary['graphs_passing_full_raw_coefficient_identity']}/"
    f"{len(heldout_plan_rows)}"
)
print(
    "Normalized coefficient identities      : "
    f"{summary['graphs_passing_full_normalized_coefficient_identity']}/"
    f"{len(heldout_plan_rows)}"
)
print(
    "Graphs passing every requirement        : "
    f"{passed_graphs}/{len(heldout_plan_rows)}"
)

print("\nSame-count order pairs:")
for row in same_count_pairs:
    print(
        f"  m={row['m']:3d}: "
        f"same_counts={row['same_counts']}, "
        f"different_normalized_Yamada="
        f"{row['normalized_yamada_different']}"
    )

print("\nFormula SHA256:", FORMULA_SHA256)
print("Plan SHA256   :", HELDOUT_PLAN_SHA256)
print("Checks CSV    :", CHECKS_CSV.resolve())
print("Summary JSON  :", SUMMARY_JSON.resolve())

if not summary["all_pass"]:
    failed = [
        r for r in check_rows
        if not (
            bool(r["raw_coefficient_exact_match"])
            and bool(r["normalized_coefficient_exact_match"])
            and bool(r["mu_exact_match"])
        )
    ]
    print("\nFIRST FAILURES:")
    for r in failed[:10]:
        print(
            r["target_length"],
            r["design"],
            "raw=", r["raw_coefficient_exact_match"],
            "norm=", r["normalized_coefficient_exact_match"],
            "mu=", r["mu_exact_match"],
        )
    raise AssertionError(
        f"FULL SYMBOLIC AUDIT FAILED: "
        f"{passed_graphs}/{len(heldout_plan_rows)} graphs passed."
    )

print("\nALL 20 FULL POLYNOMIAL IDENTITIES PASSED COEFFICIENT-BY-COEFFICIENT.")


### What a successful run establishes
A successful final line

```text
ALL 20 FULL POLYNOMIAL IDENTITIES PASSED COEFFICIENT-BY-COEFFICIENT.
```

means the displayed formula itself—not merely numerical evaluations of it—was
frozen from short data and then predicted the complete exact Yamada polynomial
for all 20 previously unseen extreme words.

This remains computational validation rather than an all-word mathematical
proof, but it is the appropriate strongest out-of-sample test of the formula.

# Reading the completed notebook

A successful complete run supports three progressively stronger computational
observations:

$$
\begin{array}{ll}
\Lambda_m,\ \Lambda_m^{\downarrow},\ \Lambda_m^{\updownarrow}
&
\text{compact homogeneous closed forms},\\[0.35em]
\mathcal A_{\mathbf n}
&
\text{mixed theta words factor through motif counts},\\[0.35em]
\mathcal N_w
&
\text{ordered pure-braid words can retain word order}.
\end{array}
$$

The long held-out tests are deliberately separated from formula discovery.
They therefore provide strong exact out-of-sample validation.  They are not,
by themselves, an all-parameter mathematical proof; the remaining analytic
task is to derive the reconstructed transfer identities directly from the
Yamada skein algebra.

For terminology:

- **Abelian** means the closed invariant factors through the count/Parikh
  vector of the local motif word.
- **Non-Abelian** means some same-count words have different Yamada
  polynomials, so ordered local operations remain observable after closure.
